<a href="https://colab.research.google.com/github/jzwillucf/Hetnet_Analysis/blob/main/HetnetMarkov.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initiate Neo4j

In [ ]:
pip install neo4j

In [ ]:
from neo4j import GraphDatabase
from neo4j.exceptions import ServiceUnavailable, AuthError
import pandas as pd

uri = "blank"
user = "neo4j"
password = "blank"

In [ ]:
try:
    with GraphDatabase.driver(uri, auth=(user, password)) as driver:
        driver.verify_connectivity()
        print("Connection to Neo4j database successful!")
except Exception as e:
    print(f"Connection failed: {e}")

# Build Graph

In [ ]:
import networkx as nx

# Query to extract an expansive graph centered on the compound Bortezomib
query = """
MATCH path = (start)-[*1..5]-(other)
WHERE toLower(toString(start.name)) CONTAINS 'bortezomib' OR toLower(toString(start.id)) CONTAINS 'bortezomib'
UNWIND relationships(path) AS r
WITH startNode(r) AS n, r, endNode(r) AS m
RETURN n, r, m LIMIT 50000
"""

G = nx.Graph()

try:
    with GraphDatabase.driver(uri, auth=(user, password)) as driver:
        with driver.session() as session:
            result = session.run(query)
            for record in result:
                n = record["n"]
                m = record["m"]
                r = record["r"]

                # Add nodes with their element_id, labels, and properties
                G.add_node(n.element_id, labels=list(n.labels), properties=dict(n))
                G.add_node(m.element_id, labels=list(m.labels), properties=dict(m))

                # Add undirected edge with type and properties
                G.add_edge(n.element_id, m.element_id, type=r.type, properties=dict(r))

    print(f"Graph successfully built with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")
except Exception as e:
    print(f"An error occurred while building the graph: {e}")


# Pagerank

In [ ]:
import networkx as nx

# Calculate the stationary distribution (PageRank is a common method for this in directed graphs)
stationary_dist = nx.pagerank(G)

# Sort the nodes by their probability in the stationary distribution (descending)
sorted_dist = sorted(stationary_dist.items(), key=lambda x: x[1], reverse=True)

# Print the top 10 nodes with the highest stationary probability
print("Top 10 nodes in the stationary distribution:")
for node, prob in sorted_dist[:10]:
    # Optionally try to get the 'name' or 'id' property if it exists
    node_data = G.nodes[node].get('properties', {})
    node_name = node_data.get('name', node_data.get('id', node))
    print(f"{node_name}: {prob:.6f}")


## Weighted vs Pagerank

This code calculates a Personalized (Anchored) PageRank to determine which nodes are most relevant specifically to Bortezomib, rather than just being 'important' in the general graph.

Anchoring: It first searches for any node containing the name 'Bortezomib' to act as the starting point (or 'anchor').
Personalization: It creates a probability distribution where the 'teleportation' chance is concentrated 100% on those Bortezomib nodes. This ensures the random walk constantly returns to the drug.
Weighting: It ensures every edge has a weight property (defaulting to 0.01 if missing) so the PageRank algorithm can factor in relationship strength.
Comparison: Finally, it calculates the scores and compares them against the Stationary Distribution (Global PageRank). This highlights nodes like 'Nausea' or 'Dizziness' that might rank low globally but rank very high when the focus is narrowed to Bortezomib.

In [ ]:
import networkx as nx
import pandas as pd

# Identify Bortezomib node(s) to use as the anchor for the walk
anchors = {}
for n, d in G.nodes(data=True):
    props = d.get('properties', {})
    name = str(props.get('name', '')).lower()
    node_id = str(props.get('id', '')).lower()
    if 'bortezomib' in name or 'bortezomib' in node_id:
        anchors[n] = 1.0

if not anchors:
    print("Bortezomib node(s) not found in the graph.")
else:
    # Normalize anchor weights if there are multiple matches
    total_anchor = sum(anchors.values())
    personalization = {n: (anchors.get(n, 0.0) / total_anchor) for n in G.nodes()}

    # Extract edge weights from properties (defaulting to 0.01 if not found)
    for u, v, d in G.edges(data=True):
        edge_props = d.get('properties', {})
        d['weight'] = float(edge_props.get('weight', 0.01))

    # Perform Anchored Walk (Personalized PageRank)
    anchored_dist = nx.pagerank(G, personalization=personalization, weight='weight')

    # Sort results
    sorted_anchored = sorted(anchored_dist.items(), key=lambda x: x[1], reverse=True)

    # Compare to Stationary Distribution
    stationary_ranks = {node: rank + 1 for rank, (node, _) in enumerate(sorted_dist)}

    comparison_data = []
    for rank, (node, prob) in enumerate(sorted_anchored[:15]):
        node_data = G.nodes[node].get('properties', {})
        node_name = node_data.get('name', node_data.get('id', node))

        stat_prob = stationary_dist.get(node, 0)
        stat_rank = stationary_ranks.get(node, 'N/A')

        comparison_data.append({
            'Node Name': node_name,
            'Anchored Rank': rank + 1,
            'Anchored Prob': round(prob, 6),
            'Stationary Rank': stat_rank,
            'Stationary Prob': round(stat_prob, 6)
        })

    print("Top 15 nodes from Anchored Walk centered on Bortezomib vs Stationary Distribution:")
    df_comparison = pd.DataFrame(comparison_data)
    display(df_comparison)


Like the last, but filtering for diseases

In [ ]:
import networkx as nx
import pandas as pd

# Identify Bortezomib node(s) to use as the anchor for the walk
anchors = {}
for n, d in G.nodes(data=True):
    props = d.get('properties', {})
    name = str(props.get('name', '')).lower()
    node_id = str(props.get('id', '')).lower()
    if 'bortezomib' in name or 'bortezomib' in node_id:
        anchors[n] = 1.0

if not anchors:
    print("Bortezomib node(s) not found in the graph.")
else:
    # Normalize anchor weights if there are multiple matches
    total_anchor = sum(anchors.values())
    personalization = {n: (anchors.get(n, 0.0) / total_anchor) for n in G.nodes()}

    # Extract edge weights from properties (defaulting to 0.01 if not found)
    for u, v, d in G.edges(data=True):
        edge_props = d.get('properties', {})
        d['weight'] = float(edge_props.get('weight', 0.01))

    # Perform Anchored Walk (Personalized PageRank)
    anchored_dist = nx.pagerank(G, personalization=personalization, weight='weight')

    # Sort results
    sorted_anchored = sorted(anchored_dist.items(), key=lambda x: x[1], reverse=True)

    # Filter for diseases
    disease_anchored = []
    for node, prob in sorted_anchored:
        labels = [str(l).lower() for l in G.nodes[node].get('labels', [])]
        is_disease = any(l for l in labels if 'disease' in l or 'side effect' in l or 'side_effect' in l or 'phenotype' in l)
        if is_disease:
            disease_anchored.append((node, prob))

    # Compare to Stationary Distribution
    stationary_ranks = {node: rank + 1 for rank, (node, _) in enumerate(sorted_dist)}

    comparison_data = []
    for rank, (node, prob) in enumerate(disease_anchored[:15]):
        node_data = G.nodes[node].get('properties', {})
        node_name = node_data.get('name', node_data.get('id', node))

        stat_prob = stationary_dist.get(node, 0)
        stat_rank = stationary_ranks.get(node, 'N/A')

        comparison_data.append({
            'Node Name': node_name,
            'Disease Rank': rank + 1,
            'Anchored Prob': round(prob, 6),
            'Stationary Rank': stat_rank,
            'Stationary Prob': round(stat_prob, 6)
        })

    print("Top 15 Disease nodes from Anchored Walk centered on Bortezomib vs Stationary Distribution:")
    df_comparison = pd.DataFrame(comparison_data)
    display(df_comparison)


In [ ]:
import random
from collections import Counter
import pandas as pd

# 1. Identify starting nodes (Bortezomib)
start_nodes = [
    n for n in G.nodes()
    if 'bortezomib' in str(G.nodes[n].get('properties', {}).get('name', '')).lower()
    or 'bortezomib' in str(G.nodes[n].get('properties', {}).get('id', '')).lower()
]

if not start_nodes:
    print("Bortezomib not found in the graph.")
else:
    # Walk parameters
    num_walks = 20000  # Number of independent walks
    walk_length = 5    # Steps per walk
    visit_counts = Counter()

    print(f"Simulating {num_walks} weighted walks of length {walk_length} anchored at Bortezomib...")

    # 2. Perform the walks
    for _ in range(num_walks):
        current = random.choice(start_nodes)

        for _ in range(walk_length):
            neighbors = list(G.neighbors(current))
            if not neighbors:
                break  # Dead end

            # Extract weights for transition probabilities
            weights = []
            for neighbor in neighbors:
                edge_data = G.get_edge_data(current, neighbor)
                w = float(edge_data.get('properties', {}).get('weight', 0.01))
                weights.append(w)

            # Move to the next node based on edge weights
            current = random.choices(neighbors, weights=weights, k=1)[0]
            visit_counts[current] += 1

    # 3. Aggregate and display the top visited disease nodes
    walk_results = []
    disease_count = 0
    for node, count in visit_counts.most_common():
        labels = [str(l).lower() for l in G.nodes[node].get('labels', [])]
        is_disease = any(l for l in labels if 'disease' in l or 'side effect' in l or 'side_effect' in l or 'phenotype' in l)

        if is_disease:
            node_name = str(G.nodes[node].get('properties', {}).get('name', G.nodes[node].get('properties', {}).get('id', node)))
            walk_results.append({
                'Node Name': node_name,
                'Total Visits': count,
                'Visit Probability': f"{count / (num_walks * walk_length):.4f}"
            })
            disease_count += 1
            if disease_count >= 15:
                break

    df_walk = pd.DataFrame(walk_results)
    print("\nTop 15 disease/side effect nodes most frequently visited during the weighted walk:")
    display(df_walk)

Above code snippets w/o side effects in results

In [ ]:
import networkx as nx
import pandas as pd

# Identify Bortezomib node(s) to use as the anchor for the walk
anchors = {}
for n, d in G.nodes(data=True):
    props = d.get('properties', {})
    name = str(props.get('name', '')).lower()
    node_id = str(props.get('id', '')).lower()
    if 'bortezomib' in name or 'bortezomib' in node_id:
        anchors[n] = 1.0

if not anchors:
    print("Bortezomib node(s) not found in the graph.")
else:
    # Normalize anchor weights if there are multiple matches
    total_anchor = sum(anchors.values())
    personalization = {n: (anchors.get(n, 0.0) / total_anchor) for n in G.nodes()}

    # Extract edge weights from properties (defaulting to 0.01 if not found)
    for u, v, d in G.edges(data=True):
        edge_props = d.get('properties', {})
        d['weight'] = float(edge_props.get('weight', 0.01))

    # Perform Anchored Walk (Personalized PageRank)
    anchored_dist = nx.pagerank(G, personalization=personalization, weight='weight')

    # Sort results
    sorted_anchored = sorted(anchored_dist.items(), key=lambda x: x[1], reverse=True)

    # Filter for diseases
    disease_anchored = []
    for node, prob in sorted_anchored:
        labels = [str(l).lower() for l in G.nodes[node].get('labels', [])]
        is_disease = any(l for l in labels if 'disease' in l)
        if is_disease:
            disease_anchored.append((node, prob))

    # Compare to Stationary Distribution
    stationary_ranks = {node: rank + 1 for rank, (node, _) in enumerate(sorted_dist)}

    comparison_data = []
    for rank, (node, prob) in enumerate(disease_anchored[:15]):
        node_data = G.nodes[node].get('properties', {})
        node_name = node_data.get('name', node_data.get('id', node))

        stat_prob = stationary_dist.get(node, 0)
        stat_rank = stationary_ranks.get(node, 'N/A')

        comparison_data.append({
            'Node Name': node_name,
            'Disease Rank': rank + 1,
            'Anchored Prob': round(prob, 6),
            'Stationary Rank': stat_rank,
            'Stationary Prob': round(stat_prob, 6)
        })

    print("Top 15 Disease nodes from Anchored Walk centered on Bortezomib vs Stationary Distribution:")
    df_comparison = pd.DataFrame(comparison_data)
    display(df_comparison)


In [ ]:
import random
from collections import Counter
import pandas as pd

# 1. Identify starting nodes (Bortezomib)
start_nodes = [
    n for n in G.nodes()
    if 'bortezomib' in str(G.nodes[n].get('properties', {}).get('name', '')).lower()
    or 'bortezomib' in str(G.nodes[n].get('properties', {}).get('id', '')).lower()
]

if not start_nodes:
    print("Bortezomib not found in the graph.")
else:
    # Walk parameters
    num_walks = 20000  # Number of independent walks
    walk_length = 5    # Steps per walk
    visit_counts = Counter()

    print(f"Simulating {num_walks} weighted walks of length {walk_length} anchored at Bortezomib...")

    # 2. Perform the walks
    for _ in range(num_walks):
        current = random.choice(start_nodes)

        for _ in range(walk_length):
            neighbors = list(G.neighbors(current))
            if not neighbors:
                break  # Dead end

            # Extract weights for transition probabilities
            weights = []
            for neighbor in neighbors:
                edge_data = G.get_edge_data(current, neighbor)
                w = float(edge_data.get('properties', {}).get('weight', 0.01))
                weights.append(w)

            # Move to the next node based on edge weights
            current = random.choices(neighbors, weights=weights, k=1)[0]
            visit_counts[current] += 1

    # 3. Aggregate and display the top visited disease nodes
    walk_results = []
    disease_count = 0
    for node, count in visit_counts.most_common():
        labels = [str(l).lower() for l in G.nodes[node].get('labels', [])]
        is_disease = any(l for l in labels if 'disease' in l)

        if is_disease:
            node_name = str(G.nodes[node].get('properties', {}).get('name', G.nodes[node].get('properties', {}).get('id', node)))
            walk_results.append({
                'Node Name': node_name,
                'Total Visits': count,
                'Visit Probability': f"{count / (num_walks * walk_length):.4f}"
            })
            disease_count += 1
            if disease_count >= 15:
                break

    df_walk = pd.DataFrame(walk_results)
    print("\nTop 15 disease/side effect nodes most frequently visited during the weighted walk:")
    display(df_walk)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Anchored PageRank
sns.barplot(data=df_comparison.head(10), x='Anchored Prob', y='Node Name', hue='Node Name', ax=axes[0], palette='Blues_r', legend=False)
axes[0].set_title('Top 10 Diseases by Anchored PageRank')
axes[0].set_xlabel('Anchored Probability')
axes[0].set_ylabel('')

# Weighted Walk
sns.barplot(data=df_walk.head(10), x='Total Visits', y='Node Name', hue='Node Name', ax=axes[1], palette='Oranges_r', legend=False)
axes[1].set_title('Top 10 Diseases by Weighted Walk')
axes[1].set_xlabel('Total Visits')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

# Weight-limited Searches

This code implements a Weight-Limited Path Search (similar to a Breadth-First Search) to find the most common connections originating from Bortezomib.

Start Identification: It looks for all nodes in the graph that represent 'Bortezomib'.
Path Traversal (BFS): Starting from the drug, it explores outward one neighbor at a time. It keeps track of the 'accumulated weight' (the distance or cost) along the path.
Thresholding: The walk for a specific path stops if it exceeds a weight_limit of 100 or if the total number of explored paths reaches 50,000. This prevents the search from running indefinitely.
Frequency Analysis: It uses a Counter to track two things:
Paths: The exact sequence of nodes (e.g., Bortezomib -> Protein A -> Disease B).
End Nodes: The final destination of each path, which helps identify which diseases or proteins are most reachable from the drug.
Reporting: Finally, it prints the top 10 most frequent paths and end nodes, giving you a clear view of the drug's primary neighborhood in the graph

In [ ]:
import networkx as nx
from collections import Counter

# 1. Identify starting nodes (Bortezomib)
start_nodes = [
    n for n in G.nodes()
    if 'bortezomib' in str(G.nodes[n].get('properties', {}).get('name', '')).lower()
    or 'bortezomib' in str(G.nodes[n].get('properties', {}).get('id', '')).lower()
]

# Set the maximum accumulated weight limit and run limit for the walk
weight_limit = 100
max_runs = 50000

visited_paths = []
path_name_counter = Counter()
end_node_counter = Counter()
run_count = 0

# 2. Perform a weight-limited walk (BFS-style traversal)
for start_node in start_nodes:
    # Queue stores tuples of: (current_node, accumulated_weight, path_taken)
    queue = [(start_node, 0.0, [start_node])]

    while queue and run_count < max_runs:
        run_count += 1
        current, acc_weight, path = queue.pop(0)

        # Record valid paths (longer than just the start node)
        if len(path) > 1:
            visited_paths.append((path, acc_weight))

            # Convert path IDs to names to find frequent semantic paths
            path_names = tuple(str(G.nodes[n].get('properties', {}).get('name', G.nodes[n].get('properties', {}).get('id', n))) for n in path)
            path_name_counter[path_names] += 1

            # Record the end node
            end_node = path[-1]
            end_node_name = str(G.nodes[end_node].get('properties', {}).get('name', G.nodes[end_node].get('properties', {}).get('id', end_node)))
            end_node_counter[end_node_name] += 1

        # Explore successors using neighbors for an undirected graph
        for neighbor in G.neighbors(current):
            if neighbor not in path:  # Prevent simple cycles in the current path
                edge_data = G.get_edge_data(current, neighbor)
                # Extract edge weight, fallback to 0.01 as done previously
                edge_weight = float(edge_data.get('properties', {}).get('weight', 0.01))

                new_weight = acc_weight + edge_weight

                # Continue walk only if the limit is not exceeded
                if new_weight <= weight_limit:
                    queue.append((neighbor, new_weight, path + [neighbor]))

print(f"Stopped after {run_count} runs (Queue limit checked).")
print(f"Found {len(visited_paths)} paths originating from Bortezomib with total accumulated weight <= {weight_limit}\n")

# 3. Display the most frequent paths found
print("Top 10 Most Frequent Paths (by node names):")
for path_names, count in path_name_counter.most_common(10):
    print(f"Frequency: {count} | Path: {' -> '.join(path_names)}")

# 4. Display the most frequent end nodes
print("\nTop 10 Most Frequent End Nodes:")
for node_name, count in end_node_counter.most_common(10):
    print(f"Frequency: {count} | Node: {node_name}")


### Algorithmic Steps: Weight-Limited Breadth-First Search (BFS)

1. **Initialization:**
   - Identify the starting node(s) (e.g., Bortezomib) and define exploration constraints: `weight_limit` (maximum allowed path cost) and `max_runs` (maximum paths to explore to prevent infinite loops).
   - Initialize an exploration `queue` with a starting tuple containing: the start node, an initial accumulated weight of `0.0`, and the initial path (just the start node).
   - Set up counters to track the frequencies of visited paths and end nodes.

2. **Traversal Loop (While queue is not empty and max_runs not reached):**
   - **Dequeue:** Pop the first tuple from the queue to get the `current_node`, `accumulated_weight`, and `path_taken`.
   - **Record State:** If the path has traversed at least one edge (length > 1), record the path and its final end-node in their respective frequency counters. *(Note: subsequent variations of this code add a filtering step here to only record paths ending in specific node types, like 'disease'.)*
   - **Expand Neighbors:** For every neighbor of the `current_node`:
     - **Cycle Check:** Ensure the neighbor is not already in the current `path_taken` to avoid getting stuck in simple loops.
     - **Calculate New Weight:** Fetch the edge weight between the `current_node` and the neighbor (defaulting to 0.01 if missing) and add it to the `accumulated_weight`.
     - **Threshold Check & Enqueue:** If the new accumulated weight is less than or equal to the `weight_limit`, append the neighbor, the new accumulated weight, and the extended path back into the queue for future exploration.
*italicized text*
3. **Aggregation and Reporting:**
   - Once the queue is exhausted or the run limit is hit, aggregate the counters.
   - Display the most frequently traversed paths and the most commonly reached end nodes to understand the most significant routes branching out from the starting node.

Of course! Here is the revised research document on Weight-Limited Breadth-First Search. I've incorporated the detailed pseudocode, a concrete walkthrough example, and a more nuanced analysis of complexity from the sources to provide a more comprehensive and helpful guide for your presentation.

***

## Weight-Limited Breadth-First Search: Equations and Concepts

This document provides a comprehensive overview of the equations, concepts, and implementation details relevant to a Weight-Limited Breadth-First Search (BFS) algorithm for your presentation. It is important to note that this is not a standard, formally named algorithm in the way that BFS or Dijkstra's algorithm are [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFCq7oZKbu3Itgg4rGmiwBGCVrQyk6qE1yOCnVyh1NpluyIb9KNw7FpcqOyvPHv5OJrOOam69qu8PKCTh7D_r758BDnFXMGpCSDd0ke44Em1kX3YSlYz0e_mfomXMCYey0Jc4HKVXtx3YUuTdl5yqoduqgawXDogw5u6UKXCHD_hjErjJbdHad21Bdn1zl96XSqWqZ_qmvVFVf439u3tXUdLbgQ3hckrFcKlpzHWuzCZj90-KvzfzuaWTQlM9BVm5Oh8HtyTZ6g-c07) . Instead, it represents a practical modification of the Breadth-First Search approach to incorporate a constraint based on the total weight or cost of a path.

### 1. Foundational Concepts: Graphs and BFS

To understand any graph algorithm, it's helpful to start with the formal definitions of graphs, paths, and the standard BFS algorithm .

#### Formal Graph and Path Notation

*   **Graph:** In mathematics, a graph `G` is formally defined as an ordered pair `G = (V, E)` [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFH2uPHm4Cv1OgtkW2rFVHwzj16EZ0jjSt_2GRqcJ19cTblE7yQhIAMp379_l03s28Dq-GnKLQYWGxT4McX_qZLLr9D54yodxlwFaB8cyuOYNBIfvaUmD5vimxUBjSFvv2evL31OqHx7-_P_6i_CBbWjnyADQkb61DsWg4wVQCsrPSav4UE)[[3]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEy-Rry1SfoDZ6ik11DA4QCM6UWDWt6pM1ungO0bmRs84RCGSmst_HtAdQDBI9x4ofCOkP9bpjBaH4eGk4AY-48nG2ZBJDAs6svnaAyKbQpdde8sTr-Bkg5VySWkyzKWFlBq9NCO_JiKs1oub7sOGx3QqPY58I=) .
    *   **V**: A finite, non-empty set of objects called **vertices** or nodes [[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFSdnHMc48A0RXDSaGR7uYqPx-Uhxgo_hfN1WKpl7Bw8NOpRWZiayHFknYT_GaHbDWib3csRDZlpkWUStiRDcxVa6srGWzjZcbXVA4cQEB8AHoTPNvbYhLyqpPZfJPOKMraYArulOCurMPv)[[5]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQF-6yuHO7Emw95BGch7txubToPuXCVycG1uNmyvMq_zxXM4T4VPrXrRQbqXOzL6CK5gNcKTTPognQvwb5PDw_CDFfQrgiPWVPD4ejszde3pnPj1ARoGq5vagy9tzP25vIPXG0Wy1ZQS9CyMhZZbPMLYEglqUJPIdM51JHlqMsZ0aKfnZCnL) .
    *   **E**: A set of pairs of vertices, known as **edges** or links, which represent the connections between them [[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFSdnHMc48A0RXDSaGR7uYqPx-Uhxgo_hfN1WKpl7Bw8NOpRWZiayHFknYT_GaHbDWib3csRDZlpkWUStiRDcxVa6srGWzjZcbXVA4cQEB8AHoTPNvbYhLyqpPZfJPOKMraYArulOCurMPv)[[6]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEcr3c0XTudZPL1QtyS14QuhtvJGgJR5v4UmVQ2aHHbTlQbeX_Ugyqiyy8ZgmYyYFGFPxNx2St3KkzMKBJqYGwBUk6FqwQo5zxRYj4d5vDKH-bQ6W0wFBkBJdI0wLpgxBSFYtuHYIEsSOaq) .
        *   In an **undirected graph**, edges are unordered pairs `{u, v}`, signifying a symmetric, bidirectional relationship [[7]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE60VfSnUS1ot-iwEkxexYKrNPjBz6mfp8UtHYK6mc7wucRMFTEVlcAWOO6aHDVolFKJULMKuw_JE_3hLgSkTBCAmFQ5FNa6bCPRQHgIEksHJbuJNT1Du47KeAcFvOrDMQQ-4KPQei0HqhtCMH-tw==)[[8]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEndNaG28SXNacLH17QS9pqySavcO2hH9TeCT2jxSBWhOWH0rkyN_chmPnypEQK5UurNt1gfX-OH_3jf3YxLLpLV2RZMu3QPw1zdjltpBNtKUk9Dfv-S4cpEPchGphi9wpu4I-AQSM0ro0OY2OjuA==) .
        *   In a **directed graph (digraph)**, edges are ordered pairs `(u, v)`, signifying an asymmetric, one-way connection from a tail `u` to a head `v` [[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFSdnHMc48A0RXDSaGR7uYqPx-Uhxgo_hfN1WKpl7Bw8NOpRWZiayHFknYT_GaHbDWib3csRDZlpkWUStiRDcxVa6srGWzjZcbXVA4cQEB8AHoTPNvbYhLyqpPZfJPOKMraYArulOCurMPv)[[9]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFsO4WhS1f4N-w5b5mab1nNu4utRQYad7KAwKHonlrhYtcWconvU58erIFXE0-L2W4PH-VenM_1yneFD7nLIElIB6st-LAGhsxp2oo1-MLqatwR8bsOevTexzb-b5NE65CU_sO8EjHo)[[10]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGZB3vI_rnOWf2LAECB2THdb3AT86xFa4nHH9AxUesuz5_3o0VJ8A7EwkgY4Z6eYvmtwrtxfyAQ2WN1OJw8HGZZNC5JgWdfnVDc-Ii-AIyZ2dDkXVNPE0vXNd6v-Wac2Sttsq_Yd7G_) .

*   **Path:** A path `P` is formally a sequence of vertices `P = (v₀, v₁, ..., vₖ)` where for each `i` from 0 to `k-1`, `(vᵢ, vᵢ₊₁)` is an edge in the graph [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFH2uPHm4Cv1OgtkW2rFVHwzj16EZ0jjSt_2GRqcJ19cTblE7yQhIAMp379_l03s28Dq-GnKLQYWGxT4McX_qZLLr9D54yodxlwFaB8cyuOYNBIfvaUmD5vimxUBjSFvv2evL31OqHx7-_P_6i_CBbWjnyADQkb61DsWg4wVQCsrPSav4UE) . A key characteristic of a simple path is that no vertices are repeated [[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFSdnHMc48A0RXDSaGR7uYqPx-Uhxgo_hfN1WKpl7Bw8NOpRWZiayHFknYT_GaHbDWib3csRDZlpkWUStiRDcxVa6srGWzjZcbXVA4cQEB8AHoTPNvbYhLyqpPZfJPOKMraYArulOCurMPv) .

#### Breadth-First Search (BFS)

At its core, the Weight-Limited BFS is built upon the standard Breadth-First Search algorithm [[11]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEuh7CHZLXCoRNSsxmgxfbxlQM0XKRO7hERkksgGoLr6axk3JA-RSihZvPAYvp2H90H_SKNzxXRSTFuaGTctmVXdPDlJPhyCsqb6yU_XlJIbMjes16PvEvRgWLfKaVoyVjTewn42az_w4VKnP36) . BFS is a graph traversal algorithm that explores all neighbor nodes at the present depth prior to moving on to the nodes at the next depth level [[12]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH9Vc9JbVS-fjKfcHaUgZ6tpu9651sq5LSLXNqJfxPkjG12lsoySIdsVqwaC4bRUhyXDCvQzmUF1TaKeWZ5hVHDRZRsLIj8RGfVq7T-TgTuIWdP82MfzJQjeBmQW-4kv8e-1rQQxUBL6QApjn8z)[[13]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEkBRoMGBy30AZblTj6Dzuvg1FGG90QrnMLUrwdXLF2kL4nsylR6nIKryYdUJ9QN67ll8spmm1ifC6ghDbVLdkqD1SlM_cmIo6QjMBL0UnijreCiSNqdVAjQxySUERMgH0VupcoOxbJShW8-amKLssUO1wdLFYAVgXY11M=) . It is used to find the shortest path in an **unweighted graph**, where "shortest" means the path with the fewest edges [[14]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFU19hEKwbXKFrzam2aVoMIFZeUIPWW4A60ROmAZ5xe-1QWGI_aWvBYEqvRtapd1Su6tnrRNyiHWJMc3C-WgtjhL4KLtWZ9Am1zLJU9v39fiyL5HN96oel-UWfkwyiuBDg=)[[15]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFG8y14FbZ3eKdX6CfK9Dj5HiHviObpp-O8yuNFwzlUfQH9foYLsh1BqtbXX-uSDcnKLOuXm5yKL7vQeQ65OqPOaYdpFLa_4SAuxeWhcVKm_5pgI0QYp9f_KbptUwFEZzhyzhEKXwFLfykJd-JjqE4gjP-O1Z_ErVPUjNBwtqWhMWrZPvA=)[[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFCq7oZKbu3Itgg4rGmiwBGCVrQyk6qE1yOCnVyh1NpluyIb9KNw7FpcqOyvPHv5OJrOOam69qu8PKCTh7D_r758BDnFXMGpCSDd0ke44Em1kX3YSlYz0e_mfomXMCYey0Jc4HKVXtx3YUuTdl5yqoduqgawXDogw5u6UKXCHD_hjErjJbdHad21Bdn1zl96XSqWqZ_qmvVFVf439u3tXUdLbgQ3hckrFcKlpzHWuzCZj90-KvzfzuaWTQlM9BVm5Oh8HtyTZ6g-c07) .

The key characteristics of a standard BFS are:
*   It uses a **First-In, First-Out (FIFO) queue** to manage the nodes to be visited [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFH2uPHm4Cv1OgtkW2rFVHwzj16EZ0jjSt_2GRqcJ19cTblE7yQhIAMp379_l03s28Dq-GnKLQYWGxT4McX_qZLLr9D54yodxlwFaB8cyuOYNBIfvaUmD5vimxUBjSFvv2evL31OqHx7-_P_6i_CBbWjnyADQkb61DsWg4wVQCsrPSav4UE)[[16]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGS1qK_xPExtF3Vv7zrQXeWMS-39MT7Xgx37b-EWn8k2cyxl0k0yI5TYPZxO8srPJXcm8sjFzq_mhnWjGf2gcO9_YsLmVBgXsqgOzknI1W_SFozjD-XeUI-pcZt_Vh2_Wmj5nS6It2geput)[[17]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFVCrKTnb08nSoXAuyb3qtptxhxet-LmG_MvCvYcuREH1owvr7gXpZBPNrSE480UoGa_eS5PVFFBGSg7u6R-du-2u1lUHJuKgaQXgclJSp0Junt0suIIImEOtho5YUq85g4BJMCxCE90o_ArQ==)[[18]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHRzUe9cp0C0BobDJvmKPopW66EiXf7o-EuIW7cDVQauUA8MgATeXvv73tTCFHd6qSwBVJ42iTWICGOYBEV3L8POmCZoMvVh48uRgQvkENGvRGnfS2jYnZJjvzYIAomopNYP0EzqMGJeYbPxR-D4Ujv1ML6tJJ3sijux9NKEsaSXWh_-hUX_hoYxERFvOUoJBU3-VKj3R1NHd9bfUD22Q==)[[10]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGZB3vI_rnOWf2LAECB2THdb3AT86xFa4nHH9AxUesuz5_3o0VJ8A7EwkgY4Z6eYvmtwrtxfyAQ2WN1OJw8HGZZNC5JgWdfnVDc-Ii-AIyZ2dDkXVNPE0vXNd6v-Wac2Sttsq_Yd7G_) .
*   It explores the graph layer by layer, starting from a source vertex `s` with `distance(s) = 0` [[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFSdnHMc48A0RXDSaGR7uYqPx-Uhxgo_hfN1WKpl7Bw8NOpRWZiayHFknYT_GaHbDWib3csRDZlpkWUStiRDcxVa6srGWzjZcbXVA4cQEB8AHoTPNvbYhLyqpPZfJPOKMraYArulOCurMPv)[[7]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE60VfSnUS1ot-iwEkxexYKrNPjBz6mfp8UtHYK6mc7wucRMFTEVlcAWOO6aHDVolFKJULMKuw_JE_3hLgSkTBCAmFQ5FNa6bCPRQHgIEksHJbuJNT1Du47KeAcFvOrDMQQ-4KPQei0HqhtCMH-tw==)[[15]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFG8y14FbZ3eKdX6CfK9Dj5HiHviObpp-O8yuNFwzlUfQH9foYLsh1BqtbXX-uSDcnKLOuXm5yKL7vQeQ65OqPOaYdpFLa_4SAuxeWhcVKm_5pgI0QYp9f_KbptUwFEZzhyzhEKXwFLfykJd-JjqE4gjP-O1Z_ErVPUjNBwtqWhMWrZPvA=) .
*   The FIFO queue guarantees that all vertices at a distance `d` are processed before any vertices at distance `d+1` are processed [[15]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFG8y14FbZ3eKdX6CfK9Dj5HiHviObpp-O8yuNFwzlUfQH9foYLsh1BqtbXX-uSDcnKLOuXm5yKL7vQeQ65OqPOaYdpFLa_4SAuxeWhcVKm_5pgI0QYp9f_KbptUwFEZzhyzhEKXwFLfykJd-JjqE4gjP-O1Z_ErVPUjNBwtqWhMWrZPvA=)[[9]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFsO4WhS1f4N-w5b5mab1nNu4utRQYad7KAwKHonlrhYtcWconvU58erIFXE0-L2W4PH-VenM_1yneFD7nLIElIB6st-LAGhsxp2oo1-MLqatwR8bsOevTexzb-b5NE65CU_sO8EjHo)[[16]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGS1qK_xPExtF3Vv7zrQXeWMS-39MT7Xgx37b-EWn8k2cyxl0k0yI5TYPZxO8srPJXcm8sjFzq_mhnWjGf2gcO9_YsLmVBgXsqgOzknI1W_SFozjD-XeUI-pcZt_Vh2_Wmj5nS6It2geput) . At any given time, the queue contains vertices from at most two adjacent levels (e.g., level `d` and `d+1`) [[15]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFG8y14FbZ3eKdX6CfK9Dj5HiHviObpp-O8yuNFwzlUfQH9foYLsh1BqtbXX-uSDcnKLOuXm5yKL7vQeQ65OqPOaYdpFLa_4SAuxeWhcVKm_5pgI0QYp9f_KbptUwFEZzhyzhEKXwFLfykJd-JjqE4gjP-O1Z_ErVPUjNBwtqWhMWrZPvA=) .
*   This layer-by-layer exploration ensures that when a vertex `v` is discovered from a predecessor `u`, the shortest path to it has been found [[7]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE60VfSnUS1ot-iwEkxexYKrNPjBz6mfp8UtHYK6mc7wucRMFTEVlcAWOO6aHDVolFKJULMKuw_JE_3hLgSkTBCAmFQ5FNa6bCPRQHgIEksHJbuJNT1Du47KeAcFvOrDMQQ-4KPQei0HqhtCMH-tw==)[[15]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFG8y14FbZ3eKdX6CfK9Dj5HiHviObpp-O8yuNFwzlUfQH9foYLsh1BqtbXX-uSDcnKLOuXm5yKL7vQeQ65OqPOaYdpFLa_4SAuxeWhcVKm_5pgI0QYp9f_KbptUwFEZzhyzhEKXwFLfykJd-JjqE4gjP-O1Z_ErVPUjNBwtqWhMWrZPvA=) . This relationship is defined by the equation:
    `distance(v) = distance(u) + 1` [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFH2uPHm4Cv1OgtkW2rFVHwzj16EZ0jjSt_2GRqcJ19cTblE7yQhIAMp379_l03s28Dq-GnKLQYWGxT4McX_qZLLr9D54yodxlwFaB8cyuOYNBIfvaUmD5vimxUBjSFvv2evL31OqHx7-_P_6i_CBbWjnyADQkb61DsWg4wVQCsrPSav4UE)[[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFSdnHMc48A0RXDSaGR7uYqPx-Uhxgo_hfN1WKpl7Bw8NOpRWZiayHFknYT_GaHbDWib3csRDZlpkWUStiRDcxVa6srGWzjZcbXVA4cQEB8AHoTPNvbYhLyqpPZfJPOKMraYArulOCurMPv)

### 2. Introducing Edge Weights and Path Cost

In many real-world scenarios, edges have varying costs (e.g., distance, time, energy), which are represented by edge weights [[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFSdnHMc48A0RXDSaGR7uYqPx-Uhxgo_hfN1WKpl7Bw8NOpRWZiayHFknYT_GaHbDWib3csRDZlpkWUStiRDcxVa6srGWzjZcbXVA4cQEB8AHoTPNvbYhLyqpPZfJPOKMraYArulOCurMPv) .

**Equation for Path Cost:**

The total cost of a path is the sum of the weights of all the edges in that path [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFH2uPHm4Cv1OgtkW2rFVHwzj16EZ0jjSt_2GRqcJ19cTblE7yQhIAMp379_l03s28Dq-GnKLQYWGxT4McX_qZLLr9D54yodxlwFaB8cyuOYNBIfvaUmD5vimxUBjSFvv2evL31OqHx7-_P_6i_CBbWjnyADQkb61DsWg4wVQCsrPSav4UE)[[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFSdnHMc48A0RXDSaGR7uYqPx-Uhxgo_hfN1WKpl7Bw8NOpRWZiayHFknYT_GaHbDWib3csRDZlpkWUStiRDcxVa6srGWzjZcbXVA4cQEB8AHoTPNvbYhLyqpPZfJPOKMraYArulOCurMPv) . For a path *P* from a starting node *s* to a node *n*, consisting of a sequence of vertices *(v₀, v₁, ..., vₖ)* where *v₀ = s* and *vₖ = n*, the path cost, denoted as *g(n)*, is calculated as:

`g(n) = Σᵢ₌₀ᵏ⁻¹ w(vᵢ, vᵢ₊₁)` [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFH2uPHm4Cv1OgtkW2rFVHwzj16EZ0jjSt_2GRqcJ19cTblE7yQhIAMp379_l03s28Dq-GnKLQYWGxT4McX_qZLLr9D54yodxlwFaB8cyuOYNBIfvaUmD5vimxUBjSFvv2evL31OqHx7-_P_6i_CBbWjnyADQkb61DsWg4wVQCsrPSav4UE)[[14]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFU19hEKwbXKFrzam2aVoMIFZeUIPWW4A60ROmAZ5xe-1QWGI_aWvBYEqvRtapd1Su6tnrRNyiHWJMc3C-WgtjhL4KLtWZ9Am1zLJU9v39fiyL5HN96oel-UWfkwyiuBDg=)

where *w(vᵢ, vᵢ₊₁)* is the weight of the edge between vertex *vᵢ* and *vᵢ₊₁* [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFH2uPHm4Cv1OgtkW2rFVHwzj16EZ0jjSt_2GRqcJ19cTblE7yQhIAMp379_l03s28Dq-GnKLQYWGxT4McX_qZLLr9D54yodxlwFaB8cyuOYNBIfvaUmD5vimxUBjSFvv2evL31OqHx7-_P_6i_CBbWjnyADQkb61DsWg4wVQCsrPSav4UE)[[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFSdnHMc48A0RXDSaGR7uYqPx-Uhxgo_hfN1WKpl7Bw8NOpRWZiayHFknYT_GaHbDWib3csRDZlpkWUStiRDcxVa6srGWzjZcbXVA4cQEB8AHoTPNvbYhLyqpPZfJPOKMraYArulOCurMPv) . In implementations, this is often calculated recursively:

`g(n) = g(parent(n)) + w(parent(n), n)`

### 3. The "Weight-Limited" Modification and Pseudocode

A "Weight-Limited" BFS introduces a constraint to the traversal: a path is only considered valid if its total accumulated cost is less than or equal to a predefined weight limit, *W*. The primary function of this limit is to **prune the search space** [[7]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE60VfSnUS1ot-iwEkxexYKrNPjBz6mfp8UtHYK6mc7wucRMFTEVlcAWOO6aHDVolFKJULMKuw_JE_3hLgSkTBCAmFQ5FNa6bCPRQHgIEksHJbuJNT1Du47KeAcFvOrDMQQ-4KPQei0HqhtCMH-tw==)[[19]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFhDA_04heAFJ1RWc10lKouRJ1-YMWZdo93vAyt8S7mmHpfGAu2ijdaqqt7MVd05Etct4277EAqy3r7C58Ba1Dc9BI9RQdYcaynRf1ghsHIoYE0d_LRghmz0abrcEmITtvLJ3fTJB0jEnylCg==) . Any path that exceeds the weight limit is discarded, which can significantly improve efficiency, especially in large graphs with many long or high-cost paths [[19]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFhDA_04heAFJ1RWc10lKouRJ1-YMWZdo93vAyt8S7mmHpfGAu2ijdaqqt7MVd05Etct4277EAqy3r7C58Ba1Dc9BI9RQdYcaynRf1ghsHIoYE0d_LRghmz0abrcEmITtvLJ3fTJB0jEnylCg==)[[20]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHGyGlhaRzNVynpnuJ6gQLsI2qaqqxVPdy2nFJs57YKc7fARCU7NSX_sYUbQgGeL8Ei7Ws7t3jlImhmb7C0Zz_IkWcsqRnoPW390l8QDcMdgKmYD6siTj0HUpucVvqExvjVaX6C95O--DNrsAWJRa72uw==) .

#### Pseudocode Implementation

This pseudocode illustrates how to manage the FIFO queue, track cumulative costs, and apply the weight-limit constraint to prune paths [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFH2uPHm4Cv1OgtkW2rFVHwzj16EZ0jjSt_2GRqcJ19cTblE7yQhIAMp379_l03s28Dq-GnKLQYWGxT4McX_qZLLr9D54yodxlwFaB8cyuOYNBIfvaUmD5vimxUBjSFvv2evL31OqHx7-_P_6i_CBbWjnyADQkb61DsWg4wVQCsrPSav4UE) .

```pseudocode
function WeightLimitedBFS(graph, startNode, goalNode, W):
  // A standard FIFO queue to store tuples of (node, path_to_node, cumulative_cost)
  queue = new FIFOQueue()

  // Add the starting node to the queue. Its path is just itself, and cost is 0.
  queue.enqueue((startNode, [startNode], 0))

  // A set to keep track of visited states (node, cost) to handle cycles
  // and prevent exploring less efficient paths to the same node.
  visited = new Set()
  visited.add((startNode, 0))

  while queue is not empty:
    // Dequeue the next path to explore based on FIFO order
    (currentNode, path, currentCost) = queue.dequeue()

    // If the current node is the goal, a valid path has been found.
    if currentNode == goalNode:
      return path, currentCost // Return the first path found within the limit

    // Explore neighbors of the current node
    for each neighbor, edgeWeight in graph.neighbors(currentNode):
      newCost = currentCost + edgeWeight

      // PRUNING STEP: Check if the new path's cost is within the weight limit W
      if newCost <= W:
        // Check if this state (neighbor, newCost) has been visited to avoid cycles
        if (neighbor, newCost) not in visited:
          // Create the new path by extending the current one
          newPath = path + [neighbor]

          // Add the new state to the queue and visited set for future exploration
          visited.add((neighbor, newCost))
          queue.enqueue((neighbor, newPath, newCost))

  // If the queue becomes empty and the goal was not reached, no path exists within the limit W
  return "No path found within the weight limit."
```

### 4. Illustrative Example: Suboptimality in Action

To demonstrate the algorithm's behavior, consider the following weighted graph. The goal is to find a path from **S** to **G** with a maximum total weight of **W = 15**.

*   **Nodes**: S (Start), A, B, C, D, G (Goal)
*   **Edges (with weights)**:
    *   S → A (2)
    *   S → B (8)
    *   A → C (7)
    *   A → D (3)
    *   B → G (2)
    *   D → G (4)
*   **Optimal Path**: S → A → D → G (Cost: 2 + 3 + 4 = 9)
*   **Suboptimal Path**: S → B → G (Cost: 8 + 2 = 10)

#### Weight-Limited BFS Walkthrough (W = 15)

This algorithm explores based on the number of edges (level-by-level) due to its FIFO queue [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFH2uPHm4Cv1OgtkW2rFVHwzj16EZ0jjSt_2GRqcJ19cTblE7yQhIAMp379_l03s28Dq-GnKLQYWGxT4McX_qZLLr9D54yodxlwFaB8cyuOYNBIfvaUmD5vimxUBjSFvv2evL31OqHx7-_P_6i_CBbWjnyADQkb61DsWg4wVQCsrPSav4UE)[[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFSdnHMc48A0RXDSaGR7uYqPx-Uhxgo_hfN1WKpl7Bw8NOpRWZiayHFknYT_GaHbDWib3csRDZlpkWUStiRDcxVa6srGWzjZcbXVA4cQEB8AHoTPNvbYhLyqpPZfJPOKMraYArulOCurMPv) .

| Step | Action | Queue State (Node, Cost) | Notes |
| :--- | :--- | :--- | :--- |
| **1** | Initialize with start node S. | `[(S, 0)]` | |
| **2** | Dequeue S. Discover neighbors A and B. Both paths are within the limit W=15. Enqueue A (cost 2), then B (cost 8). | `[(A, 2), (B, 8)]` | FIFO order is maintained. |
| **3** | Dequeue A (front of queue). Discover neighbors C and D. Enqueue C (cost 2+7=9), then D (cost 2+3=5). | `[(B, 8), (C, 9), (D, 5)]` | A is processed before B, despite B's path being shorter in edges to the goal. |
| **4** | Dequeue B. Discover neighbor G. Path cost is 8+2=10, which is ≤ 15. | `[(C, 9), (D, 5)]` | **Goal G is found!** |

**Result:** The algorithm terminates and returns the path **S → B → G** with a cost of **10**. It finds a valid path within the weight limit but fails to find the optimal one. Its cost-unaware, level-by-level exploration led it to expand the S→B branch, which offered a path to G with fewer edges (2) than the optimal path through S→A→D (3 edges).

#### Dijkstra's Algorithm Walkthrough (For Comparison)

Dijkstra's uses a **priority queue** to always expand the node with the lowest cumulative cost, guaranteeing optimality [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFH2uPHm4Cv1OgtkW2rFVHwzj16EZ0jjSt_2GRqcJ19cTblE7yQhIAMp379_l03s28Dq-GnKLQYWGxT4McX_qZLLr9D54yodxlwFaB8cyuOYNBIfvaUmD5vimxUBjSFvv2evL31OqHx7-_P_6i_CBbWjnyADQkb61DsWg4wVQCsrPSav4UE)[[15]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFG8y14FbZ3eKdX6CfK9Dj5HiHviObpp-O8yuNFwzlUfQH9foYLsh1BqtbXX-uSDcnKLOuXm5yKL7vQeQ65OqPOaYdpFLa_4SAuxeWhcVKm_5pgI0QYp9f_KbptUwFEZzhyzhEKXwFLfykJd-JjqE4gjP-O1Z_ErVPUjNBwtqWhMWrZPvA=)[[21]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEF3HlgXrLB5y_XSyNYdQjptU5lntKjlAiM2e6OOPAmO92gHBvrBAyTev8hwqzU4SmIXGwzWZoFshwL1k9xeb8hWaXrqLahz1FNhUwVfEmMhQ8xDfn4PfJp_xGvV_6AYZ-d7X3tt4WLTP7Gn4Bvi8OWZg==) .

| Step | Action | Priority Queue (Cost, Node) | Distances |
| :--- | :--- | :--- | :--- |
| **1** | Initialize with start node S. | `[(0, S)]` | `{S: 0}` |
| **2** | Pop S (cost 0). Discover A (cost 2) and B (cost 8). Push both to PQ. | `[(2, A), (8, B)]` | `{S:0, A:2, B:8}` |
| **3** | Pop A (cost 2, lowest in PQ). Discover C (cost 2+7=9) and D (cost 2+3=5). Push both. | `[(5, D), (8, B), (9, C)]` | `{S:0, A:2, B:8, C:9, D:5}` |
| **4** | Pop D (cost 5, now lowest). Discover G (cost 5+4=9). Push to PQ. | `[(8, B), (9, C), (9, G)]` | `{S:0, A:2, B:8, C:9, D:5, G:9}` |
| **5** | Pop B (cost 8). Discover G. New cost to G is 8+2=10. Since 10 > 9, the distance to G is not updated. | `[(9, C), (9, G)]` | No change. |
| **6** | Pop C (cost 9). No new paths. | `[(9, G)]` | No change. |
| **7** | Pop G (cost 9). | Empty | **Goal G is found!** |

**Result:** Dijkstra's algorithm terminates and returns the path **S → A → D → G** with a cost of **9**. By always expanding the lowest-cost path on the frontier, it correctly identifies the optimal solution [[15]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFG8y14FbZ3eKdX6CfK9Dj5HiHviObpp-O8yuNFwzlUfQH9foYLsh1BqtbXX-uSDcnKLOuXm5yKL7vQeQ65OqPOaYdpFLa_4SAuxeWhcVKm_5pgI0QYp9f_KbptUwFEZzhyzhEKXwFLfykJd-JjqE4gjP-O1Z_ErVPUjNBwtqWhMWrZPvA=) .

### 5. Key Equations and Concepts for Your Presentation

Here are the essential equations and concepts to feature in your presentation.

*   **Graph Definition:** The formal definition of a graph `G`.
    `G = (V, E)`
    *   **V**: Set of vertices [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFH2uPHm4Cv1OgtkW2rFVHwzj16EZ0jjSt_2GRqcJ19cTblE7yQhIAMp379_l03s28Dq-GnKLQYWGxT4McX_qZLLr9D54yodxlwFaB8cyuOYNBIfvaUmD5vimxUBjSFvv2evL31OqHx7-_P_6i_CBbWjnyADQkb61DsWg4wVQCsrPSav4UE)[[5]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQF-6yuHO7Emw95BGch7txubToPuXCVycG1uNmyvMq_zxXM4T4VPrXrRQbqXOzL6CK5gNcKTTPognQvwb5PDw_CDFfQrgiPWVPD4ejszde3pnPj1ARoGq5vagy9tzP25vIPXG0Wy1ZQS9CyMhZZbPMLYEglqUJPIdM51JHlqMsZ0aKfnZCnL) .
    *   **E**: Set of edges [[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFSdnHMc48A0RXDSaGR7uYqPx-Uhxgo_hfN1WKpl7Bw8NOpRWZiayHFknYT_GaHbDWib3csRDZlpkWUStiRDcxVa6srGWzjZcbXVA4cQEB8AHoTPNvbYhLyqpPZfJPOKMraYArulOCurMPv)[[6]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEcr3c0XTudZPL1QtyS14QuhtvJGgJR5v4UmVQ2aHHbTlQbeX_Ugyqiyy8ZgmYyYFGFPxNx2St3KkzMKBJqYGwBUk6FqwQo5zxRYj4d5vDKH-bQ6W0wFBkBJdI0wLpgxBSFYtuHYIEsSOaq) .

*   **BFS Distance (Unweighted):** The foundational distance relationship in standard BFS.
    `distance(v) = distance(u) + 1` [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFH2uPHm4Cv1OgtkW2rFVHwzj16EZ0jjSt_2GRqcJ19cTblE7yQhIAMp379_l03s28Dq-GnKLQYWGxT4McX_qZLLr9D54yodxlwFaB8cyuOYNBIfvaUmD5vimxUBjSFvv2evL31OqHx7-_P_6i_CBbWjnyADQkb61DsWg4wVQCsrPSav4UE)[[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFSdnHMc48A0RXDSaGR7uYqPx-Uhxgo_hfN1WKpl7Bw8NOpRWZiayHFknYT_GaHbDWib3csRDZlpkWUStiRDcxVa6srGWzjZcbXVA4cQEB8AHoTPNvbYhLyqpPZfJPOKMraYArulOCurMPv)

*   **Path Cost (g(n)):** The cost of the path from the start node to node *n*.
    *   **Formal Definition:** `g(n) = Σᵢ₌₀ᵏ⁻¹ w(vᵢ, vᵢ₊₁)` [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFH2uPHm4Cv1OgtkW2rFVHwzj16EZ0jjSt_2GRqcJ19cTblE7yQhIAMp379_l03s28Dq-GnKLQYWGxT4McX_qZLLr9D54yodxlwFaB8cyuOYNBIfvaUmD5vimxUBjSFvv2evL31OqHx7-_P_6i_CBbWjnyADQkb61DsWg4wVQCsrPSav4UE)[[14]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFU19hEKwbXKFrzam2aVoMIFZeUIPWW4A60ROmAZ5xe-1QWGI_aWvBYEqvRtapd1Su6tnrRNyiHWJMc3C-WgtjhL4KLtWZ9Am1zLJU9v39fiyL5HN96oel-UWfkwyiuBDg=)
    *   **Recursive Definition:** `g(n) = g(parent(n)) + w(parent(n), n)`

*   **Weight-Limit Constraint (Pruning Condition):** The fundamental rule of the algorithm. For a path to a neighbor node *v* from a current node *u*:
    `g(u) + w(u, v) ≤ W`
    where *W* is the maximum allowed path weight [[7]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE60VfSnUS1ot-iwEkxexYKrNPjBz6mfp8UtHYK6mc7wucRMFTEVlcAWOO6aHDVolFKJULMKuw_JE_3hLgSkTBCAmFQ5FNa6bCPRQHgIEksHJbuJNT1Du47KeAcFvOrDMQQ-4KPQei0HqhtCMH-tw==) .

*   **Complexity Analysis:**
    *   **Worst-Case Time Complexity:** `O(V + E)`, where every vertex and edge is visited once [[9]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFsO4WhS1f4N-w5b5mab1nNu4utRQYad7KAwKHonlrhYtcWconvU58erIFXE0-L2W4PH-VenM_1yneFD7nLIElIB6st-LAGhsxp2oo1-MLqatwR8bsOevTexzb-b5NE65CU_sO8EjHo)[[16]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGS1qK_xPExtF3Vv7zrQXeWMS-39MT7Xgx37b-EWn8k2cyxl0k0yI5TYPZxO8srPJXcm8sjFzq_mhnWjGf2gcO9_YsLmVBgXsqgOzknI1W_SFozjD-XeUI-pcZt_Vh2_Wmj5nS6It2geput)[[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFCq7oZKbu3Itgg4rGmiwBGCVrQyk6qE1yOCnVyh1NpluyIb9KNw7FpcqOyvPHv5OJrOOam69qu8PKCTh7D_r758BDnFXMGpCSDd0ke44Em1kX3YSlYz0e_mfomXMCYey0Jc4HKVXtx3YUuTdl5yqoduqgawXDogw5u6UKXCHD_hjErjJbdHad21Bdn1zl96XSqWqZ_qmvVFVf439u3tXUdLbgQ3hckrFcKlpzHWuzCZj90-KvzfzuaWTQlM9BVm5Oh8HtyTZ6g-c07) .
    *   **Practical Time Complexity:** The performance is highly dependent on how effectively the weight limit `W` prunes the search space [[12]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH9Vc9JbVS-fjKfcHaUgZ6tpu9651sq5LSLXNqJfxPkjG12lsoySIdsVqwaC4bRUhyXDCvQzmUF1TaKeWZ5hVHDRZRsLIj8RGfVq7T-TgTuIWdP82MfzJQjeBmQW-4kv8e-1rQQxUBL6QApjn8z)[[19]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFhDA_04heAFJ1RWc10lKouRJ1-YMWZdo93vAyt8S7mmHpfGAu2ijdaqqt7MVd05Etct4277EAqy3r7C58Ba1Dc9BI9RQdYcaynRf1ghsHIoYE0d_LRghmz0abrcEmITtvLJ3fTJB0jEnylCg==)[[20]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHGyGlhaRzNVynpnuJ6gQLsI2qaqqxVPdy2nFJs57YKc7fARCU7NSX_sYUbQgGeL8Ei7Ws7t3jlImhmb7C0Zz_IkWcsqRnoPW390l8QDcMdgKmYD6siTj0HUpucVvqExvjVaX6C95O--DNrsAWJRa72uw==) . It is more accurately described as `O(V' + E')`, where `V'` and `E'` are the vertices and edges reachable within the cost limit `W`.
        *   **Tight `W` / Large Edge Weights:** If `W` is small or edge weights are large, many paths are pruned early. The algorithm explores a small fraction of the graph, making performance significantly faster than the worst case [[12]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH9Vc9JbVS-fjKfcHaUgZ6tpu9651sq5LSLXNqJfxPkjG12lsoySIdsVqwaC4bRUhyXDCvQzmUF1TaKeWZ5hVHDRZRsLIj8RGfVq7T-TgTuIWdP82MfzJQjeBmQW-4kv8e-1rQQxUBL6QApjn8z) .
        *   **Loose `W` / Small Edge Weights:** If `W` is large or edge weights are uniformly small, the pruning condition is rarely met. The algorithm's behavior approaches that of a standard BFS, and its performance will be close to the `O(V + E)` upper bound.
    *   **Space Complexity:** `O(b^d)`, where 'b' is the branching factor and 'd' is the maximum depth (number of edges) of a path whose cost is under `W` [[17]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFVCrKTnb08nSoXAuyb3qtptxhxet-LmG_MvCvYcuREH1owvr7gXpZBPNrSE480UoGa_eS5PVFFBGSg7u6R-du-2u1lUHJuKgaQXgclJSp0Junt0suIIImEOtho5YUq85g4BJMCxCE90o_ArQ==)[[21]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEF3HlgXrLB5y_XSyNYdQjptU5lntKjlAiM2e6OOPAmO92gHBvrBAyTev8hwqzU4SmIXGwzWZoFshwL1k9xeb8hWaXrqLahz1FNhUwVfEmMhQ8xDfn4PfJp_xGvV_6AYZ-d7X3tt4WLTP7Gn4Bvi8OWZg==)[[22]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGkOsaScttRMzWxSIuTwTnWse8vG63S_MrEJtaWapA9CQUOe83viBMBJ0zhgDLxG9fzbCTaHVZ6yxaU1l4OcpGLW2zQA46Z5ReZxy9Zo_02yjUAdx8-Wol6G9Z7-rgYuRwq0mrApuDFnLgr4Q==)[[23]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFo4ri3s0wR8mZRUn4Rzxq5T6rxgT1XHUf8zuN4OdMMJmBSh9F3_f_lDQlQjHy4h5TqUcwV8r5PvneYn4hAjaNPZpWifYVOIjaxde77adi9TKUFvm-XQk3k5iKB8J4TxxxUPWjVduWn)[[24]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQF-ll8QBanzpKwocpKg8WK-NDaaxO5K4CNGWdnfluSf_pCYX-fQlv5ssmovopG8z0iRZA0w_E5hRmmj_EFFSd7UjA1LBVmolv4-ifrnaSdqfPbTCg5_t021Ah7NJPYxrZZe3nXzq-ZJNSuoWFT7euK_vUQsleFhYDeBuTOJTWdi) . This exponential complexity is a major consideration, as memory usage can grow rapidly [[18]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHRzUe9cp0C0BobDJvmKPopW66EiXf7o-EuIW7cDVQauUA8MgATeXvv73tTCFHd6qSwBVJ42iTWICGOYBEV3L8POmCZoMvVh48uRgQvkENGvRGnfS2jYnZJjvzYIAomopNYP0EzqMGJeYbPxR-D4Ujv1ML6tJJ3sijux9NKEsaSXWh_-hUX_hoYxERFvOUoJBU3-VKj3R1NHd9bfUD22Q==)[[19]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFhDA_04heAFJ1RWc10lKouRJ1-YMWZdo93vAyt8S7mmHpfGAu2ijdaqqt7MVd05Etct4277EAqy3r7C58Ba1Dc9BI9RQdYcaynRf1ghsHIoYE0d_LRghmz0abrcEmITtvLJ3fTJB0jEnylCg==)[[20]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHGyGlhaRzNVynpnuJ6gQLsI2qaqqxVPdy2nFJs57YKc7fARCU7NSX_sYUbQgGeL8Ei7Ws7t3jlImhmb7C0Zz_IkWcsqRnoPW390l8QDcMdgKmYD6siTj0HUpucVvqExvjVaX6C95O--DNrsAWJRa72uw==) .

### 6. Comparison with Related Algorithms

*   **Uniform-Cost Search (UCS) / Dijkstra's Algorithm:** UCS and Dijkstra's are optimal for finding the least-cost path in graphs with non-negative edge weights [[16]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGS1qK_xPExtF3Vv7zrQXeWMS-39MT7Xgx37b-EWn8k2cyxl0k0yI5TYPZxO8srPJXcm8sjFzq_mhnWjGf2gcO9_YsLmVBgXsqgOzknI1W_SFozjD-XeUI-pcZt_Vh2_Wmj5nS6It2geput)[[17]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFVCrKTnb08nSoXAuyb3qtptxhxet-LmG_MvCvYcuREH1owvr7gXpZBPNrSE480UoGa_eS5PVFFBGSg7u6R-du-2u1lUHJuKgaQXgclJSp0Junt0suIIImEOtho5YUq85g4BJMCxCE90o_ArQ==)[[21]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEF3HlgXrLB5y_XSyNYdQjptU5lntKjlAiM2e6OOPAmO92gHBvrBAyTev8hwqzU4SmIXGwzWZoFshwL1k9xeb8hWaXrqLahz1FNhUwVfEmMhQ8xDfn4PfJp_xGvV_6AYZ-d7X3tt4WLTP7Gn4Bvi8OWZg==)[[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFCq7oZKbu3Itgg4rGmiwBGCVrQyk6qE1yOCnVyh1NpluyIb9KNw7FpcqOyvPHv5OJrOOam69qu8PKCTh7D_r758BDnFXMGpCSDd0ke44Em1kX3YSlYz0e_mfomXMCYey0Jc4HKVXtx3YUuTdl5yqoduqgawXDogw5u6UKXCHD_hjErjJbdHad21Bdn1zl96XSqWqZ_qmvVFVf439u3tXUdLbgQ3hckrFcKlpzHWuzCZj90-KvzfzuaWTQlM9BVm5Oh8HtyTZ6g-c07)[[23]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFo4ri3s0wR8mZRUn4Rzxq5T6rxgT1XHUf8zuN4OdMMJmBSh9F3_f_lDQlQjHy4h5TqUcwV8r5PvneYn4hAjaNPZpWifYVOIjaxde77adi9TKUFvm-XQk3k5iKB8J4TxxxUPWjVduWn) . They achieve this by using a **priority queue** to always expand the node with the lowest path cost `g(n)` [[9]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFsO4WhS1f4N-w5b5mab1nNu4utRQYad7KAwKHonlrhYtcWconvU58erIFXE0-L2W4PH-VenM_1yneFD7nLIElIB6st-LAGhsxp2oo1-MLqatwR8bsOevTexzb-b5NE65CU_sO8EjHo)[[12]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH9Vc9JbVS-fjKfcHaUgZ6tpu9651sq5LSLXNqJfxPkjG12lsoySIdsVqwaC4bRUhyXDCvQzmUF1TaKeWZ5hVHDRZRsLIj8RGfVq7T-TgTuIWdP82MfzJQjeBmQW-4kv8e-1rQQxUBL6QApjn8z)[[3]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEy-Rry1SfoDZ6ik11DA4QCM6UWDWt6pM1ungO0bmRs84RCGSmst_HtAdQDBI9x4ofCOkP9bpjBaH4eGk4AY-48nG2ZBJDAs6svnaAyKbQpdde8sTr-Bkg5VySWkyzKWFlBq9NCO_JiKs1oub7sOGx3QqPY58I=)[[25]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFdu-ggwfNE7YKV2jtQXItHWI4Qtw2KVIVgnb3E1Ruv55CnDzTp4iSJEqo2aTGIbHK7iCvZboezp5haj5njfJqAvOj5z_pcRirCl_NTf5ZhfNRhbvlPXHINvyUkWvurVrtCOQf842UVa0GcgRQ3sU5LqyaJAd_2TXP__zvYCHlnXsbc5UUjhwdO48BzfszAYj2Naa20fCYZl7EtNJH2fNfIv67k0IqCfMlT89_NlV-2QRcsOiAEzep06_2-XeGrEsySDBYxYV0JG7ja) . As demonstrated in the walkthrough, this greedy choice guarantees optimality [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFH2uPHm4Cv1OgtkW2rFVHwzj16EZ0jjSt_2GRqcJ19cTblE7yQhIAMp379_l03s28Dq-GnKLQYWGxT4McX_qZLLr9D54yodxlwFaB8cyuOYNBIfvaUmD5vimxUBjSFvv2evL31OqHx7-_P_6i_CBbWjnyADQkb61DsWg4wVQCsrPSav4UE)[[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFSdnHMc48A0RXDSaGR7uYqPx-Uhxgo_hfN1WKpl7Bw8NOpRWZiayHFknYT_GaHbDWib3csRDZlpkWUStiRDcxVa6srGWzjZcbXVA4cQEB8AHoTPNvbYhLyqpPZfJPOKMraYArulOCurMPv)[[15]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFG8y14FbZ3eKdX6CfK9Dj5HiHviObpp-O8yuNFwzlUfQH9foYLsh1BqtbXX-uSDcnKLOuXm5yKL7vQeQ65OqPOaYdpFLa_4SAuxeWhcVKm_5pgI0QYp9f_KbptUwFEZzhyzhEKXwFLfykJd-JjqE4gjP-O1Z_ErVPUjNBwtqWhMWrZPvA=)[[19]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFhDA_04heAFJ1RWc10lKouRJ1-YMWZdo93vAyt8S7mmHpfGAu2ijdaqqt7MVd05Etct4277EAqy3r7C58Ba1Dc9BI9RQdYcaynRf1ghsHIoYE0d_LRghmz0abrcEmITtvLJ3fTJB0jEnylCg==) . A Weight-Limited BFS lacks this guarantee because its FIFO queue is not cost-aware [[17]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFVCrKTnb08nSoXAuyb3qtptxhxet-LmG_MvCvYcuREH1owvr7gXpZBPNrSE480UoGa_eS5PVFFBGSg7u6R-du-2u1lUHJuKgaQXgclJSp0Junt0suIIImEOtho5YUq85g4BJMCxCE90o_ArQ==)[[11]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEuh7CHZLXCoRNSsxmgxfbxlQM0XKRO7hERkksgGoLr6axk3JA-RSihZvPAYvp2H90H_SKNzxXRSTFuaGTctmVXdPDlJPhyCsqb6yU_XlJIbMjes16PvEvRgWLfKaVoyVjTewn42az_w4VKnP36) .

*   **A\* Search:** The A\* algorithm is an informed search that uses a heuristic function, *h(n)*, to estimate the cost to the goal. Its evaluation function is *f(n) = g(n) + h(n)*. Weight-Limited BFS is an uninformed search as it does not use a heuristic to guide it toward a goal.

*   **Depth-Limited Search (DLS):** DLS is a modification of Depth-First Search (DFS) that imposes a hard limit on the depth (number of edges) of the search path [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFCq7oZKbu3Itgg4rGmiwBGCVrQyk6qE1yOCnVyh1NpluyIb9KNw7FpcqOyvPHv5OJrOOam69qu8PKCTh7D_r758BDnFXMGpCSDd0ke44Em1kX3YSlYz0e_mfomXMCYey0Jc4HKVXtx3YUuTdl5yqoduqgawXDogw5u6UKXCHD_hjErjJbdHad21Bdn1zl96XSqWqZ_qmvVFVf439u3tXUdLbgQ3hckrFcKlpzHWuzCZj90-KvzfzuaWTQlM9BVm5Oh8HtyTZ6g-c07)[[25]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFdu-ggwfNE7YKV2jtQXItHWI4Qtw2KVIVgnb3E1Ruv55CnDzTp4iSJEqo2aTGIbHK7iCvZboezp5haj5njfJqAvOj5z_pcRirCl_NTf5ZhfNRhbvlPXHINvyUkWvurVrtCOQf842UVa0GcgRQ3sU5LqyaJAd_2TXP__zvYCHlnXsbc5UUjhwdO48BzfszAYj2Naa20fCYZl7EtNJH2fNfIv67k0IqCfMlT89_NlV-2QRcsOiAEzep06_2-XeGrEsySDBYxYV0JG7ja) . This is conceptually similar to Weight-Limited BFS, but DLS limits the *number of edges*, while Weight-Limited BFS limits the *sum of edge weights*.

*   **Iterative Lengthening Search (ILS):** ILS is a strategy that finds the least-cost path by performing a series of cost-limited searches with an incrementally increasing cost limit [[6]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEcr3c0XTudZPL1QtyS14QuhtvJGgJR5v4UmVQ2aHHbTlQbeX_Ugyqiyy8ZgmYyYFGFPxNx2St3KkzMKBJqYGwBUk6FqwQo5zxRYj4d5vDKH-bQ6W0wFBkBJdI0wLpgxBSFYtuHYIEsSOaq)[[24]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQF-ll8QBanzpKwocpKg8WK-NDaaxO5K4CNGWdnfluSf_pCYX-fQlv5ssmovopG8z0iRZA0w_E5hRmmj_EFFSd7UjA1LBVmolv4-ifrnaSdqfPbTCg5_t021Ah7NJPYxrZZe3nXzq-ZJNSuoWFT7euK_vUQsleFhYDeBuTOJTWdi) .
    1.  It starts with an initial cost limit, `L` [[8]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEndNaG28SXNacLH17QS9pqySavcO2hH9TeCT2jxSBWhOWH0rkyN_chmPnypEQK5UurNt1gfX-OH_3jf3YxLLpLV2RZMu3QPw1zdjltpBNtKUk9Dfv-S4cpEPchGphi9wpu4I-AQSM0ro0OY2OjuA==) .
    2.  It performs a search, exploring only paths with a cost `g(n) ≤ L`.
    3.  If no solution is found, it identifies the smallest path cost among all paths that were pruned for exceeding the limit `L`. This becomes the new, larger cost limit for the next iteration [[8]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEndNaG28SXNacLH17QS9pqySavcO2hH9TeCT2jxSBWhOWH0rkyN_chmPnypEQK5UurNt1gfX-OH_3jf3YxLLpLV2RZMu3QPw1zdjltpBNtKUk9Dfv-S4cpEPchGphi9wpu4I-AQSM0ro0OY2OjuA==)[[26]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEfDwZ_Fm0ZApuh2y7uSx2aBit1zlF9mFeNcV4n__1EFz0g_KNlT7Iu7p1IY4qOE80WArYFJoL6OVVueHR-aPtOIM66h7mjeSKc3Wez7Kj5xeo--6AvGHiaRlQdYY0eIEeLlMlXqyUn1Ieta-dJ8ah8QH2gPAcHl-2ZPreiUkXFLM3fEoqBjBTGnMEdMlgoWGBd78Ir0FAI) .
    4.  This process repeats until a solution is found, which is guaranteed to be optimal [[26]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEfDwZ_Fm0ZApuh2y7uSx2aBit1zlF9mFeNcV4n__1EFz0g_KNlT7Iu7p1IY4qOE80WArYFJoL6OVVueHR-aPtOIM66h7mjeSKc3Wez7Kj5xeo--6AvGHiaRlQdYY0eIEeLlMlXqyUn1Ieta-dJ8ah8QH2gPAcHl-2ZPreiUkXFLM3fEoqBjBTGnMEdMlgoWGBd78Ir0FAI) .
    Each iteration of ILS is functionally equivalent to running a Weight-Limited BFS with a specific threshold `W` [[27]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHyPellPNr6bJEZ-4-uXGwVOq3k-pOvoiLE0TEnySzzdiWutkuz2GAD2E5OJT1gixHTtttjBQlUEYq3MgeerXogzSypRCpWLhj6Icw55KNKVv1cqJaGWM0RXYWX_Ssp_no=) .

### 7. Practical Consideration: Iteration Limits

In graphs with cycles or in very large, dense graphs, a search could still become computationally expensive even with a weight limit [[19]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFhDA_04heAFJ1RWc10lKouRJ1-YMWZdo93vAyt8S7mmHpfGAu2ijdaqqt7MVd05Etct4277EAqy3r7C58Ba1Dc9BI9RQdYcaynRf1ghsHIoYE0d_LRghmz0abrcEmITtvLJ3fTJB0jEnylCg==)[[28]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHrmNHxdV7Rp2f8G0BqDm5ZWcKZr57raL2Dn8s3qQJ8dWVOko0G51To0ORAT6pPKAPjYC42WQMoQqeXMiWoXdnMUtB2xEqYWPQTNBmjPTMfAPnariHOB2JVHme8BOQHvSTRKqybmA9DARM7EQ==) . If edge weights are very small, the algorithm might explore a vast number of nodes before the cumulative weight `g(n)` exceeds the limit `W` [[20]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHGyGlhaRzNVynpnuJ6gQLsI2qaqqxVPdy2nFJs57YKc7fARCU7NSX_sYUbQgGeL8Ei7Ws7t3jlImhmb7C0Zz_IkWcsqRnoPW390l8QDcMdgKmYD6siTj0HUpucVvqExvjVaX6C95O--DNrsAWJRa72uw==) . To ensure termination and manage resources, a practical implementation might include a secondary termination condition, such as a `max_runs` or iteration limit [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFCq7oZKbu3Itgg4rGmiwBGCVrQyk6qE1yOCnVyh1NpluyIb9KNw7FpcqOyvPHv5OJrOOam69qu8PKCTh7D_r758BDnFXMGpCSDd0ke44Em1kX3YSlYz0e_mfomXMCYey0Jc4HKVXtx3YUuTdl5yqoduqgawXDogw5u6UKXCHD_hjErjJbdHad21Bdn1zl96XSqWqZ_qmvVFVf439u3tXUdLbgQ3hckrFcKlpzHWuzCZj90-KvzfzuaWTQlM9BVm5Oh8HtyTZ6g-c07)[[28]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHrmNHxdV7Rp2f8G0BqDm5ZWcKZr57raL2Dn8s3qQJ8dWVOko0G51To0ORAT6pPKAPjYC42WQMoQqeXMiWoXdnMUtB2xEqYWPQTNBmjPTMfAPnariHOB2JVHme8BOQHvSTRKqybmA9DARM7EQ==) . This acts as a safeguard against infinite loops (e.g., with zero-weight cycles) and caps total computation [[19]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFhDA_04heAFJ1RWc10lKouRJ1-YMWZdo93vAyt8S7mmHpfGAu2ijdaqqt7MVd05Etct4277EAqy3r7C58Ba1Dc9BI9RQdYcaynRf1ghsHIoYE0d_LRghmz0abrcEmITtvLJ3fTJB0jEnylCg==)[[28]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHrmNHxdV7Rp2f8G0BqDm5ZWcKZr57raL2Dn8s3qQJ8dWVOko0G51To0ORAT6pPKAPjYC42WQMoQqeXMiWoXdnMUtB2xEqYWPQTNBmjPTMfAPnariHOB2JVHme8BOQHvSTRKqybmA9DARM7EQ==) .

### Executive Summary

The Weight-Limited Breadth-First Search is a practical, though not formally named, modification of BFS for weighted graphs with a cost constraint. It explores the graph in a layer-by-layer fashion based on the number of edges, but it incorporates the concept of path cost, `g(n)`, from algorithms like UCS/Dijkstra's.

Its defining feature is the **weight-limit constraint**, `g(n) ≤ W`, which is used to prune any search path whose cumulative weight exceeds a predefined limit `W` [[7]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE60VfSnUS1ot-iwEkxexYKrNPjBz6mfp8UtHYK6mc7wucRMFTEVlcAWOO6aHDVolFKJULMKuw_JE_3hLgSkTBCAmFQ5FNa6bCPRQHgIEksHJbuJNT1Du47KeAcFvOrDMQQ-4KPQei0HqhtCMH-tw==) . This makes it a useful approach for problems where solutions are only valid if they are within a certain "budget," such as finding all locations reachable within a specific travel time.

A critical point for your presentation is that due to its reliance on a FIFO queue, this algorithm is **not optimal** [[11]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEuh7CHZLXCoRNSsxmgxfbxlQM0XKRO7hERkksgGoLr6axk3JA-RSihZvPAYvp2H90H_SKNzxXRSTFuaGTctmVXdPDlJPhyCsqb6yU_XlJIbMjes16PvEvRgWLfKaVoyVjTewn42az_w4VKnP36) . It explores based on the number of edges, not cost, and does not guarantee finding the least-cost path within the given weight limit [[17]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFVCrKTnb08nSoXAuyb3qtptxhxet-LmG_MvCvYcuREH1owvr7gXpZBPNrSE480UoGa_eS5PVFFBGSg7u6R-du-2u1lUHJuKgaQXgclJSp0Junt0suIIImEOtho5YUq85g4BJMCxCE90o_ArQ==) . This was clearly demonstrated in the walkthrough, which contrasts its behavior with Dijkstra's algorithm. Dijkstra's uses a priority queue to guarantee optimality by always expanding the lowest-cost path on the frontier [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFH2uPHm4Cv1OgtkW2rFVHwzj16EZ0jjSt_2GRqcJ19cTblE7yQhIAMp379_l03s28Dq-GnKLQYWGxT4McX_qZLLr9D54yodxlwFaB8cyuOYNBIfvaUmD5vimxUBjSFvv2evL31OqHx7-_P_6i_CBbWjnyADQkb61DsWg4wVQCsrPSav4UE)[[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFSdnHMc48A0RXDSaGR7uYqPx-Uhxgo_hfN1WKpl7Bw8NOpRWZiayHFknYT_GaHbDWib3csRDZlpkWUStiRDcxVa6srGWzjZcbXVA4cQEB8AHoTPNvbYhLyqpPZfJPOKMraYArulOCurMPv)[[15]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFG8y14FbZ3eKdX6CfK9Dj5HiHviObpp-O8yuNFwzlUfQH9foYLsh1BqtbXX-uSDcnKLOuXm5yKL7vQeQ65OqPOaYdpFLa_4SAuxeWhcVKm_5pgI0QYp9f_KbptUwFEZzhyzhEKXwFLfykJd-JjqE4gjP-O1Z_ErVPUjNBwtqWhMWrZPvA=)[[3]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEy-Rry1SfoDZ6ik11DA4QCM6UWDWt6pM1ungO0bmRs84RCGSmst_HtAdQDBI9x4ofCOkP9bpjBaH4eGk4AY-48nG2ZBJDAs6svnaAyKbQpdde8sTr-Bkg5VySWkyzKWFlBq9NCO_JiKs1oub7sOGx3QqPY58I=) .

However, the Weight-Limited BFS serves as a conceptual building block for more advanced, optimal algorithms like **Iterative Lengthening Search (ILS)**, which can be seen as repeatedly executing a Weight-Limited BFS with an increasing cost threshold until the optimal solution is found [[27]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHyPellPNr6bJEZ-4-uXGwVOq3k-pOvoiLE0TEnySzzdiWutkuz2GAD2E5OJT1gixHTtttjBQlUEYq3MgeerXogzSypRCpWLhj6Icw55KNKVv1cqJaGWM0RXYWX_Ssp_no=) .

For your presentation, the key equations to highlight are the formal graph definition `G = (V, E)`, the path cost calculation `g(n) = Σ w(vᵢ, vᵢ₊₁)`, and the pruning condition `g(n) ≤ W`. Supplementing these with the provided pseudocode, the comparative walkthrough, and the nuanced discussion of practical complexity will provide a thorough and insightful explanation.

Same but limited to diseases

In [ ]:
import networkx as nx
from collections import Counter

# 1. Identify starting nodes (Bortezomib)
start_nodes = [
    n for n in G.nodes()
    if 'bortezomib' in str(G.nodes[n].get('properties', {}).get('name', '')).lower()
    or 'bortezomib' in str(G.nodes[n].get('properties', {}).get('id', '')).lower()
]

# Set the maximum accumulated weight limit and run limit for the walk
weight_limit = 100
max_runs = 50000

visited_paths = []
path_name_counter = Counter()
end_node_counter = Counter()
run_count = 0

# 2. Perform a weight-limited walk (BFS-style traversal)
for start_node in start_nodes:
    # Queue stores tuples of: (current_node, accumulated_weight, path_taken)
    queue = [(start_node, 0.0, [start_node])]

    while queue and run_count < max_runs:
        run_count += 1
        current, acc_weight, path = queue.pop(0)

        # Record valid paths ending on a disease
        if len(path) > 1:
            end_node = path[-1]
            labels = [str(l).lower() for l in G.nodes[end_node].get('labels', [])]
            is_disease = any(l for l in labels if 'disease' in l or 'side effect' in l or 'side_effect' in l or 'phenotype' in l)

            if is_disease:
                visited_paths.append((path, acc_weight))

                # Convert path IDs to names to find frequent semantic paths
                path_names = tuple(str(G.nodes[n].get('properties', {}).get('name', G.nodes[n].get('properties', {}).get('id', n))) for n in path)
                path_name_counter[path_names] += 1

                # Record the end node
                end_node_name = str(G.nodes[end_node].get('properties', {}).get('name', G.nodes[end_node].get('properties', {}).get('id', end_node)))
                end_node_counter[end_node_name] += 1

        # Explore successors using neighbors for an undirected graph
        for neighbor in G.neighbors(current):
            if neighbor not in path:  # Prevent simple cycles in the current path
                edge_data = G.get_edge_data(current, neighbor)
                # Extract edge weight, fallback to 0.01 as done previously
                edge_weight = float(edge_data.get('properties', {}).get('weight', 0.01))

                new_weight = acc_weight + edge_weight

                # Continue walk only if the limit is not exceeded
                if new_weight <= weight_limit:
                    queue.append((neighbor, new_weight, path + [neighbor]))

print(f"Stopped after {run_count} runs (Queue limit checked).")
print(f"Found {len(visited_paths)} paths originating from Bortezomib ending on Disease/Side Effect with total accumulated weight <= {weight_limit}\n")

# 3. Display the most frequent paths found
print("Top 10 Most Frequent Disease/Side Effect Paths (by node names):")
for path_names, count in path_name_counter.most_common(10):
    print(f"Frequency: {count} | Path: {' -> '.join(path_names)}")

# 4. Display the most frequent end nodes
print("\nTop 10 Most Frequent Disease/Side Effect End Nodes:")
for node_name, count in end_node_counter.most_common(10):
    print(f"Frequency: {count} | Node: {node_name}")


In [ ]:
import networkx as nx
from collections import Counter

# 1. Identify starting nodes (Bortezomib)
start_nodes = [
    n for n in G.nodes()
    if 'bortezomib' in str(G.nodes[n].get('properties', {}).get('name', '')).lower()
    or 'bortezomib' in str(G.nodes[n].get('properties', {}).get('id', '')).lower()
]

# Set the maximum accumulated weight limit and run limit for the walk
weight_limit = 60
max_runs = 150000

visited_paths = []
path_name_counter = Counter()
end_node_counter = Counter()
disease_nodes_visited = []
run_count = 0
total_disease_visits = 0

# 2. Perform a weight-limited walk (BFS-style traversal)
for start_node in start_nodes:
    # Queue stores tuples of: (current_node, accumulated_weight, path_taken)
    queue = [(start_node, 0.0, [start_node])]

    while queue and run_count < max_runs:
        run_count += 1
        current, acc_weight, path = queue.pop(0)

        # Record valid paths ending on a disease
        if len(path) > 1:
            end_node = path[-1]
            labels = [str(l).lower() for l in G.nodes[end_node].get('labels', [])]
            is_disease = any(l for l in labels if 'disease' in l)

            if is_disease:
                total_disease_visits += 1
                visited_paths.append((path, acc_weight))

                # Convert path IDs to names to find frequent semantic paths
                path_names = tuple(str(G.nodes[n].get('properties', {}).get('name', G.nodes[n].get('properties', {}).get('id', n))) for n in path)
                path_name_counter[path_names] += 1

                # Record the end node
                end_node_name = str(G.nodes[end_node].get('properties', {}).get('name', G.nodes[end_node].get('properties', {}).get('id', end_node)))
                end_node_counter[end_node_name] += 1
                disease_nodes_visited.append(end_node_name)

        # Explore successors using neighbors for an undirected graph
        for neighbor in G.neighbors(current):
            if neighbor not in path:  # Prevent simple cycles in the current path
                edge_data = G.get_edge_data(current, neighbor)
                # Extract edge weight, fallback to 0.01 as done previously
                edge_weight = float(edge_data.get('properties', {}).get('weight', 0.01))

                new_weight = acc_weight + edge_weight

                # Continue walk only if the limit is not exceeded
                if new_weight <= weight_limit:
                    queue.append((neighbor, new_weight, path + [neighbor]))

print(f"Stopped after {run_count} runs (Queue limit checked).")
print(f"Found {len(visited_paths)} paths originating from Bortezomib ending on Disease with total accumulated weight <= {weight_limit}")
print(f"Total disease nodes visited across the walks: {total_disease_visits}")
print(f"Total unique disease nodes discovered: {len(end_node_counter)}\n")

# 3. Display the most frequent paths found
print("Top 10 Most Frequent Disease Paths (by node names):")
for path_names, count in path_name_counter.most_common(10):
    print(f"Frequency: {count} | Path: {' -> '.join(path_names)}")

# 4. Display the most frequent end nodes
print("\nTop 10 Most Frequent Disease End Nodes:")
for node_name, count in end_node_counter.most_common(10):
    print(f"Frequency: {count} | Node: {node_name}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from collections import Counter

# Convert the list of visited disease nodes to a Counter, then to a DataFrame for easy plotting
df_end_nodes = pd.DataFrame(Counter(disease_nodes_visited).most_common(10), columns=['Node Name', 'Frequency'])

plt.figure(figsize=(10, 6))
sns.barplot(data=df_end_nodes, x='Frequency', y='Node Name', hue='Node Name', palette='Purples_r', legend=False)
plt.title('Top 10 Disease Nodes by Weight-Restricted Walk')
plt.xlabel('Frequency (Number of Paths)')
plt.ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
import networkx as nx
from collections import Counter

# 1. Identify starting nodes (Bortezomib)
start_nodes = [
    n for n in G.nodes()
    if 'bortezomib' in str(G.nodes[n].get('properties', {}).get('name', '')).lower()
    or 'bortezomib' in str(G.nodes[n].get('properties', {}).get('id', '')).lower()
]

# Set the maximum accumulated weight limit and run limit for the walk
weight_limit = 60
max_runs = 150000

visited_paths = []
path_name_counter = Counter()
end_node_counter = Counter()
disease_nodes_visited = []
run_count = 0
total_disease_visits = 0

# 2. Perform a weight-limited walk (BFS-style traversal)
for start_node in start_nodes:
    # Queue stores tuples of: (current_node, accumulated_weight, path_taken)
    queue = [(start_node, 0.0, [start_node])]

    while queue and run_count < max_runs:
        run_count += 1
        current, acc_weight, path = queue.pop(0)

        # Record valid paths
        if len(path) > 1:
            end_node = path[-1]

            visited_paths.append((path, acc_weight))

            # Convert path IDs to names to find frequent semantic paths
            path_names = tuple(str(G.nodes[n].get('properties', {}).get('name', G.nodes[n].get('properties', {}).get('id', n))) for n in path)
            path_name_counter[path_names] += 1

            # Record the end node
            end_node_name = str(G.nodes[end_node].get('properties', {}).get('name', G.nodes[end_node].get('properties', {}).get('id', end_node)))
            end_node_counter[end_node_name] += 1

            # Still check and track if it's a disease
            labels = [str(l).lower() for l in G.nodes[end_node].get('labels', [])]
            is_disease = any(l for l in labels if 'disease' in l)

            if is_disease:
                total_disease_visits += 1
                disease_nodes_visited.append(end_node_name)

        # Explore successors using neighbors for an undirected graph
        for neighbor in G.neighbors(current):
            if neighbor not in path:  # Prevent simple cycles in the current path
                edge_data = G.get_edge_data(current, neighbor)
                # Extract edge weight, fallback to 0.01 as done previously
                edge_weight = float(edge_data.get('properties', {}).get('weight', 0.01))

                new_weight = acc_weight + edge_weight

                # Continue walk only if the limit is not exceeded
                if new_weight <= weight_limit:
                    queue.append((neighbor, new_weight, path + [neighbor]))

print(f"Stopped after {run_count} runs (Queue limit checked).")
print(f"Found {len(visited_paths)} paths originating from Bortezomib with total accumulated weight <= {weight_limit}")
print(f"Total disease nodes visited across the walks: {total_disease_visits}")
print(f"Total unique disease nodes discovered: {len(set(disease_nodes_visited))}\n")

# 3. Display the most frequent paths found
print("Top 10 Most Frequent Paths (by node names):")
for path_names, count in path_name_counter.most_common(10):
    print(f"Frequency: {count} | Path: {' -> '.join(path_names)}")

# 4. Display the most frequent end nodes
print("\nTop 10 Most Frequent End Nodes:")
for node_name, count in end_node_counter.most_common(10):
    print(f"Frequency: {count} | Node: {node_name}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from collections import Counter

# Calculate counts for the top 10 visited diseases
top_diseases = Counter(disease_nodes_visited).most_common(10)
df_top_diseases = pd.DataFrame(top_diseases, columns=['Disease', 'Count'])

# Generate the plot
plt.figure(figsize=(10, 6))
sns.barplot(data=df_top_diseases, x='Count', y='Disease', hue='Disease', palette='viridis', legend=False)
plt.title('Top 10 Visited Diseases')
plt.xlabel('Visit Count')
plt.ylabel('Disease Name')
plt.tight_layout()
plt.show()

# MCMC

This code implements a Metropolis-Hastings Markov Chain Monte Carlo (MCMC) simulation.

It starts a 'random walker' at the Bortezomib node and prepares to take 3 million steps through the graph.
The Metropolis-Hastings Rule: For every step, the walker proposes moving to a random neighbor. The walker only moves if a calculated acceptance probability (the ratio) is met. This ratio is designed so that the walker eventually spends more time at nodes with higher 'Anchored PageRank' scores.
Degree Correction: The math (pi_proposed / deg_proposed) / (pi_current / deg_current) accounts for the fact that nodes with many connections are naturally easier to stumble into, ensuring the final counts aren't biased by high-degree nodes.
Trace Analysis: Once the 3 million steps are done, it counts how many times each node was visited.
Filtering & Validation: It filters for disease-related nodes and displays them in a table. If the MCMC simulation is successful, the nodes visited most frequently by the walker should match the top results from your earlier Personalized PageRank calculations.

In [ ]:
import random
from collections import Counter
import pandas as pd

# 1. Start node (Bortezomib)
start_nodes = [
    n for n in G.nodes()
    if 'bortezomib' in str(G.nodes[n].get('properties', {}).get('name', '')).lower()
    or 'bortezomib' in str(G.nodes[n].get('properties', {}).get('id', '')).lower()
]
current_node = start_nodes[0] if start_nodes else list(G.nodes())[0]

# MCMC parameters
num_steps = 3000000
visited_states = []

# LOWERING temperature (<1.0) sharpens the target distribution and LOWERS the acceptance rate.
temperature = 0.9

# Target distribution: anchored_dist (Personalized PageRank)
# Proposal distribution: Uniform over neighbors

accepted_moves = 0
for _ in range(num_steps):
    visited_states.append(current_node)

    neighbors = list(G.neighbors(current_node))
    if not neighbors:
        continue # Nowhere to go, stay in place

    # Propose a neighbor uniformly
    proposed_node = random.choice(neighbors)

    # Calculate Metropolis-Hastings acceptance probability
    # Apply the temperature parameter to the probabilities
    pi_current = anchored_dist.get(current_node, 1e-9) ** (1.0 / temperature)
    pi_proposed = anchored_dist.get(proposed_node, 1e-9) ** (1.0 / temperature)

    deg_current = len(neighbors)
    deg_proposed = G.degree(proposed_node)

    ratio = (pi_proposed / deg_proposed) / (pi_current / deg_current)

    # Accept or reject
    if random.random() < ratio:
        current_node = proposed_node
        accepted_moves += 1

print(f"MCMC completed {num_steps} steps.")
print(f"Acceptance rate: {accepted_moves/num_steps:.2%}\n")

# Analyze the MCMC trace
mcmc_counts = Counter(visited_states)

# Filter for diseases/side effects
disease_mcmc = []
for node, count in mcmc_counts.items():
    labels = [str(l).lower() for l in G.nodes[node].get('labels', [])]
    is_disease = any(l for l in labels if 'disease' in l)
    if is_disease:
        node_name = str(G.nodes[node].get('properties', {}).get('name', G.nodes[node].get('properties', {}).get('id', node)))
        disease_mcmc.append({'Node Name': node_name, 'MCMC Visits': count})

# Display results
df_mcmc = pd.DataFrame(disease_mcmc).sort_values(by='MCMC Visits', ascending=False).reset_index(drop=True)
print("Top 15 Disease/Side Effect nodes visited by Metropolis-Hastings MCMC:")
display(df_mcmc.head(15))

### Algorithmic Steps: Metropolis-Hastings MCMC

1. **Initialization:**
   - Identify the starting node (e.g., Bortezomib) and set it as the initial state (`current_node`).
   - Define simulation parameters: total steps (3,000,000) and `temperature` (0.9) to sharpen the target distribution.

2. **Iteration (for each step):**
   - **Record State:** Append the `current_node` to the trace of visited states.
   - **Propose Move:** Find all immediate neighbors of the `current_node`. Randomly select one as the `proposed_node`.
   - **Calculate Acceptance Probability:** Compute the Metropolis-Hastings ratio to decide whether to move.
     - Fetch the target probability (`pi`) for both the current and proposed nodes from the previously computed Anchored PageRank distribution, adjusted by the temperature exponent.
     - Get the degree (number of neighbors) for both nodes to apply degree correction, preventing bias towards naturally highly connected nodes.
     - Calculate the ratio: `(pi_proposed / deg_proposed) / (pi_current / deg_current)`.
   - **Accept/Reject:** Generate a random uniform number between 0 and 1. If this random number is less than the calculated ratio, accept the move (`current_node = proposed_node`). Otherwise, reject the move and stay at the `current_node`.

3. **Aggregation and Filtering:**
   - Count the total number of visits to each node in the simulation trace.
   - Filter the nodes to retain only those with a 'disease' or 'side effect' label.
   - Sort the diseases by visit count in descending order to identify the most probable side effects based on the simulation.

 Here is a summary of the algorithms and analytical methods used in this notebook to explore the relationships surrounding Bortezomib:

1. Graph Construction & Data Retrieval
Cypher Querying: Used to perform a multi-hop (1 to 5 steps) traversal in the Neo4j database to extract a subgraph centered on nodes containing the term 'Bortezomib'.
NetworkX Graph Building: The extracted data was converted into an undirected graph structure, preserving node labels (e.g., Disease, Compound) and relationship properties.
2. Centrality & Ranking Algorithms
PageRank (Stationary Distribution): Measures the global importance of nodes across the entire subgraph based on the graph's link structure. This identified generally 'popular' nodes like CDC20 and RPS4Y1.
Personalized (Anchored) PageRank: A modified version of PageRank where the 'teleportation' probability is restricted to the Bortezomib nodes. This measures local importance relative to the drug, helping identify specific side effects or phenotypes like Nausea and Dizziness.
3. Traversal & Path Analysis
Weight-Limited Breadth-First Search (BFS): An exploratory walk starting from Bortezomib that accumulates edge weights. It was used to find the most frequent paths and end-nodes (specifically filtering for Diseases/Side Effects) within a specific 'cost' or 'distance' threshold.
4. Stochastic Simulation
Metropolis-Hastings MCMC (Markov Chain Monte Carlo): A sampling algorithm used to simulate a random walk that converges to the Personalized PageRank distribution. By analyzing the 'trace' (how many times the walker visited each node), we confirmed the ranking of disease-related nodes through a probabilistic simulation rather than a deterministic calculation.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(3, 1, figsize=(12, 18))

# 1. Personalized PageRank
sns.barplot(data=df_comparison.head(10), x='Anchored Prob', y='Node Name', hue='Node Name', ax=axes[0], palette='Blues_r', legend=False)
axes[0].set_title('Top 10 Diseases by Personalized PageRank (Anchored to Bortezomib)')
axes[0].set_xlabel('Anchored Probability')
axes[0].set_ylabel('')

# 2. Weighted Walk
sns.barplot(data=df_walk.head(10), x='Total Visits', y='Node Name', hue='Node Name', ax=axes[1], palette='Oranges_r', legend=False)
axes[1].set_title('Top 10 Diseases by Weighted Walk Visits')
axes[1].set_xlabel('Total Visits')
axes[1].set_ylabel('')

# 3. MCMC
sns.barplot(data=df_mcmc.head(10), x='MCMC Visits', y='Node Name', hue='Node Name', ax=axes[2], palette='Greens_r', legend=False)
axes[2].set_title('Top 10 Diseases by MCMC Visits')
axes[2].set_xlabel('MCMC Visits')
axes[2].set_ylabel('')

plt.tight_layout()
plt.show()

Of course! Here is the revised research document on the Metropolis-Hastings algorithm. I've enhanced it with a detailed mathematical proof and a more advanced, practical example based on the provided sources to give you a comprehensive overview with clear, presentation-ready equations.

***

## The Metropolis-Hastings Algorithm: Core Equations and Concepts

The Metropolis-Hastings (MH) algorithm is a foundational Markov chain Monte Carlo (MCMC) method [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEMvyW8UcxRDn8aPxfTZxnlKNz0hjPWaRu5BDgCsPRnLTGd-jWieppItVMGnDPMy3df2IFCbGPG1ug-Jr9HtfJ-7_Q7dtRULPdLT_9nJceQ33Q6A7Nz4zhQJc7UJ-jYspZ8LTrp6zOK)[[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHi-fWVDR8fuS34305HX3hlvka1jNnGyBd6jzkNhaytQAojmqp5U1q4ROsxufQGy0lSlppz_F-w7sBc0iAGPVLbaDnZnUheU5W7dLVCGTz-s0wHgsc15s_OwkhVoI8pEmHiCPTRTt8KI-k3NIbz3N8wuSaQssBPxy400HnKcQ==) . It is designed to generate a sequence of random samples from a probability distribution, which is particularly useful when direct sampling from that distribution is difficult or impossible [[3]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG5Jlr7KHPpcEk-qUzUfy-humD1D-QT1o-KzhUwyg2vUI8q_hTkVYFeRIk3lCnY0la-EfT978trnIbSof9yFg0UgrlQbnRgSMz4VYjq5KHSUsuD0y4N7wxAokWgYsAK0b69rg470qsHiksyNaXMH188tDJP2iTYKmC72S06cQX4NbJIQcpsHJron79Hdw==) . This powerful algorithm is a cornerstone in diverse fields like statistics, physics, and machine learning [[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEHHHGEvBYLwcVWKjMVUn886rAfaVlOYIRLupGSHCSollwL5NjhlU2PPXVK_SN5if7RJFsmY71cCsUx5EmvyBTX_AulkDBkTDQxu3AKdh6F4u-DLQvpRYG-bD0UzYik_gcky9Mg-9nRWXdNN7Br-9meWEhALaGiT6wUTfqRJbGOa5g=)[[3]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG5Jlr7KHPpcEk-qUzUfy-humD1D-QT1o-KzhUwyg2vUI8q_hTkVYFeRIk3lCnY0la-EfT978trnIbSof9yFg0UgrlQbnRgSMz4VYjq5KHSUsuD0y4N7wxAokWgYsAK0b69rg470qsHiksyNaXMH188tDJP2iTYKmC72S06cQX4NbJIQcpsHJron79Hdw==)[[5]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHDBTgOfLV-2q4tjX8zB2D2WDSbe-h6gxKftK1DUNJWK3SfmZRWZZkhS2rgeyQGrqMVtl7kNfyRVDE0QqAlls2v-1GUR3Qe7VlsxG1rNB2c4OOkpzu0_7QTb2z1Fb7cCxpHzJZCaw==) .

### Core Concept

The fundamental principle of the Metropolis-Hastings algorithm is to construct a Markov chain whose stationary distribution is the desired target distribution, denoted as π(x) [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEMvyW8UcxRDn8aPxfTZxnlKNz0hjPWaRu5BDgCsPRnLTGd-jWieppItVMGnDPMy3df2IFCbGPG1ug-Jr9HtfJ-7_Q7dtRULPdLT_9nJceQ33Q6A7Nz4zhQJc7UJ-jYspZ8LTrp6zOK) . A Markov chain is a sequence of random variables where the future state depends only on the current state, not on the sequence of events that preceded it [[6]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGMFVMsmelQUzrMrdLTTDVJSGssFlTDOSxuaDk2LHlV6OsDk2gqnX_rQKYoEuj8xPcSFX0f_PNj68_dLIG9cULcP2Cn7t6lugSR0AMWuZInhyJeaBLgn0zNYVvoGgpNPbDqmLTPG-uo4yA=) . By simulating this chain for a sufficient number of steps, the states visited by the chain will form a set of samples that approximate the target distribution π(x) [[7]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH6Q0NMeZvGlVgw6OIznWibpX7OcFHzb8z5-kQIk_LaS9w9W6kscg2ww2m1kwPVWl3SWAWaW_daZWa1rBvNu_D1lSPYO6nb8dmc6QDNukGIcXGcA5u7f2Rbx1BRLS188Dd0d18vSW4_5npBVQ5pnAgruoZ6Adp_5UB9XAeM9JHAlNak2nQHhNraSWdY)[[8]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG7p2BSKhR_OWI36SSh4CvukmX3D3R4CLam22uG8WR2k5Y3n5LcxBkVifGP2_yKhWi7vfGctpni-K6ekdu-oDIah-sE_V3mfbNlAKTVDz4frqT3X2SyaHoXQ9joWRNsmPxaH-e4fdC-Wk_F6_RKLjzsUcSllk6jnGRVmLit4uAULGsUKh4tAwDAsymeN-JkKTGxDUiDB_bS-MmBdbIt0pJR-PRCGyJX3PbHgyI5RIBumgUkQsyNowiXzrW66w==) .

### The Algorithm Step-by-Step

The algorithm iteratively generates samples. Starting from an initial state, it proposes a new state and decides whether to accept or reject it based on a specific probability.

1.  **Initialization**: Choose an initial state, `x(0)`, for the Markov chain. This can be any point within the state space of the distribution.

2.  **Proposal**: At each iteration *t*, generate a new candidate state, `x'`, from a **proposal distribution**, `q(x' | x(t-1))`. This distribution proposes a new state based on the current state, `x(t-1)`.

3.  **Calculate Acceptance Probability**: Compute the acceptance probability, `α(x', x(t-1))`, which determines the likelihood of accepting the new candidate state `x'`. The central equation for this probability is:

    > **α(x', x(t-1)) = min(1, [π(x') * q(x(t-1) | x')] / [π(x(t-1)) * q(x' | x(t-1))])** [[9]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFGHYHEmrLmQc5J3eDqrxLIW_LIaAvXjVcRku3s6yn9cFF_hwCAHw8YiowD8eZlIqlAXHhJWSika_hy9J58yj4r4rRzbs-XCTioAPnCqp8Diujx82AxT0ECp6St)[[10]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEZIIyyYXX9HQ0eYhV9nRya-frdKGBakS-5V1wZ5V-GKtEazcnCjsH-ayB8MLRsQ-OW_CIyW7T3MTYnT675-4lIkBEjsCBhslIEG8xYNSnOxgfFpVMhLA5XRsZ-gMCMx-R-ZkdQyxYVqpKsyhi9EbjW2bEbHMMO)[[11]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGC0w_TwTc5TCGecdPaXAw6Qo2q0pfSCVFTXausRaTVbwVB2VG_5EyP8Y91KnxMSFZoNRkjVWdz2gtldqmDw8zpKHOPBcafLszXwREEvtV5rLr0Uo08lRo5IuvfkJL0G269lHzqFU8KECpqkeaWPs04zH2Z1SxUVnS4ViWGQ6-lEx0x)[[6]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGMFVMsmelQUzrMrdLTTDVJSGssFlTDOSxuaDk2LHlV6OsDk2gqnX_rQKYoEuj8xPcSFX0f_PNj68_dLIG9cULcP2Cn7t6lugSR0AMWuZInhyJeaBLgn0zNYVvoGgpNPbDqmLTPG-uo4yA=)[[12]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHDsyx6g2sTT0H1Xj030WqcUsTsrirJr9w63KayGxeOx5oklOYu7GIyidFCUrEa7Rbr-H2xel1ZJZL8-HnCu5G5Q3ECmn1Av2rUvSDMZQ13iBH-gaT68VnmKXf29cJp8Obz3ZRJT3_lSVMELhYcVtlqU8DHzW4YVjCLoNCvZ6YylAvQaROzeIGYS18j7vORiHL8n05CtZuR)[[13]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGl7p85qkuyzznLpZJMGHanC9o7CZUpwi6bBEzqekhhw5wMTzglKeM2c2MQGn7OV8qn7C9ZSju35rM51xRZniVavSDK7H1hbAj7zBWhXxiAqDupjLj2qfybV_S6ZKGVIXdY4QZrZy061FoGW3D7q9NlxXGeZ44QChcAnNDgQxj38DHuVlQ=)[[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEMvyW8UcxRDn8aPxfTZxnlKNz0hjPWaRu5BDgCsPRnLTGd-jWieppItVMGnDPMy3df2IFCbGPG1ug-Jr9HtfJ-7_Q7dtRULPdLT_9nJceQ33Q6A7Nz4zhQJc7UJ-jYspZ8LTrp6zOK)[[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHi-fWVDR8fuS34305HX3hlvka1jNnGyBd6jzkNhaytQAojmqp5U1q4ROsxufQGy0lSlppz_F-w7sBc0iAGPVLbaDnZnUheU5W7dLVCGTz-s0wHgsc15s_OwkhVoI8pEmHiCPTRTt8KI-k3NIbz3N8wuSaQssBPxy400HnKcQ==)[[7]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH6Q0NMeZvGlVgw6OIznWibpX7OcFHzb8z5-kQIk_LaS9w9W6kscg2ww2m1kwPVWl3SWAWaW_daZWa1rBvNu_D1lSPYO6nb8dmc6QDNukGIcXGcA5u7f2Rbx1BRLS188Dd0d18vSW4_5npBVQ5pnAgruoZ6Adp_5UB9XAeM9JHAlNak2nQHhNraSWdY)[[8]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG7p2BSKhR_OWI36SSh4CvukmX3D3R4CLam22uG8WR2k5Y3n5LcxBkVifGP2_yKhWi7vfGctpni-K6ekdu-oDIah-sE_V3mfbNlAKTVDz4frqT3X2SyaHoXQ9joWRNsmPxaH-e4fdC-Wk_F6_RKLjzsUcSllk6jnGRVmLit4uAULGsUKh4tAwDAsymeN-JkKTGxDUiDB_bS-MmBdbIt0pJR-PRCGyJX3PbHgyI5RIBumgUkQsyNowiXzrW66w==)[[14]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHLLNxfEn3Sb5Qe6CUQM8pzyKW5EmzSfvTmIJlebynkDDFWFhgo-NVrAFnDc9Fm5qSy0dS6UzCkdzyKaeMCsuYezQXFDKr5jlMNVlIskqXGfWK4dJeDuFu-NCcFgmcI4bSzoFmtf7GA-iVzMx4cgCjD)[[15]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH-yKT2ZGzyavv1voSGSLJ1a7kYhtIwkJu4D1LNR6QqKaZgrtcrsfW8hBmvcxWHfOCMqPmHKOHQ1Sz9_j2DEZyoAYjyNZeEe6HHbK4yQCeOkA6mdA_16iwqzF-7EFnrSBP3JpD43A==)[[16]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGEiJIpqzQEqbmUaAFgDwu0GTbdcroDxooR2r3n3ECxbJwSBRB9U75iauM0ATkim0M2oivWQOE_N5eIwka1ttRCr37XRKFGOVB535at0_dFtig6dMEbGRp4xuGIlL9Du4i2q0kMCPp_6-U=)[[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEHHHGEvBYLwcVWKjMVUn886rAfaVlOYIRLupGSHCSollwL5NjhlU2PPXVK_SN5if7RJFsmY71cCsUx5EmvyBTX_AulkDBkTDQxu3AKdh6F4u-DLQvpRYG-bD0UzYik_gcky9Mg-9nRWXdNN7Br-9meWEhALaGiT6wUTfqRJbGOa5g=)[[3]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG5Jlr7KHPpcEk-qUzUfy-humD1D-QT1o-KzhUwyg2vUI8q_hTkVYFeRIk3lCnY0la-EfT978trnIbSof9yFg0UgrlQbnRgSMz4VYjq5KHSUsuD0y4N7wxAokWgYsAK0b69rg470qsHiksyNaXMH188tDJP2iTYKmC72S06cQX4NbJIQcpsHJron79Hdw==)[[17]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQF-5OpxoCmublJ2diq-oMtD3fZsmpt_UISftailCDR1K_C9sW1_JqbF14O150eI7xC3HEMSwbggZzTDHdFBP7aqd4hQmXUkSdNHtNUt60F97kA_rhDmU1cRdBRos7vg5T6LN9GuiCCsx7zTOCm6nKxEE0A6GgK0S95awgXYqZ7fa_gpitn60hbOjZLTCcL330YrR7lFqbO7cjxDFv8Hb0HKsyUi5SkqZd2zhDMX7ekzSTU=)[[5]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHDBTgOfLV-2q4tjX8zB2D2WDSbe-h6gxKftK1DUNJWK3SfmZRWZZkhS2rgeyQGrqMVtl7kNfyRVDE0QqAlls2v-1GUR3Qe7VlsxG1rNB2c4OOkpzu0_7QTb2z1Fb7cCxpHzJZCaw==)[[18]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHf4eAcGAPSUCpiLHt0TLoSGxOhkZUEa7Z9HOKdxw-V2X3p-QKxPD5MFxJRI7FXVFwJohty1wLvDZOMjLYzpPCec0i9B9q7bjuzFk674V6wnKRz_c8BuJb61MsMZgSE8xHwGUKH74LA-ENUDhPqgH-CvH2EAQ==)

    *   `π(x)` is the target distribution you want to sample from.
    *   `q(x' | x(t-1))` is the proposal probability of moving from state `x(t-1)` to `x'`.
    *   `q(x(t-1) | x')` is the proposal probability of moving in the reverse direction, from `x'` to `x(t-1)`.

4.  **Accept or Reject**: Draw a random number, `u`, from a uniform distribution on the interval [[9]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFGHYHEmrLmQc5J3eDqrxLIW_LIaAvXjVcRku3s6yn9cFF_hwCAHw8YiowD8eZlIqlAXHhJWSika_hy9J58yj4r4rRzbs-XCTioAPnCqp8Diujx82AxT0ECp6St) .
    *   If `u ≤ α(x', x(t-1))`, **accept** the proposal. The next state of the chain is `x(t) = x'`.
    *   If `u > α(x', x(t-1))`, **reject** the proposal. The chain remains in its current state, so the next state is `x(t) = x(t-1)`.

5.  **Iteration**: Repeat steps 2-4 for a large number of iterations. The initial samples are typically discarded (the "burn-in" period) to allow the chain to converge to its stationary distribution. The subsequent samples are used as an approximation of the target distribution π(x).

### Key Components and Equations

The effectiveness of the MH algorithm hinges on the interplay between the target and proposal distributions.

*   **Target Distribution (π(x))**: This is the probability distribution you aim to sample [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEMvyW8UcxRDn8aPxfTZxnlKNz0hjPWaRu5BDgCsPRnLTGd-jWieppItVMGnDPMy3df2IFCbGPG1ug-Jr9HtfJ-7_Q7dtRULPdLT_9nJceQ33Q6A7Nz4zhQJc7UJ-jYspZ8LTrp6zOK) . A major advantage of the MH algorithm is that you only need to know `π(x)` up to a constant of proportionality [[7]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH6Q0NMeZvGlVgw6OIznWibpX7OcFHzb8z5-kQIk_LaS9w9W6kscg2ww2m1kwPVWl3SWAWaW_daZWa1rBvNu_D1lSPYO6nb8dmc6QDNukGIcXGcA5u7f2Rbx1BRLS188Dd0d18vSW4_5npBVQ5pnAgruoZ6Adp_5UB9XAeM9JHAlNak2nQHhNraSWdY)[[15]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH-yKT2ZGzyavv1voSGSLJ1a7kYhtIwkJu4D1LNR6QqKaZgrtcrsfW8hBmvcxWHfOCMqPmHKOHQ1Sz9_j2DEZyoAYjyNZeEe6HHbK4yQCeOkA6mdA_16iwqzF-7EFnrSBP3JpD43A==) . This is because any normalizing constants in `π(x)` appear in both the numerator and denominator of the acceptance ratio, so they cancel out.

*   **Proposal Distribution (q(x' | x))**: The choice of this distribution is critical for the algorithm's efficiency [[6]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGMFVMsmelQUzrMrdLTTDVJSGssFlTDOSxuaDk2LHlV6OsDk2gqnX_rQKYoEuj8xPcSFX0f_PNj68_dLIG9cULcP2Cn7t6lugSR0AMWuZInhyJeaBLgn0zNYVvoGgpNPbDqmLTPG-uo4yA=) . It must be easy to sample from and should allow the chain to explore the entire state space effectively. Common choices include:
    *   **Symmetric Proposal (Metropolis Algorithm)**: If the proposal distribution is symmetric, meaning `q(x' | x) = q(x | x')`, the acceptance ratio simplifies significantly. This special case is the original Metropolis algorithm [[9]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFGHYHEmrLmQc5J3eDqrxLIW_LIaAvXjVcRku3s6yn9cFF_hwCAHw8YiowD8eZlIqlAXHhJWSika_hy9J58yj4r4rRzbs-XCTioAPnCqp8Diujx82AxT0ECp6St)[[10]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEZIIyyYXX9HQ0eYhV9nRya-frdKGBakS-5V1wZ5V-GKtEazcnCjsH-ayB8MLRsQ-OW_CIyW7T3MTYnT675-4lIkBEjsCBhslIEG8xYNSnOxgfFpVMhLA5XRsZ-gMCMx-R-ZkdQyxYVqpKsyhi9EbjW2bEbHMMO)[[6]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGMFVMsmelQUzrMrdLTTDVJSGssFlTDOSxuaDk2LHlV6OsDk2gqnX_rQKYoEuj8xPcSFX0f_PNj68_dLIG9cULcP2Cn7t6lugSR0AMWuZInhyJeaBLgn0zNYVvoGgpNPbDqmLTPG-uo4yA=)[[17]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQF-5OpxoCmublJ2diq-oMtD3fZsmpt_UISftailCDR1K_C9sW1_JqbF14O150eI7xC3HEMSwbggZzTDHdFBP7aqd4hQmXUkSdNHtNUt60F97kA_rhDmU1cRdBRos7vg5T6LN9GuiCCsx7zTOCm6nKxEE0A6GgK0S95awgXYqZ7fa_gpitn60hbOjZLTCcL330YrR7lFqbO7cjxDFv8Hb0HKsyUi5SkqZd2zhDMX7ekzSTU=)[[5]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHDBTgOfLV-2q4tjX8zB2D2WDSbe-h6gxKftK1DUNJWK3SfmZRWZZkhS2rgeyQGrqMVtl7kNfyRVDE0QqAlls2v-1GUR3Qe7VlsxG1rNB2c4OOkpzu0_7QTb2z1Fb7cCxpHzJZCaw==)[[18]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHf4eAcGAPSUCpiLHt0TLoSGxOhkZUEa7Z9HOKdxw-V2X3p-QKxPD5MFxJRI7FXVFwJohty1wLvDZOMjLYzpPCec0i9B9q7bjuzFk674V6wnKRz_c8BuJb61MsMZgSE8xHwGUKH74LA-ENUDhPqgH-CvH2EAQ==) . The acceptance probability becomes:
        > **α(x', x) = min(1, π(x') / π(x))**
    *   **Independent Metropolis-Hastings**: Here, the proposed state is drawn from a distribution that is independent of the current state, so `q(x' | x) = q(x')` [[6]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGMFVMsmelQUzrMrdLTTDVJSGssFlTDOSxuaDk2LHlV6OsDk2gqnX_rQKYoEuj8xPcSFX0f_PNj68_dLIG9cULcP2Cn7t6lugSR0AMWuZInhyJeaBLgn0zNYVvoGgpNPbDqmLTPG-uo4yA=)[[7]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH6Q0NMeZvGlVgw6OIznWibpX7OcFHzb8z5-kQIk_LaS9w9W6kscg2ww2m1kwPVWl3SWAWaW_daZWa1rBvNu_D1lSPYO6nb8dmc6QDNukGIcXGcA5u7f2Rbx1BRLS188Dd0d18vSW4_5npBVQ5pnAgruoZ6Adp_5UB9XAeM9JHAlNak2nQHhNraSWdY) . The acceptance ratio is:
        > **α(x', x) = min(1, [π(x') * q(x)] / [π(x) * q(x')])**
    *   **Random Walk Metropolis-Hastings**: This is a broad class of methods where the new state is a "walk" from the current state. The proposal can be symmetric (e.g., `x' = x + w`, where `w` is from a symmetric distribution like a Gaussian) or asymmetric, requiring the full Hastings correction [[9]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFGHYHEmrLmQc5J3eDqrxLIW_LIaAvXjVcRku3s6yn9cFF_hwCAHw8YiowD8eZlIqlAXHhJWSika_hy9J58yj4r4rRzbs-XCTioAPnCqp8Diujx82AxT0ECp6St)[[6]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGMFVMsmelQUzrMrdLTTDVJSGssFlTDOSxuaDk2LHlV6OsDk2gqnX_rQKYoEuj8xPcSFX0f_PNj68_dLIG9cULcP2Cn7t6lugSR0AMWuZInhyJeaBLgn0zNYVvoGgpNPbDqmLTPG-uo4yA=)[[17]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQF-5OpxoCmublJ2diq-oMtD3fZsmpt_UISftailCDR1K_C9sW1_JqbF14O150eI7xC3HEMSwbggZzTDHdFBP7aqd4hQmXUkSdNHtNUt60F97kA_rhDmU1cRdBRos7vg5T6LN9GuiCCsx7zTOCm6nKxEE0A6GgK0S95awgXYqZ7fa_gpitn60hbOjZLTCcL330YrR7lFqbO7cjxDFv8Hb0HKsyUi5SkqZd2zhDMX7ekzSTU=) .

*   **Acceptance Rate**: This is the fraction of proposed samples that are accepted. It serves as a crucial diagnostic tool. A very high rate may mean the steps are too small, leading to slow exploration. A very low rate may mean the steps are too large, leading to inefficient rejection of most moves. For many problems, an acceptance rate between 23% and 50% is considered a good target [[16]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGEiJIpqzQEqbmUaAFgDwu0GTbdcroDxooR2r3n3ECxbJwSBRB9U75iauM0ATkim0M2oivWQOE_N5eIwka1ttRCr37XRKFGOVB535at0_dFtig6dMEbGRp4xuGIlL9Du4i2q0kMCPp_6-U=) .

### Mathematical Foundation: The Detailed Balance Condition

The Metropolis-Hastings algorithm is constructed to satisfy the **detailed balance condition**, which is a sufficient (but not necessary) condition for ensuring the Markov chain's stationary distribution is the desired target distribution `π(x)` [[10]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEZIIyyYXX9HQ0eYhV9nRya-frdKGBakS-5V1wZ5V-GKtEazcnCjsH-ayB8MLRsQ-OW_CIyW7T3MTYnT675-4lIkBEjsCBhslIEG8xYNSnOxgfFpVMhLA5XRsZ-gMCMx-R-ZkdQyxYVqpKsyhi9EbjW2bEbHMMO)[[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEMvyW8UcxRDn8aPxfTZxnlKNz0hjPWaRu5BDgCsPRnLTGd-jWieppItVMGnDPMy3df2IFCbGPG1ug-Jr9HtfJ-7_Q7dtRULPdLT_9nJceQ33Q6A7Nz4zhQJc7UJ-jYspZ8LTrp6zOK)[[14]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHLLNxfEn3Sb5Qe6CUQM8pzyKW5EmzSfvTmIJlebynkDDFWFhgo-NVrAFnDc9Fm5qSy0dS6UzCkdzyKaeMCsuYezQXFDKr5jlMNVlIskqXGfWK4dJeDuFu-NCcFgmcI4bSzoFmtf7GA-iVzMx4cgCjD)[[15]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH-yKT2ZGzyavv1voSGSLJ1a7kYhtIwkJu4D1LNR6QqKaZgrtcrsfW8hBmvcxWHfOCMqPmHKOHQ1Sz9_j2DEZyoAYjyNZeEe6HHbK4yQCeOkA6mdA_16iwqzF-7EFnrSBP3JpD43A==)[[18]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHf4eAcGAPSUCpiLHt0TLoSGxOhkZUEa7Z9HOKdxw-V2X3p-QKxPD5MFxJRI7FXVFwJohty1wLvDZOMjLYzpPCec0i9B9q7bjuzFk674V6wnKRz_c8BuJb61MsMZgSE8xHwGUKH74LA-ENUDhPqgH-CvH2EAQ==) . The detailed balance equation states that the probability flow between any two states, x and x', is equal in both directions when the chain is in its stationary state [[9]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFGHYHEmrLmQc5J3eDqrxLIW_LIaAvXjVcRku3s6yn9cFF_hwCAHw8YiowD8eZlIqlAXHhJWSika_hy9J58yj4r4rRzbs-XCTioAPnCqp8Diujx82AxT0ECp6St)[[10]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEZIIyyYXX9HQ0eYhV9nRya-frdKGBakS-5V1wZ5V-GKtEazcnCjsH-ayB8MLRsQ-OW_CIyW7T3MTYnT675-4lIkBEjsCBhslIEG8xYNSnOxgfFpVMhLA5XRsZ-gMCMx-R-ZkdQyxYVqpKsyhi9EbjW2bEbHMMO) .

> **π(x) * P(x → x') = π(x') * P(x' → x)**

Here, `P(x' | x)` is the total transition probability of the Markov chain moving from state `x` to `x'`. A distribution `π` that satisfies this condition is guaranteed to be a stationary distribution for the chain [[9]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFGHYHEmrLmQc5J3eDqrxLIW_LIaAvXjVcRku3s6yn9cFF_hwCAHw8YiowD8eZlIqlAXHhJWSika_hy9J58yj4r4rRzbs-XCTioAPnCqp8Diujx82AxT0ECp6St)[[11]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGC0w_TwTc5TCGecdPaXAw6Qo2q0pfSCVFTXausRaTVbwVB2VG_5EyP8Y91KnxMSFZoNRkjVWdz2gtldqmDw8zpKHOPBcafLszXwREEvtV5rLr0Uo08lRo5IuvfkJL0G269lHzqFU8KECpqkeaWPs04zH2Z1SxUVnS4ViWGQ6-lEx0x)[[6]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGMFVMsmelQUzrMrdLTTDVJSGssFlTDOSxuaDk2LHlV6OsDk2gqnX_rQKYoEuj8xPcSFX0f_PNj68_dLIG9cULcP2Cn7t6lugSR0AMWuZInhyJeaBLgn0zNYVvoGgpNPbDqmLTPG-uo4yA=)[[12]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHDsyx6g2sTT0H1Xj030WqcUsTsrirJr9w63KayGxeOx5oklOYu7GIyidFCUrEa7Rbr-H2xel1ZJZL8-HnCu5G5Q3ECmn1Av2rUvSDMZQ13iBH-gaT68VnmKXf29cJp8Obz3ZRJT3_lSVMELhYcVtlqU8DHzW4YVjCLoNCvZ6YylAvQaROzeIGYS18j7vORiHL8n05CtZuR) .

#### Proof of Detailed Balance

The genius of the MH algorithm is that the acceptance probability `α` is engineered to guarantee this condition holds. The total transition probability for `x ≠ x'` is the probability of proposing `x'` and then accepting it: `P(x → x') = q(x'|x) * α(x', x)`.

Let's prove that `π(x) * P(x → x') = π(x') * P(x' → x)`.

1.  **Analyze the left-hand side:**
    `π(x) P(x → x') = π(x) * q(x'|x) * α(x', x)`
    `= π(x) * q(x'|x) * min(1, [π(x')q(x|x')] / [π(x)q(x'|x)])`

2.  **Bring the outer term inside the `min` function** using the identity `a * min(1, b/a) = min(a, b)`:
    `= min(π(x)q(x'|x) * 1, π(x)q(x'|x) * [π(x')q(x|x')] / [π(x)q(x'|x)])`
    `= min(π(x)q(x'|x), π(x')q(x|x'))`

3.  **Analyze the right-hand side:**
    `π(x') P(x' → x) = π(x') * q(x|x') * α(x, x')`
    `= π(x') * q(x|x') * min(1, [π(x)q(x'|x)] / [π(x')q(x|x')])`
    `= min(π(x')q(x|x'), π(x)q(x'|x))`

The expressions for both sides are identical and symmetric [[16]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGEiJIpqzQEqbmUaAFgDwu0GTbdcroDxooR2r3n3ECxbJwSBRB9U75iauM0ATkim0M2oivWQOE_N5eIwka1ttRCr37XRKFGOVB535at0_dFtig6dMEbGRp4xuGIlL9Du4i2q0kMCPp_6-U=) . This completes the proof [[11]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGC0w_TwTc5TCGecdPaXAw6Qo2q0pfSCVFTXausRaTVbwVB2VG_5EyP8Y91KnxMSFZoNRkjVWdz2gtldqmDw8zpKHOPBcafLszXwREEvtV5rLr0Uo08lRo5IuvfkJL0G269lHzqFU8KECpqkeaWPs04zH2Z1SxUVnS4ViWGQ6-lEx0x)[[12]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHDsyx6g2sTT0H1Xj030WqcUsTsrirJr9w63KayGxeOx5oklOYu7GIyidFCUrEa7Rbr-H2xel1ZJZL8-HnCu5G5Q3ECmn1Av2rUvSDMZQ13iBH-gaT68VnmKXf29cJp8Obz3ZRJT3_lSVMELhYcVtlqU8DHzW4YVjCLoNCvZ6YylAvQaROzeIGYS18j7vORiHL8n05CtZuR)[[13]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGl7p85qkuyzznLpZJMGHanC9o7CZUpwi6bBEzqekhhw5wMTzglKeM2c2MQGn7OV8qn7C9ZSju35rM51xRZniVavSDK7H1hbAj7zBWhXxiAqDupjLj2qfybV_S6ZKGVIXdY4QZrZy061FoGW3D7q9NlxXGeZ44QChcAnNDgQxj38DHuVlQ=) . By satisfying detailed balance, and if the chain is ergodic (able to reach any state from any other state), it is guaranteed to converge to `π(x)` as its unique stationary distribution [[6]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGMFVMsmelQUzrMrdLTTDVJSGssFlTDOSxuaDk2LHlV6OsDk2gqnX_rQKYoEuj8xPcSFX0f_PNj68_dLIG9cULcP2Cn7t6lugSR0AMWuZInhyJeaBLgn0zNYVvoGgpNPbDqmLTPG-uo4yA=)[[12]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHDsyx6g2sTT0H1Xj030WqcUsTsrirJr9w63KayGxeOx5oklOYu7GIyidFCUrEa7Rbr-H2xel1ZJZL8-HnCu5G5Q3ECmn1Av2rUvSDMZQ13iBH-gaT68VnmKXf29cJp8Obz3ZRJT3_lSVMELhYcVtlqU8DHzW4YVjCLoNCvZ6YylAvQaROzeIGYS18j7vORiHL8n05CtZuR)[[3]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG5Jlr7KHPpcEk-qUzUfy-humD1D-QT1o-KzhUwyg2vUI8q_hTkVYFeRIk3lCnY0la-EfT978trnIbSof9yFg0UgrlQbnRgSMz4VYjq5KHSUsuD0y4N7wxAokWgYsAK0b69rg470qsHiksyNaXMH188tDJP2iTYKmC72S06cQX4NbJIQcpsHJron79Hdw==) .

### Advanced Application: MCMC on Graphs

To make the abstract equations tangible, let's consider applying the MH algorithm to sample nodes on a graph `G = (V, E)`.

#### 1. The Proposal Mechanism: A Random Walk

A simple proposal mechanism `q(v|u)` is to choose one of the neighbors of the current node `u` uniformly at random.
*   The probability of proposing a move to a neighbor `v` from `u` is `q(v|u) = 1 / deg(u)`, where `deg(u)` is the degree of node `u`.
*   The reverse proposal probability is `q(u|v) = 1 / deg(v)`.

The ratio of these proposal probabilities is the **degree correction term**:

> **q(u|v) / q(v|u) = (1 / deg(v)) / (1 / deg(u)) = deg(u) / deg(v)**

This term corrects for the proposal's natural bias to move towards higher-degree nodes.

#### 2. The Target Distribution: From Uniform to Personalized PageRank

The power of MH is that we can choose our target distribution `π(v)`.

*   **Simple Case: Uniform Sampling**: If the goal is to sample all nodes with equal probability, the target distribution `π` is uniform (`π(v) = π(u)` for all `u,v`). The `π` terms cancel, and the acceptance probability simplifies to:
    > **α(u,v) = min(1, deg(u) / deg(v))**
    This equation shows the correction in action: moves to higher-degree nodes (`deg(v) > deg(u)`) may be rejected, while moves to lower-degree nodes are always accepted.

*   **Advanced Case: Personalized PageRank (PPR)**: We can use a more sophisticated target distribution like Personalized PageRank, which measures node importance relative to a set of "anchor" nodes `S` [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEMvyW8UcxRDn8aPxfTZxnlKNz0hjPWaRu5BDgCsPRnLTGd-jWieppItVMGnDPMy3df2IFCbGPG1ug-Jr9HtfJ-7_Q7dtRULPdLT_9nJceQ33Q6A7Nz4zhQJc7UJ-jYspZ8LTrp6zOK)[[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHi-fWVDR8fuS34305HX3hlvka1jNnGyBd6jzkNhaytQAojmqp5U1q4ROsxufQGy0lSlppz_F-w7sBc0iAGPVLbaDnZnUheU5W7dLVCGTz-s0wHgsc15s_OwkhVoI8pEmHiCPTRTt8KI-k3NIbz3N8wuSaQssBPxy400HnKcQ==) . The PPR score vector `π` is the stationary distribution of a random walk that either follows a link (with probability `d`) or "teleports" to an anchor node (with probability `1-d`). It is the solution to the iterative equation:
    > **π = d * M * π + (1 - d) * v**
    *   `π`: The vector of PPR scores, our target distribution.
    *   `d`: The damping factor (e.g., 0.85), balancing graph structure vs. anchor influence [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHi-fWVDR8fuS34305HX3hlvka1jNnGyBd6jzkNhaytQAojmqp5U1q4ROsxufQGy0lSlppz_F-w7sBc0iAGPVLbaDnZnUheU5W7dLVCGTz-s0wHgsc15s_OwkhVoI8pEmHiCPTRTt8KI-k3NIbz3N8wuSaQssBPxy400HnKcQ==) .
    *   `M`: The column-stochastic transition matrix of the graph.
    *   `v`: The teleportation vector, a probability distribution non-zero only for anchor nodes (e.g., `v_i = 1/|S|` if node `i` is in `S`). This biases the distribution, assigning higher scores to nodes "close" to the anchors.

#### 3. Temperature Control and Simulated Annealing

We can introduce a temperature parameter `T` to "sharpen" or "flatten" the target distribution, a technique central to Simulated Annealing [[9]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFGHYHEmrLmQc5J3eDqrxLIW_LIaAvXjVcRku3s6yn9cFF_hwCAHw8YiowD8eZlIqlAXHhJWSika_hy9J58yj4r4rRzbs-XCTioAPnCqp8Diujx82AxT0ECp6St)[[10]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEZIIyyYXX9HQ0eYhV9nRya-frdKGBakS-5V1wZ5V-GKtEazcnCjsH-ayB8MLRsQ-OW_CIyW7T3MTYnT675-4lIkBEjsCBhslIEG8xYNSnOxgfFpVMhLA5XRsZ-gMCMx-R-ZkdQyxYVqpKsyhi9EbjW2bEbHMMO)[[11]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGC0w_TwTc5TCGecdPaXAw6Qo2q0pfSCVFTXausRaTVbwVB2VG_5EyP8Y91KnxMSFZoNRkjVWdz2gtldqmDw8zpKHOPBcafLszXwREEvtV5rLr0Uo08lRo5IuvfkJL0G269lHzqFU8KECpqkeaWPs04zH2Z1SxUVnS4ViWGQ6-lEx0x)[[7]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH6Q0NMeZvGlVgw6OIznWibpX7OcFHzb8z5-kQIk_LaS9w9W6kscg2ww2m1kwPVWl3SWAWaW_daZWa1rBvNu_D1lSPYO6nb8dmc6QDNukGIcXGcA5u7f2Rbx1BRLS188Dd0d18vSW4_5npBVQ5pnAgruoZ6Adp_5UB9XAeM9JHAlNak2nQHhNraSWdY)[[8]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG7p2BSKhR_OWI36SSh4CvukmX3D3R4CLam22uG8WR2k5Y3n5LcxBkVifGP2_yKhWi7vfGctpni-K6ekdu-oDIah-sE_V3mfbNlAKTVDz4frqT3X2SyaHoXQ9joWRNsmPxaH-e4fdC-Wk_F6_RKLjzsUcSllk6jnGRVmLit4uAULGsUKh4tAwDAsymeN-JkKTGxDUiDB_bS-MmBdbIt0pJR-PRCGyJX3PbHgyI5RIBumgUkQsyNowiXzrW66w==)[[14]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHLLNxfEn3Sb5Qe6CUQM8pzyKW5EmzSfvTmIJlebynkDDFWFhgo-NVrAFnDc9Fm5qSy0dS6UzCkdzyKaeMCsuYezQXFDKr5jlMNVlIskqXGfWK4dJeDuFu-NCcFgmcI4bSzoFmtf7GA-iVzMx4cgCjD) . The distribution is modified as:

> **π_T(x) ∝ [π(x)]^(1/T)**

*   **Low Temperature (T < 1, T → 0)**: **Sharpens** the distribution, exaggerating probability differences and concentrating the search on the highest-probability states [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEMvyW8UcxRDn8aPxfTZxnlKNz0hjPWaRu5BDgCsPRnLTGd-jWieppItVMGnDPMy3df2IFCbGPG1ug-Jr9HtfJ-7_Q7dtRULPdLT_9nJceQ33Q6A7Nz4zhQJc7UJ-jYspZ8LTrp6zOK)[[8]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG7p2BSKhR_OWI36SSh4CvukmX3D3R4CLam22uG8WR2k5Y3n5LcxBkVifGP2_yKhWi7vfGctpni-K6ekdu-oDIah-sE_V3mfbNlAKTVDz4frqT3X2SyaHoXQ9joWRNsmPxaH-e4fdC-Wk_F6_RKLjzsUcSllk6jnGRVmLit4uAULGsUKh4tAwDAsymeN-JkKTGxDUiDB_bS-MmBdbIt0pJR-PRCGyJX3PbHgyI5RIBumgUkQsyNowiXzrW66w==)[[15]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH-yKT2ZGzyavv1voSGSLJ1a7kYhtIwkJu4D1LNR6QqKaZgrtcrsfW8hBmvcxWHfOCMqPmHKOHQ1Sz9_j2DEZyoAYjyNZeEe6HHbK4yQCeOkA6mdA_16iwqzF-7EFnrSBP3JpD43A==) . This turns the sampler into a greedy optimizer.
*   **High Temperature (T > 1, T → ∞)**: **Flattens** the distribution towards uniform, encouraging broad exploration and allowing the algorithm to escape local minima [[10]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEZIIyyYXX9HQ0eYhV9nRya-frdKGBakS-5V1wZ5V-GKtEazcnCjsH-ayB8MLRsQ-OW_CIyW7T3MTYnT675-4lIkBEjsCBhslIEG8xYNSnOxgfFpVMhLA5XRsZ-gMCMx-R-ZkdQyxYVqpKsyhi9EbjW2bEbHMMO)[[13]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGl7p85qkuyzznLpZJMGHanC9o7CZUpwi6bBEzqekhhw5wMTzglKeM2c2MQGn7OV8qn7C9ZSju35rM51xRZniVavSDK7H1hbAj7zBWhXxiAqDupjLj2qfybV_S6ZKGVIXdY4QZrZy061FoGW3D7q9NlxXGeZ44QChcAnNDgQxj38DHuVlQ=)[[7]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH6Q0NMeZvGlVgw6OIznWibpX7OcFHzb8z5-kQIk_LaS9w9W6kscg2ww2m1kwPVWl3SWAWaW_daZWa1rBvNu_D1lSPYO6nb8dmc6QDNukGIcXGcA5u7f2Rbx1BRLS188Dd0d18vSW4_5npBVQ5pnAgruoZ6Adp_5UB9XAeM9JHAlNak2nQHhNraSWdY)[[15]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH-yKT2ZGzyavv1voSGSLJ1a7kYhtIwkJu4D1LNR6QqKaZgrtcrsfW8hBmvcxWHfOCMqPmHKOHQ1Sz9_j2DEZyoAYjyNZeEe6HHbK4yQCeOkA6mdA_16iwqzF-7EFnrSBP3JpD43A==) .
*   **T = 1**: Recovers the original target distribution `π(x)`.

Using the energy analogy `E(x) = -log(π(x))`, the temperature-adjusted distribution becomes the Boltzmann distribution `π_T(x) ∝ exp(-E(x) / T)` [[14]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHLLNxfEn3Sb5Qe6CUQM8pzyKW5EmzSfvTmIJlebynkDDFWFhgo-NVrAFnDc9Fm5qSy0dS6UzCkdzyKaeMCsuYezQXFDKr5jlMNVlIskqXGfWK4dJeDuFu-NCcFgmcI4bSzoFmtf7GA-iVzMx4cgCjD) . For a symmetric proposal, the acceptance probability is the classic formula for **Simulated Annealing**:

> **α_T = min(1, exp[-(E(x') - E(x)) / T])** [[11]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGC0w_TwTc5TCGecdPaXAw6Qo2q0pfSCVFTXausRaTVbwVB2VG_5EyP8Y91KnxMSFZoNRkjVWdz2gtldqmDw8zpKHOPBcafLszXwREEvtV5rLr0Uo08lRo5IuvfkJL0G269lHzqFU8KECpqkeaWPs04zH2Z1SxUVnS4ViWGQ6-lEx0x)[[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHi-fWVDR8fuS34305HX3hlvka1jNnGyBd6jzkNhaytQAojmqp5U1q4ROsxufQGy0lSlppz_F-w7sBc0iAGPVLbaDnZnUheU5W7dLVCGTz-s0wHgsc15s_OwkhVoI8pEmHiCPTRTt8KI-k3NIbz3N8wuSaQssBPxy400HnKcQ==)[[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEHHHGEvBYLwcVWKjMVUn886rAfaVlOYIRLupGSHCSollwL5NjhlU2PPXVK_SN5if7RJFsmY71cCsUx5EmvyBTX_AulkDBkTDQxu3AKdh6F4u-DLQvpRYG-bD0UzYik_gcky9Mg-9nRWXdNN7Br-9meWEhALaGiT6wUTfqRJbGOa5g=)[[3]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG5Jlr7KHPpcEk-qUzUfy-humD1D-QT1o-KzhUwyg2vUI8q_hTkVYFeRIk3lCnY0la-EfT978trnIbSof9yFg0UgrlQbnRgSMz4VYjq5KHSUsuD0y4N7wxAokWgYsAK0b69rg470qsHiksyNaXMH188tDJP2iTYKmC72S06cQX4NbJIQcpsHJron79Hdw==)

This allows "uphill" moves to higher-energy states with a probability that decreases as temperature `T` is lowered [[10]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEZIIyyYXX9HQ0eYhV9nRya-frdKGBakS-5V1wZ5V-GKtEazcnCjsH-ayB8MLRsQ-OW_CIyW7T3MTYnT675-4lIkBEjsCBhslIEG8xYNSnOxgfFpVMhLA5XRsZ-gMCMx-R-ZkdQyxYVqpKsyhi9EbjW2bEbHMMO)[[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEHHHGEvBYLwcVWKjMVUn886rAfaVlOYIRLupGSHCSollwL5NjhlU2PPXVK_SN5if7RJFsmY71cCsUx5EmvyBTX_AulkDBkTDQxu3AKdh6F4u-DLQvpRYG-bD0UzYik_gcky9Mg-9nRWXdNN7Br-9meWEhALaGiT6wUTfqRJbGOa5g=)[[17]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQF-5OpxoCmublJ2diq-oMtD3fZsmpt_UISftailCDR1K_C9sW1_JqbF14O150eI7xC3HEMSwbggZzTDHdFBP7aqd4hQmXUkSdNHtNUt60F97kA_rhDmU1cRdBRos7vg5T6LN9GuiCCsx7zTOCm6nKxEE0A6GgK0S95awgXYqZ7fa_gpitn60hbOjZLTCcL330YrR7lFqbO7cjxDFv8Hb0HKsyUi5SkqZd2zhDMX7ekzSTU=) .

#### 4. The Complete Equation for a Temperature-Controlled Graph Walk

By combining the PPR target distribution, the degree correction, and the temperature parameter, we can construct a sophisticated acceptance probability for a move from node `u` to `v`:

> **α(v, u) = min( 1, [ (π_PPR(v) / π_PPR(u))^(1/T) ] * [ deg(u) / deg(v) ] )**

This powerful equation integrates:
1.  The ratio of target probabilities from Personalized PageRank (`π_PPR`).
2.  A temperature `T` to control the exploration-exploitation trade-off.
3.  The Hastings correction term (`deg(u)/deg(v)`) for the asymmetric random walk proposal.

### General Applications

The versatility of the Metropolis-Hastings algorithm has led to its widespread adoption in:

*   **Bayesian Inference**: It is a primary tool for sampling from posterior distributions, which are often complex and have no closed-form solution [[15]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH-yKT2ZGzyavv1voSGSLJ1a7kYhtIwkJu4D1LNR6QqKaZgrtcrsfW8hBmvcxWHfOCMqPmHKOHQ1Sz9_j2DEZyoAYjyNZeEe6HHbK4yQCeOkA6mdA_16iwqzF-7EFnrSBP3JpD43A==)[[5]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHDBTgOfLV-2q4tjX8zB2D2WDSbe-h6gxKftK1DUNJWK3SfmZRWZZkhS2rgeyQGrqMVtl7kNfyRVDE0QqAlls2v-1GUR3Qe7VlsxG1rNB2c4OOkpzu0_7QTb2z1Fb7cCxpHzJZCaw==) .
*   **Statistical Physics**: The original Metropolis algorithm was developed to study the equilibrium states of physical systems [[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEHHHGEvBYLwcVWKjMVUn886rAfaVlOYIRLupGSHCSollwL5NjhlU2PPXVK_SN5if7RJFsmY71cCsUx5EmvyBTX_AulkDBkTDQxu3AKdh6F4u-DLQvpRYG-bD0UzYik_gcky9Mg-9nRWXdNN7Br-9meWEhALaGiT6wUTfqRJbGOa5g=) .
*   **Machine Learning**: It is used for inference in probabilistic models and for training generative models [[3]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG5Jlr7KHPpcEk-qUzUfy-humD1D-QT1o-KzhUwyg2vUI8q_hTkVYFeRIk3lCnY0la-EfT978trnIbSof9yFg0UgrlQbnRgSMz4VYjq5KHSUsuD0y4N7wxAokWgYsAK0b69rg470qsHiksyNaXMH188tDJP2iTYKmC72S06cQX4NbJIQcpsHJron79Hdw==) .
*   **Global Optimization**: The simulated annealing variant uses a temperature parameter to find the global minimum of a function [[9]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFGHYHEmrLmQc5J3eDqrxLIW_LIaAvXjVcRku3s6yn9cFF_hwCAHw8YiowD8eZlIqlAXHhJWSika_hy9J58yj4r4rRzbs-XCTioAPnCqp8Diujx82AxT0ECp6St)[[11]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGC0w_TwTc5TCGecdPaXAw6Qo2q0pfSCVFTXausRaTVbwVB2VG_5EyP8Y91KnxMSFZoNRkjVWdz2gtldqmDw8zpKHOPBcafLszXwREEvtV5rLr0Uo08lRo5IuvfkJL0G269lHzqFU8KECpqkeaWPs04zH2Z1SxUVnS4ViWGQ6-lEx0x) .
*   **Numerical Integration**: It can be used to estimate high-dimensional integrals by averaging function values over the samples drawn from the distribution [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHi-fWVDR8fuS34305HX3hlvka1jNnGyBd6jzkNhaytQAojmqp5U1q4ROsxufQGy0lSlppz_F-w7sBc0iAGPVLbaDnZnUheU5W7dLVCGTz-s0wHgsc15s_OwkhVoI8pEmHiCPTRTt8KI-k3NIbz3N8wuSaQssBPxy400HnKcQ==) .

### Executive Summary

The Metropolis-Hastings algorithm is a robust MCMC method for sampling from a probability distribution `π(x)`. Its core is the acceptance probability equation, `α = min(1, [π(x')q(x|x')] / [π(x)q(x'|x)])`, which uses a proposal distribution `q` to explore the state space and decides whether to accept new states. The algorithm's mathematical foundation is the **detailed balance condition**, `π(x)P(x'|x) = π(x')P(x|x')`, which the acceptance probability is specifically constructed to satisfy, thereby guaranteeing that the generated samples converge to the target distribution `π(x)`. A key feature is that it only requires `π(x)` to be known up to a constant, making it broadly applicable.

The algorithm's flexibility is demonstrated in diverse applications, from Bayesian inference to complex graph analysis. For instance, in a **random walk on a graph**, the proposal `q(v|u) = 1/deg(u)` is asymmetric, requiring a **degree correction** term `deg(u)/deg(v)` in the acceptance probability. This allows sampling from arbitrary target distributions, such as the one defined by **Personalized PageRank (PPR)**. Furthermore, the algorithm can be adapted for optimization through techniques like **Simulated Annealing**, which introduces a temperature parameter `T` to modify the target distribution to `π_T(x) ∝ [π(x)]^(1/T)`. This allows the algorithm to balance exploration (at high `T`) and exploitation (at low `T`) to find a global optimum. A complete formula combining these concepts for a graph walk is `α(v, u) = min(1, [ (π_PPR(v) / π_PPR(u))^(1/T) ] * [ deg(u) / deg(v) ])`.

Of course! Here is the revised research document on the Metropolis-Hastings algorithm, updated to include a detailed section on Gibbs Sampling, which is a highly relevant special case of the Metropolis-Hastings algorithm, perfect for your presentation.

### **The Metropolis-Hastings Algorithm: Key Equations and Concepts for MCMC**

The Metropolis-Hastings algorithm is a cornerstone of computational statistics and a powerful Markov chain Monte Carlo (MCMC) method . It is used to generate a sequence of random samples from a probability distribution, which is especially useful when direct sampling is difficult or impossible [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEqERRtU19c_OiHulbkObilkii10EP0cflaeXeIynioY3oG2feEiyKEnPfGNqzNXmSj6ZTExYR2NKAP0dQs98d6O7NSDW2AcXQ9sX2RJ26DH0dmT97iRHCqV8jLjLKk16kcdZYpPrmsLdmoAmTMlbjHUCKksbdp3xoipJMooMrb_e9FP0SALzehHrwkx-8g17qzRoCkL5ZU5FRw7rCq) . This is a common scenario in Bayesian statistics, statistical physics, and machine learning, where one often encounters complex, high-dimensional distributions known only up to a constant of proportionality [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEsNYRfktqhMFNEymo6MRcdQhgolfgv7LOvqqwqlgje5fyUHhLoHD9AKMpFomtkYQtRhTBuYN3yR_t4QP0CVqZtks_ohghRiIJgs-HBdUIojug5kMl_xJ2SDKYJdyRKvfrkeTSrve1HqRgb9qPmNia6azWFDiif5KH5JiEB9w==)[[3]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFwZrWr5_NH1GdmaHlCh3dRku63M-7hiZXtC28Mxn9Q1f8elDo8WKQbiDwfNNJjvU4CIFc_VAAby8pRF-c-9buLja-PQvbvLl8LQK3pkQdQUu4UD_vTrqj04nCWMrzz1vZjoiygMCYHXw==) .

At its core, the algorithm constructs a Markov chain—a sequence of random variables where the future state depends only on the current state—whose stationary distribution is the target distribution you want to sample from [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEqERRtU19c_OiHulbkObilkii10EP0cflaeXeIynioY3oG2feEiyKEnPfGNqzNXmSj6ZTExYR2NKAP0dQs98d6O7NSDW2AcXQ9sX2RJ26DH0dmT97iRHCqV8jLjLKk16kcdZYpPrmsLdmoAmTMlbjHUCKksbdp3xoipJMooMrb_e9FP0SALzehHrwkx-8g17qzRoCkL5ZU5FRw7rCq)[[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE6zoQZfz35016yGzIkTyC3ikVi2QG2q41fXtNfddzlMSFyaq798EkF5i3jSt1uBJ49zpmS6MCC6YvkUDshN3kT8NNaS6ECjxDSefTRY0Zm68f1KlgxF0tjXKj96WK_bj6geWltMFDcUqNmalQDNFtkYGK7aOT9URW8) . It achieves this by iteratively generating a candidate sample from a proposal distribution and then deciding whether to accept or reject it based on a specific probability [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEsNYRfktqhMFNEymo6MRcdQhgolfgv7LOvqqwqlgje5fyUHhLoHD9AKMpFomtkYQtRhTBuYN3yR_t4QP0CVqZtks_ohghRiIJgs-HBdUIojug5kMl_xJ2SDKYJdyRKvfrkeTSrve1HqRgb9qPmNia6azWFDiif5KH5JiEB9w==)[[5]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHJwRbpy65QtNLU5ZFpiNBZA9QAPwpZvSIL4yL8QGEZRQJl2ePC-nRXBwOn59mk00-dkq195AKrTX9_HbofNffRZZwUuVx_sibNCQSYj8e9yRlxN7M0ocp3rp9ghjNYCIVT0TSHTW-64DJEtWECOm5k9nzC2ruwfm59xxTWKinBqHZF6roBmzyVRTcxcqn13X0wZYKhHhHGuuAXHiYql3n8ZtV23HxILBfE_8Z4I9h2HlM=) . Over many iterations, the collection of accepted samples forms a representative sample of the desired distribution [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEqERRtU19c_OiHulbkObilkii10EP0cflaeXeIynioY3oG2feEiyKEnPfGNqzNXmSj6ZTExYR2NKAP0dQs98d6O7NSDW2AcXQ9sX2RJ26DH0dmT97iRHCqV8jLjLKk16kcdZYpPrmsLdmoAmTMlbjHUCKksbdp3xoipJMooMrb_e9FP0SALzehHrwkx-8g17qzRoCkL5ZU5FRw7rCq) .

---

### **The General Metropolis-Hastings Algorithm**

The algorithm proceeds through a simple yet powerful iterative process. Let's say we want to sample from a target probability distribution `P(x)`.

1.  **Initialization**: Start with an initial value for the parameter, `x_t`.

2.  **Proposal**: Generate a new candidate value, `x'`, from a **proposal distribution**, `q(x'|x_t)`. This distribution defines how to "propose" a new state given the current state `x_t` [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEsNYRfktqhMFNEymo6MRcdQhgolfgv7LOvqqwqlgje5fyUHhLoHD9AKMpFomtkYQtRhTBuYN3yR_t4QP0CVqZtks_ohghRiIJgs-HBdUIojug5kMl_xJ2SDKYJdyRKvfrkeTSrve1HqRgb9qPmNia6azWFDiif5KH5JiEB9w==)[[5]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHJwRbpy65QtNLU5ZFpiNBZA9QAPwpZvSIL4yL8QGEZRQJl2ePC-nRXBwOn59mk00-dkq195AKrTX9_HbofNffRZZwUuVx_sibNCQSYj8e9yRlxN7M0ocp3rp9ghjNYCIVT0TSHTW-64DJEtWECOm5k9nzC2ruwfm59xxTWKinBqHZF6roBmzyVRTcxcqn13X0wZYKhHhHGuuAXHiYql3n8ZtV23HxILBfE_8Z4I9h2HlM=) .

3.  **Calculate Acceptance Probability**: Compute the acceptance probability, `α`, which determines the likelihood of accepting the new candidate `x'`. The formula is the heart of the algorithm:

    $$
    \alpha = \min\left(1, \frac{P(x') q(x_t|x')}{P(x_t) q(x'|x_t)}\right)
    $$

    Where:
    *   **`P(x)`** is the target probability distribution (or a function proportional to it).
    *   **`q(x'|x_t)`** is the forward proposal probability: the probability of proposing `x'` given the current state `x_t` .
    *   **`q(x_t|x')`** is the reverse proposal probability: the probability of proposing `x_t` from the new state `x'` [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEqERRtU19c_OiHulbkObilkii10EP0cflaeXeIynioY3oG2feEiyKEnPfGNqzNXmSj6ZTExYR2NKAP0dQs98d6O7NSDW2AcXQ9sX2RJ26DH0dmT97iRHCqV8jLjLKk16kcdZYpPrmsLdmoAmTMlbjHUCKksbdp3xoipJMooMrb_e9FP0SALzehHrwkx-8g17qzRoCkL5ZU5FRw7rCq) .

4.  **Accept or Reject**: Generate a random number `u` from a uniform distribution between 0 and 1, `u ~ U(0, 1)`.
    *   If `u ≤ α`, **accept** the new candidate: `x_{t+1} = x'`.
    *   Otherwise, **reject** the candidate and the next state is the same as the current state: `x_{t+1} = x_t`.

5.  **Repeat**: Repeat steps 2-4 for a large number of iterations. After an initial "burn-in" period, the collected samples `x_t` will be drawn from the target distribution `P(x)` [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEsNYRfktqhMFNEymo6MRcdQhgolfgv7LOvqqwqlgje5fyUHhLoHD9AKMpFomtkYQtRhTBuYN3yR_t4QP0CVqZtks_ohghRiIJgs-HBdUIojug5kMl_xJ2SDKYJdyRKvfrkeTSrve1HqRgb9qPmNia6azWFDiif5KH5JiEB9w==) .

### **Key Mathematical Principles**

*   **Detailed Balance (Reversibility)**: The algorithm's convergence is guaranteed by the fact that the acceptance probability `α` is constructed to satisfy the detailed balance condition [[3]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFwZrWr5_NH1GdmaHlCh3dRku63M-7hiZXtC28Mxn9Q1f8elDo8WKQbiDwfNNJjvU4CIFc_VAAby8pRF-c-9buLja-PQvbvLl8LQK3pkQdQUu4UD_vTrqj04nCWMrzz1vZjoiygMCYHXw==)[[6]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEWIu_Y6vinTr4NTf5ulEF5gdQa5TOlhswKhWmqvaq6K4kemLLt7aWypJQbmGLoat7nY3LiH4eplIrTRvt7GUtAkpahvO3vJk1xFXpcjUrff2R-18NT90MQMdmFjRcyRlJmw6lqjSvEE2f0ay5LS28Y)[[7]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE4to9G-t9PXMcqj6uIcHAFlXOxcaeFvntnIMMrI9rm7tn-eakbRGXr4Tj9ASXTJi2xGK-76NLHuAMAVBbfbMTP_H8F39Zo5ogp9kJQw8v_yXwdws9GiEUjA53IhCScmwlRrlb9b40nBiWyMfNdwb-3BmUdMnDgFKQl3vHoX7mHKSouc8V-SBxWif4fsKqEQ4bzYprRA5EMaxYaK6oGCARdZO0VcZ7PCG-JFygByfEZARN2885OVz4I)[[8]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHHBr3lRNSXcGzJUdgKjf3en_Iz6ERUJEiuu26XauWXnXmS7hAd7Ctkhu5FffIwOUnYT8ByiqgFK8fGR1RQl9q-x6P6qsCLF72dWWH_X3CvctQDslwaLdAlozjpj0nsc7JkvHr9rdIiZGp7PTOoqX4Qlj-dmjvPS8gjSzwM8Ohv) . This condition ensures that the transition rates between any two states `i` and `j` are balanced with respect to the target distribution `π`:
    $$
    \pi(i) P(i \to j) = \pi(j) P(j \to i)
    $$
    Here, `P(i → j)` is the overall transition probability. The Metropolis-Hastings acceptance ratio is precisely the component needed to enforce this balance .

*   **Transition Kernel**: The complete single-step transition probability, known as the transition kernel `K(x, y)`, can be formally written. It describes the probability of moving from state `x` to state `y`:
    $$
    K(x, y) = \alpha(x, y) q(y|x) + \delta_x(y) \int (1 - \alpha(x, z))q(z|x)dz
    $$
    Here, `δ_x(y)` is the Dirac delta function. This equation states that the probability of moving to a new state `y` is the probability of proposing `y` and accepting it, while the probability of staying at `x` includes the probability of rejecting all other proposed moves [[3]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFwZrWr5_NH1GdmaHlCh3dRku63M-7hiZXtC28Mxn9Q1f8elDo8WKQbiDwfNNJjvU4CIFc_VAAby8pRF-c-9buLja-PQvbvLl8LQK3pkQdQUu4UD_vTrqj04nCWMrzz1vZjoiygMCYHXw==)[[7]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE4to9G-t9PXMcqj6uIcHAFlXOxcaeFvntnIMMrI9rm7tn-eakbRGXr4Tj9ASXTJi2xGK-76NLHuAMAVBbfbMTP_H8F39Zo5ogp9kJQw8v_yXwdws9GiEUjA53IhCScmwlRrlb9b40nBiWyMfNdwb-3BmUdMnDgFKQl3vHoX7mHKSouc8V-SBxWif4fsKqEQ4bzYprRA5EMaxYaK6oGCARdZO0VcZ7PCG-JFygByfEZARN2885OVz4I) .

---

### **Common Implementations and Proposal Choices**

The choice of the proposal distribution `q` is critical and defines different "flavors" of the algorithm.

*   **The Metropolis Algorithm (Symmetric Proposal)**: The original algorithm proposed by Metropolis et al. used a symmetric proposal distribution, where `q(x'|x_t) = q(x_t|x')` [[5]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHJwRbpy65QtNLU5ZFpiNBZA9QAPwpZvSIL4yL8QGEZRQJl2ePC-nRXBwOn59mk00-dkq195AKrTX9_HbofNffRZZwUuVx_sibNCQSYj8e9yRlxN7M0ocp3rp9ghjNYCIVT0TSHTW-64DJEtWECOm5k9nzC2ruwfm59xxTWKinBqHZF6roBmzyVRTcxcqn13X0wZYKhHhHGuuAXHiYql3n8ZtV23HxILBfE_8Z4I9h2HlM=) . In this case, the proposal densities in the acceptance ratio cancel out, simplifying the formula significantly:
    $$
    \alpha = \min\left(1, \frac{P(x')}{P(x_t)}\right)
    $$

*   **Random-Walk Metropolis-Hastings (RWMH)**: This is a very common implementation where the proposal is generated by adding a random perturbation to the current state: `x' = x_t + ε_t` . The perturbation `ε_t` is drawn from a symmetric distribution centered at zero, like a Normal distribution, making the proposal symmetric and thus using the simplified Metropolis acceptance rule .

*   **Independent Metropolis-Hastings**: In this variant, the proposal `x'` is drawn from a distribution `q(x')` that is independent of the current state `x_t` [[9]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQF7q_j4rEW4so9A0IqAm1dCcjNivBC8NvJx_QaQ2dZcJr2NkBv20qDZLtVogrBToE5M5VqU09NzEyXGace83uMdG961iDjnaGgbmWzVtQ-iPw_SB6OLew0EJUL8GWCsAAGm89stDpIufhopNkPz5tO-aRan7K1wuBXBvhMaER0w0PeWPKFJH6G2ETuL) . This requires using the full Metropolis-Hastings acceptance formula as the proposal is generally not symmetric.

### **Gibbs Sampling: A Special Case with Guaranteed Acceptance**

Gibbs Sampling is another widely used MCMC algorithm, first described by Stuart and Donald Geman in 1984 [[10]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGbRPexR70URrDq_dDIN2gbrf95MhJt5imU6s1f04frhWcvVCOcdtfeoOP2dFd2wyaOwm9PWQ4rC8WghY3XNlVFU0ZXSojRKp0AcgNLH0y2EgCnTM7LDzKSQ0EF9EK2JhF33Zt-drTSjHD4fgIYxDxEcQ==)[[11]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGL79R4obsYXg5GitiZr45me2eeJz8gXjv-O5XHln4hv9jonfIsU939zuxtRLco-9zZ5UCYRt103pFMt3KqWB57iJOcvdC7DMBxQAaTkA3bUr1J5RfQuH2JsK86elY7OxZv1FIP_8zqREdemvFOUO0K6Kr6oZbvuNaxnRpH9XBYwE0dTNgWa9eBNO8=) . It can be viewed as a special case of the Metropolis-Hastings algorithm [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEqERRtU19c_OiHulbkObilkii10EP0cflaeXeIynioY3oG2feEiyKEnPfGNqzNXmSj6ZTExYR2NKAP0dQs98d6O7NSDW2AcXQ9sX2RJ26DH0dmT97iRHCqV8jLjLKk16kcdZYpPrmsLdmoAmTMlbjHUCKksbdp3xoipJMooMrb_e9FP0SALzehHrwkx-8g17qzRoCkL5ZU5FRw7rCq)[[12]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE_l1KzP4l7Qgm57qvbCESCZGdshIgA8p1w91NS9OilNxTQpQ-Jnkr4wRUMFMJTIar7c7fWtQuW8HtwS7VqAV67o2__AfCgI8SvSyspuT9lc-Yh4b_XUobYy1GqlFKE6NQ9kLUevzTWFe4=)[[10]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGbRPexR70URrDq_dDIN2gbrf95MhJt5imU6s1f04frhWcvVCOcdtfeoOP2dFd2wyaOwm9PWQ4rC8WghY3XNlVFU0ZXSojRKp0AcgNLH0y2EgCnTM7LDzKSQ0EF9EK2JhF33Zt-drTSjHD4fgIYxDxEcQ==)[[13]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHC1E61KpQ5c7ZZREVpv3eaAeoivoAxihsLuimHtqGwvRgLJojwPxwJEHEwRrDOoohiCeOrBEi03T_0OKLruB0xmhpFnmmJopQUmLjxpWP_iYzvkaGXG-5CyFiNS2ATXOlLKe2Xj6II6SdBP99yVVbHNlDXPar4GVJxsR7HY20=) . Its power lies in breaking down a complex, high-dimensional sampling problem into a series of simpler, one-dimensional sampling steps [[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE6zoQZfz35016yGzIkTyC3ikVi2QG2q41fXtNfddzlMSFyaq798EkF5i3jSt1uBJ49zpmS6MCC6YvkUDshN3kT8NNaS6ECjxDSefTRY0Zm68f1KlgxF0tjXKj96WK_bj6geWltMFDcUqNmalQDNFtkYGK7aOT9URW8)[[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEsNYRfktqhMFNEymo6MRcdQhgolfgv7LOvqqwqlgje5fyUHhLoHD9AKMpFomtkYQtRhTBuYN3yR_t4QP0CVqZtks_ohghRiIJgs-HBdUIojug5kMl_xJ2SDKYJdyRKvfrkeTSrve1HqRgb9qPmNia6azWFDiif5KH5JiEB9w==)[[10]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGbRPexR70URrDq_dDIN2gbrf95MhJt5imU6s1f04frhWcvVCOcdtfeoOP2dFd2wyaOwm9PWQ4rC8WghY3XNlVFU0ZXSojRKp0AcgNLH0y2EgCnTM7LDzKSQ0EF9EK2JhF33Zt-drTSjHD4fgIYxDxEcQ==) . It is particularly effective in Bayesian inference [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEqERRtU19c_OiHulbkObilkii10EP0cflaeXeIynioY3oG2feEiyKEnPfGNqzNXmSj6ZTExYR2NKAP0dQs98d6O7NSDW2AcXQ9sX2RJ26DH0dmT97iRHCqV8jLjLKk16kcdZYpPrmsLdmoAmTMlbjHUCKksbdp3xoipJMooMrb_e9FP0SALzehHrwkx-8g17qzRoCkL5ZU5FRw7rCq)[[10]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGbRPexR70URrDq_dDIN2gbrf95MhJt5imU6s1f04frhWcvVCOcdtfeoOP2dFd2wyaOwm9PWQ4rC8WghY3XNlVFU0ZXSojRKp0AcgNLH0y2EgCnTM7LDzKSQ0EF9EK2JhF33Zt-drTSjHD4fgIYxDxEcQ==)[[11]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGL79R4obsYXg5GitiZr45me2eeJz8gXjv-O5XHln4hv9jonfIsU939zuxtRLco-9zZ5UCYRt103pFMt3KqWB57iJOcvdC7DMBxQAaTkA3bUr1J5RfQuH2JsK86elY7OxZv1FIP_8zqREdemvFOUO0K6Kr6oZbvuNaxnRpH9XBYwE0dTNgWa9eBNO8=) .

#### **The Core Condition: Full Conditional Distributions**

The crucial requirement for using Gibbs sampling is that while the joint distribution `P(x) = P(x₁, x₂, ..., xₙ)` may be too complex to sample from, you must be able to sample from the **full conditional distribution (FCD)** for each variable `xᵢ` [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEqERRtU19c_OiHulbkObilkii10EP0cflaeXeIynioY3oG2feEiyKEnPfGNqzNXmSj6ZTExYR2NKAP0dQs98d6O7NSDW2AcXQ9sX2RJ26DH0dmT97iRHCqV8jLjLKk16kcdZYpPrmsLdmoAmTMlbjHUCKksbdp3xoipJMooMrb_e9FP0SALzehHrwkx-8g17qzRoCkL5ZU5FRw7rCq)[[5]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHJwRbpy65QtNLU5ZFpiNBZA9QAPwpZvSIL4yL8QGEZRQJl2ePC-nRXBwOn59mk00-dkq195AKrTX9_HbofNffRZZwUuVx_sibNCQSYj8e9yRlxN7M0ocp3rp9ghjNYCIVT0TSHTW-64DJEtWECOm5k9nzC2ruwfm59xxTWKinBqHZF6roBmzyVRTcxcqn13X0wZYKhHhHGuuAXHiYql3n8ZtV23HxILBfE_8Z4I9h2HlM=)[[14]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEG-zbpZQ3fgcuLz5I0ilNfyipOJ8IKDU7yU8cDgwjZDXDMYJt2-fGKx9y7KKgBsnRD3GedXPi_RskkyJr_RXewbjRoXuNG3UgL7K1tk_4xHeXcLBIfsjKUUjo=)[[15]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGYbu6dGzE6E3cRkmbipuvn_kTqQ6oGGW1Zh97pGfgu4LRgELSuQgA3JglxJyqXkQkVL0mTeXB1onnU01NPFvxCf6WyrzzrK5h2taaq6MukF52pTN_u1mTCNrauNND3F-AVHdmMRi3xmeu8kyny1KfFrlnY2vCY8bmfJfg=) . The FCD for `xᵢ` is its distribution conditioned on the current values of all other variables, `x₋ᵢ` [[5]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHJwRbpy65QtNLU5ZFpiNBZA9QAPwpZvSIL4yL8QGEZRQJl2ePC-nRXBwOn59mk00-dkq195AKrTX9_HbofNffRZZwUuVx_sibNCQSYj8e9yRlxN7M0ocp3rp9ghjNYCIVT0TSHTW-64DJEtWECOm5k9nzC2ruwfm59xxTWKinBqHZF6roBmzyVRTcxcqn13X0wZYKhHhHGuuAXHiYql3n8ZtV23HxILBfE_8Z4I9h2HlM=)[[3]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFwZrWr5_NH1GdmaHlCh3dRku63M-7hiZXtC28Mxn9Q1f8elDo8WKQbiDwfNNJjvU4CIFc_VAAby8pRF-c-9buLja-PQvbvLl8LQK3pkQdQUu4UD_vTrqj04nCWMrzz1vZjoiygMCYHXw==)[[6]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEWIu_Y6vinTr4NTf5ulEF5gdQa5TOlhswKhWmqvaq6K4kemLLt7aWypJQbmGLoat7nY3LiH4eplIrTRvt7GUtAkpahvO3vJk1xFXpcjUrff2R-18NT90MQMdmFjRcyRlJmw6lqjSvEE2f0ay5LS28Y) :

$$
P(x_i | \mathbf{x}_{-i}) = P(x_i | x_1, \dots, x_{i-1}, x_{i+1}, \dots, x_n)
$$

In many Bayesian models, these FCDs are often standard, well-known distributions (like Normal, Gamma, or Beta), making them easy to sample from, even when the joint posterior is intractable [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEqERRtU19c_OiHulbkObilkii10EP0cflaeXeIynioY3oG2feEiyKEnPfGNqzNXmSj6ZTExYR2NKAP0dQs98d6O7NSDW2AcXQ9sX2RJ26DH0dmT97iRHCqV8jLjLKk16kcdZYpPrmsLdmoAmTMlbjHUCKksbdp3xoipJMooMrb_e9FP0SALzehHrwkx-8g17qzRoCkL5ZU5FRw7rCq)[[5]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHJwRbpy65QtNLU5ZFpiNBZA9QAPwpZvSIL4yL8QGEZRQJl2ePC-nRXBwOn59mk00-dkq195AKrTX9_HbofNffRZZwUuVx_sibNCQSYj8e9yRlxN7M0ocp3rp9ghjNYCIVT0TSHTW-64DJEtWECOm5k9nzC2ruwfm59xxTWKinBqHZF6roBmzyVRTcxcqn13X0wZYKhHhHGuuAXHiYql3n8ZtV23HxILBfE_8Z4I9h2HlM=)[[7]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE4to9G-t9PXMcqj6uIcHAFlXOxcaeFvntnIMMrI9rm7tn-eakbRGXr4Tj9ASXTJi2xGK-76NLHuAMAVBbfbMTP_H8F39Zo5ogp9kJQw8v_yXwdws9GiEUjA53IhCScmwlRrlb9b40nBiWyMfNdwb-3BmUdMnDgFKQl3vHoX7mHKSouc8V-SBxWif4fsKqEQ4bzYprRA5EMaxYaK6oGCARdZO0VcZ7PCG-JFygByfEZARN2885OVz4I)[[15]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGYbu6dGzE6E3cRkmbipuvn_kTqQ6oGGW1Zh97pGfgu4LRgELSuQgA3JglxJyqXkQkVL0mTeXB1onnU01NPFvxCf6WyrzzrK5h2taaq6MukF52pTN_u1mTCNrauNND3F-AVHdmMRi3xmeu8kyny1KfFrlnY2vCY8bmfJfg=)[[16]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFk3xPS0BBxEnw-7-U9eYZxoNAHGg6FJqvX4eQVKBm4E49sDkpQfRH9keqZUdWOdF0nSsfJX1p2e-3rqqVBLqNbg71QbpjiKi8DUV43hy39UKGpEGVESzhY_221RdJrZ81sGlRhvZggb9ySou-TYVYX) . A key property is that the FCD is proportional to the joint distribution when viewed as a function of `xᵢ` with all other variables held constant [[10]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGbRPexR70URrDq_dDIN2gbrf95MhJt5imU6s1f04frhWcvVCOcdtfeoOP2dFd2wyaOwm9PWQ4rC8WghY3XNlVFU0ZXSojRKp0AcgNLH0y2EgCnTM7LDzKSQ0EF9EK2JhF33Zt-drTSjHD4fgIYxDxEcQ==) .

#### **The Iterative Update Rule**

Starting with an initial state `x⁽⁰⁾`, the Gibbs sampler generates the next state `x⁽ᵗ⁺¹⁾` by cycling through each variable and drawing a new value from its FCD, using the most recently updated values for the other variables [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEqERRtU19c_OiHulbkObilkii10EP0cflaeXeIynioY3oG2feEiyKEnPfGNqzNXmSj6ZTExYR2NKAP0dQs98d6O7NSDW2AcXQ9sX2RJ26DH0dmT97iRHCqV8jLjLKk16kcdZYpPrmsLdmoAmTMlbjHUCKksbdp3xoipJMooMrb_e9FP0SALzehHrwkx-8g17qzRoCkL5ZU5FRw7rCq)[[3]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFwZrWr5_NH1GdmaHlCh3dRku63M-7hiZXtC28Mxn9Q1f8elDo8WKQbiDwfNNJjvU4CIFc_VAAby8pRF-c-9buLja-PQvbvLl8LQK3pkQdQUu4UD_vTrqj04nCWMrzz1vZjoiygMCYHXw==)[[13]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHC1E61KpQ5c7ZZREVpv3eaAeoivoAxihsLuimHtqGwvRgLJojwPxwJEHEwRrDOoohiCeOrBEi03T_0OKLruB0xmhpFnmmJopQUmLjxpWP_iYzvkaGXG-5CyFiNS2ATXOlLKe2Xj6II6SdBP99yVVbHNlDXPar4GVJxsR7HY20=)[[17]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGYEwRRutYUEDSo9KvjVdPBscaXRyPAM0MHi3fh54r71Q6VwI4vDRJUYrUYPd5zyHDM8A_y3mWXwSSlG1YUsU5FtdOlE0WMTyVn7B77_hw23u--q4RezNnZW5FRMBymsm2h6MSVmQ833QWHJ-Lxkp2CO0lENXE7eUFKGhJ91_N2ZT6BIff8f8vk5N5yirMTKhyK5XPTEboglI5PEX1o-zurKPF-9vVlUYAYOQ==) . A full iteration looks like this:

1.  Draw `x₁⁽ᵗ⁺¹⁾` from `P(x₁ | x₂⁽ᵗ⁾, x₃⁽ᵗ⁾, ..., xₙ⁽ᵗ⁾)`
2.  Draw `x₂⁽ᵗ⁺¹⁾` from `P(x₂ | x₁⁽ᵗ⁺¹⁾, x₃⁽ᵗ⁾, ..., xₙ⁽ᵗ⁾)`
3.  ...
4.  Draw `xᵢ⁽ᵗ⁺¹⁾` from `P(xᵢ | x₁⁽ᵗ⁺¹⁾, ..., xᵢ₋₁⁽ᵗ⁺¹⁾, xᵢ₊₁⁽ᵗ⁾, ..., xₙ⁽ᵗ⁾)`
5.  ...
6.  Draw `xₙ⁽ᵗ⁺¹⁾` from `P(xₙ | x₁⁽ᵗ⁺¹⁾, x₂⁽ᵗ⁺¹⁾, ..., xₙ₋₁⁽ᵗ⁺¹⁾)`

This process is repeated for many iterations, and after a burn-in period, the collected samples `x⁽ᵗ⁾` form a sample from the joint distribution `P(x)` [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEqERRtU19c_OiHulbkObilkii10EP0cflaeXeIynioY3oG2feEiyKEnPfGNqzNXmSj6ZTExYR2NKAP0dQs98d6O7NSDW2AcXQ9sX2RJ26DH0dmT97iRHCqV8jLjLKk16kcdZYpPrmsLdmoAmTMlbjHUCKksbdp3xoipJMooMrb_e9FP0SALzehHrwkx-8g17qzRoCkL5ZU5FRw7rCq)[[18]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG1nNLYaSi0ewB_aKTnGPySE7sWS5LTogP5Xc_oDDXoNshNthKCeSpT9VHvtxLpLnJiWhU6Cpb4fejkkac9ROrc76_KLnSIcbbj17FGypjpQ9AXa2ldlSEvJMhtWnwFMpsjjz9n4rs9BMEdANR_4R96y0i6t1_QFus2wGkd1q9OWrzZNYAyw6C3_30aTpg8MEoKe88PsEiEiZn4qtsyt7lvWWWwf47D0kpMPg==)[[19]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHNyGqmt5NRKRq1svCc5fxvhg2wTcHJFylA0fuAQzO0qMLxOirKX9uXokRZUc4GZM_SuVaQqzUDIlKqkzJkIo4x7u39KltEgDtuBbQB9BPJmJImS9pN-CmAuRcXmmpkmz4eCJtuAvzy8d2HXesTpnzIB2aXv5MEa0m3jBkJkPGtCD5SxRyA0jpba4iJh2Y3iQ6Jq-5c2WdC5WoNeWUXzr1plmo=) .

#### **Why Gibbs Sampling is a Valid MCMC Method**

The sequence of samples `x⁽⁰⁾, x⁽¹⁾, ...` forms a Markov chain whose stationary distribution is the target joint distribution `P(x)` [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEqERRtU19c_OiHulbkObilkii10EP0cflaeXeIynioY3oG2feEiyKEnPfGNqzNXmSj6ZTExYR2NKAP0dQs98d6O7NSDW2AcXQ9sX2RJ26DH0dmT97iRHCqV8jLjLKk16kcdZYpPrmsLdmoAmTMlbjHUCKksbdp3xoipJMooMrb_e9FP0SALzehHrwkx-8g17qzRoCkL5ZU5FRw7rCq)[[7]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE4to9G-t9PXMcqj6uIcHAFlXOxcaeFvntnIMMrI9rm7tn-eakbRGXr4Tj9ASXTJi2xGK-76NLHuAMAVBbfbMTP_H8F39Zo5ogp9kJQw8v_yXwdws9GiEUjA53IhCScmwlRrlb9b40nBiWyMfNdwb-3BmUdMnDgFKQl3vHoX7mHKSouc8V-SBxWif4fsKqEQ4bzYprRA5EMaxYaK6oGCARdZO0VcZ7PCG-JFygByfEZARN2885OVz4I)[[13]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHC1E61KpQ5c7ZZREVpv3eaAeoivoAxihsLuimHtqGwvRgLJojwPxwJEHEwRrDOoohiCeOrBEi03T_0OKLruB0xmhpFnmmJopQUmLjxpWP_iYzvkaGXG-5CyFiNS2ATXOlLKe2Xj6II6SdBP99yVVbHNlDXPar4GVJxsR7HY20=)[[18]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG1nNLYaSi0ewB_aKTnGPySE7sWS5LTogP5Xc_oDDXoNshNthKCeSpT9VHvtxLpLnJiWhU6Cpb4fejkkac9ROrc76_KLnSIcbbj17FGypjpQ9AXa2ldlSEvJMhtWnwFMpsjjz9n4rs9BMEdANR_4R96y0i6t1_QFus2wGkd1q9OWrzZNYAyw6C3_30aTpg8MEoKe88PsEiEiZn4qtsyt7lvWWWwf47D0kpMPg==) . This is guaranteed because Gibbs sampling is a special case of Metropolis-Hastings where the proposal is always accepted [[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE6zoQZfz35016yGzIkTyC3ikVi2QG2q41fXtNfddzlMSFyaq798EkF5i3jSt1uBJ49zpmS6MCC6YvkUDshN3kT8NNaS6ECjxDSefTRY0Zm68f1KlgxF0tjXKj96WK_bj6geWltMFDcUqNmalQDNFtkYGK7aOT9URW8)[[12]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE_l1KzP4l7Qgm57qvbCESCZGdshIgA8p1w91NS9OilNxTQpQ-Jnkr4wRUMFMJTIar7c7fWtQuW8HtwS7VqAV67o2__AfCgI8SvSyspuT9lc-Yh4b_XUobYy1GqlFKE6NQ9kLUevzTWFe4=)[[20]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEINQNuwP-YOlHjFuVPQU1Z2VCvr5YXQH7AtwAaLUITwyR6z4555YpxmzQdSb78V6VycK9BwyZxKxJDmei3QWPL7RMgCkWLkfKjBcBRu_jepwdiB1f8yIQniP25xhFDT3hoLgN-uyYab92-H2i50-KHIzyJdy1OSXaVo5gCtgDhW-hOuN0lVO2kzSocopbXhcMhB-n7hWmZPnr-S5pqQL8=) .

Let's see why. For updating a single component `xᵢ`, the proposal `x'` is drawn from the full conditional distribution: `q(x'|x) = P(x'ᵢ | x₋ᵢ)` . The Metropolis-Hastings acceptance probability is:

$$
\alpha = \min\left(1, \frac{P(x') q(x|x')}{P(x) q(x'|x)}\right)
$$

We can expand the terms:
*   `P(x') = P(x'ᵢ | x₋ᵢ) P(x₋ᵢ)`
*   `P(x) = P(xᵢ | x₋ᵢ) P(x₋ᵢ)`
*   `q(x'|x) = P(x'ᵢ | x₋ᵢ)`
*   `q(x|x') = P(xᵢ | x'₋ᵢ) = P(xᵢ | x₋ᵢ)` (since only `xᵢ` changes, `x'₋ᵢ = x₋ᵢ`)

Substituting these into the ratio inside the `min` function:

$$
\frac{P(x') q(x|x')}{P(x) q(x'|x)} = \frac{P(x'_i | x_{-i}) P(x_{-i})}{P(x_i | x_{-i}) P(x_{-i})} \times \frac{P(x_i | x_{-i})}{P(x'_i | x_{-i})} = 1
$$

Since the ratio is 1, the acceptance probability `α = min(1, 1) = 1` [[20]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEINQNuwP-YOlHjFuVPQU1Z2VCvr5YXQH7AtwAaLUITwyR6z4555YpxmzQdSb78V6VycK9BwyZxKxJDmei3QWPL7RMgCkWLkfKjBcBRu_jepwdiB1f8yIQniP25xhFDT3hoLgN-uyYab92-H2i50-KHIzyJdy1OSXaVo5gCtgDhW-hOuN0lVO2kzSocopbXhcMhB-n7hWmZPnr-S5pqQL8=) . This means every proposal is accepted, and the detailed balance condition is satisfied, guaranteeing convergence to the correct target distribution [[7]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE4to9G-t9PXMcqj6uIcHAFlXOxcaeFvntnIMMrI9rm7tn-eakbRGXr4Tj9ASXTJi2xGK-76NLHuAMAVBbfbMTP_H8F39Zo5ogp9kJQw8v_yXwdws9GiEUjA53IhCScmwlRrlb9b40nBiWyMfNdwb-3BmUdMnDgFKQl3vHoX7mHKSouc8V-SBxWif4fsKqEQ4bzYprRA5EMaxYaK6oGCARdZO0VcZ7PCG-JFygByfEZARN2885OVz4I)[[8]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHHBr3lRNSXcGzJUdgKjf3en_Iz6ERUJEiuu26XauWXnXmS7hAd7Ctkhu5FffIwOUnYT8ByiqgFK8fGR1RQl9q-x6P6qsCLF72dWWH_X3CvctQDslwaLdAlozjpj0nsc7JkvHr9rdIiZGp7PTOoqX4Qlj-dmjvPS8gjSzwM8Ohv)[[13]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHC1E61KpQ5c7ZZREVpv3eaAeoivoAxihsLuimHtqGwvRgLJojwPxwJEHEwRrDOoohiCeOrBEi03T_0OKLruB0xmhpFnmmJopQUmLjxpWP_iYzvkaGXG-5CyFiNS2ATXOlLKe2Xj6II6SdBP99yVVbHNlDXPar4GVJxsR7HY20=)[[21]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGODXnrlgJI-rlXCbP3teR_4kGgicIPTrrXNTHVQE3JFNeqLmLKQsuS2KHOAmbAmUMZYzCswWryFJ6-zFD8sNJXSi_srs2lj0MWTtFOJGNeXr9tVSSCDzsOfGbbxFgSqf9D9sn9_uOe1DQA4KNh_KAZisS0J-X-RzKr) . For convergence from any starting point, the chain must be irreducible, meaning it can reach all parts of the state space [[13]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHC1E61KpQ5c7ZZREVpv3eaAeoivoAxihsLuimHtqGwvRgLJojwPxwJEHEwRrDOoohiCeOrBEi03T_0OKLruB0xmhpFnmmJopQUmLjxpWP_iYzvkaGXG-5CyFiNS2ATXOlLKe2Xj6II6SdBP99yVVbHNlDXPar4GVJxsR7HY20=)[[21]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGODXnrlgJI-rlXCbP3teR_4kGgicIPTrrXNTHVQE3JFNeqLmLKQsuS2KHOAmbAmUMZYzCswWryFJ6-zFD8sNJXSi_srs2lj0MWTtFOJGNeXr9tVSSCDzsOfGbbxFgSqf9D9sn9_uOe1DQA4KNh_KAZisS0J-X-RzKr) .

---
## **Advanced MCMC Variants**

For many problems, we can improve sampling efficiency by using more information about the target distribution `P(x)` or by using more sophisticated structures. Advanced MCMC methods can use the gradient of the log-target density, `∇log P(x)`, to propose moves towards regions of higher probability, or employ multiple interacting chains to explore the sample space more effectively .

### **Metropolis-Adjusted Langevin Algorithm (MALA)**

MALA is a powerful MCMC method that uses gradient information to guide its proposals [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEqERRtU19c_OiHulbkObilkii10EP0cflaeXeIynioY3oG2feEiyKEnPfGNqzNXmSj6ZTExYR2NKAP0dQs98d6O7NSDW2AcXQ9sX2RJ26DH0dmT97iRHCqV8jLjLKk16kcdZYpPrmsLdmoAmTMlbjHUCKksbdp3xoipJMooMrb_e9FP0SALzehHrwkx-8g17qzRoCkL5ZU5FRw7rCq)[[6]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEWIu_Y6vinTr4NTf5ulEF5gdQa5TOlhswKhWmqvaq6K4kemLLt7aWypJQbmGLoat7nY3LiH4eplIrTRvt7GUtAkpahvO3vJk1xFXpcjUrff2R-18NT90MQMdmFjRcyRlJmw6lqjSvEE2f0ay5LS28Y) .

#### **Foundation in Langevin Diffusion**

MALA is derived from the discretization of a continuous-time process called Langevin diffusion, described by the following Stochastic Differential Equation (SDE) [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEqERRtU19c_OiHulbkObilkii10EP0cflaeXeIynioY3oG2feEiyKEnPfGNqzNXmSj6ZTExYR2NKAP0dQs98d6O7NSDW2AcXQ9sX2RJ26DH0dmT97iRHCqV8jLjLKk16kcdZYpPrmsLdmoAmTMlbjHUCKksbdp3xoipJMooMrb_e9FP0SALzehHrwkx-8g17qzRoCkL5ZU5FRw7rCq)[[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE6zoQZfz35016yGzIkTyC3ikVi2QG2q41fXtNfddzlMSFyaq798EkF5i3jSt1uBJ49zpmS6MCC6YvkUDshN3kT8NNaS6ECjxDSefTRY0Zm68f1KlgxF0tjXKj96WK_bj6geWltMFDcUqNmalQDNFtkYGK7aOT9URW8) :

$$
dX_t = \nabla\log P(X_t)dt + \sqrt{2} dW_t
$$

*   The **drift term**, `∇log P(X_t)dt`, pushes the state `X_t` towards areas where the probability density `P(x)` is higher .
*   The **diffusion term**, `√2 dW_t`, represents random fluctuations from a standard Brownian motion `W_t` [[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE6zoQZfz35016yGzIkTyC3ikVi2QG2q41fXtNfddzlMSFyaq798EkF5i3jSt1uBJ49zpmS6MCC6YvkUDshN3kT8NNaS6ECjxDSefTRY0Zm68f1KlgxF0tjXKj96WK_bj6geWltMFDcUqNmalQDNFtkYGK7aOT9URW8)[[7]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE4to9G-t9PXMcqj6uIcHAFlXOxcaeFvntnIMMrI9rm7tn-eakbRGXr4Tj9ASXTJi2xGK-76NLHuAMAVBbfbMTP_H8F39Zo5ogp9kJQw8v_yXwdws9GiEUjA53IhCScmwlRrlb9b40nBiWyMfNdwb-3BmUdMnDgFKQl3vHoX7mHKSouc8V-SBxWif4fsKqEQ4bzYprRA5EMaxYaK6oGCARdZO0VcZ7PCG-JFygByfEZARN2885OVz4I) .
The unique stationary distribution of this SDE is exactly our target distribution `P(x)` [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEsNYRfktqhMFNEymo6MRcdQhgolfgv7LOvqqwqlgje5fyUHhLoHD9AKMpFomtkYQtRhTBuYN3yR_t4QP0CVqZtks_ohghRiIJgs-HBdUIojug5kMl_xJ2SDKYJdyRKvfrkeTSrve1HqRgb9qPmNia6azWFDiif5KH5JiEB9w==)[[9]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQF7q_j4rEW4so9A0IqAm1dCcjNivBC8NvJx_QaQ2dZcJr2NkBv20qDZLtVogrBToE5M5VqU09NzEyXGace83uMdG961iDjnaGgbmWzVtQ-iPw_SB6OLew0EJUL8GWCsAAGm89stDpIufhopNkPz5tO-aRan7K1wuBXBvhMaER0w0PeWPKFJH6G2ETuL)[[12]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE_l1KzP4l7Qgm57qvbCESCZGdshIgA8p1w91NS9OilNxTQpQ-Jnkr4wRUMFMJTIar7c7fWtQuW8HtwS7VqAV67o2__AfCgI8SvSyspuT9lc-Yh4b_XUobYy1GqlFKE6NQ9kLUevzTWFe4=) .

#### **The MALA Proposal Equation**

To implement this, we discretize the SDE using the **Euler-Maruyama method** with a small step size `ε` [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEqERRtU19c_OiHulbkObilkii10EP0cflaeXeIynioY3oG2feEiyKEnPfGNqzNXmSj6ZTExYR2NKAP0dQs98d6O7NSDW2AcXQ9sX2RJ26DH0dmT97iRHCqV8jLjLKk16kcdZYpPrmsLdmoAmTMlbjHUCKksbdp3xoipJMooMrb_e9FP0SALzehHrwkx-8g17qzRoCkL5ZU5FRw7rCq)[[9]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQF7q_j4rEW4so9A0IqAm1dCcjNivBC8NvJx_QaQ2dZcJr2NkBv20qDZLtVogrBToE5M5VqU09NzEyXGace83uMdG961iDjnaGgbmWzVtQ-iPw_SB6OLew0EJUL8GWCsAAGm89stDpIufhopNkPz5tO-aRan7K1wuBXBvhMaER0w0PeWPKFJH6G2ETuL) . This gives us the MALA proposal equation:

$$
x' = x_t + \epsilon \nabla \log P(x_t) + \sqrt{2\epsilon}Z
$$

Here, `Z` is a random vector drawn from a standard multivariate normal distribution, `Z ~ N(0, I)` [[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE6zoQZfz35016yGzIkTyC3ikVi2QG2q41fXtNfddzlMSFyaq798EkF5i3jSt1uBJ49zpmS6MCC6YvkUDshN3kT8NNaS6ECjxDSefTRY0Zm68f1KlgxF0tjXKj96WK_bj6geWltMFDcUqNmalQDNFtkYGK7aOT9URW8)[[5]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHJwRbpy65QtNLU5ZFpiNBZA9QAPwpZvSIL4yL8QGEZRQJl2ePC-nRXBwOn59mk00-dkq195AKrTX9_HbofNffRZZwUuVx_sibNCQSYj8e9yRlxN7M0ocp3rp9ghjNYCIVT0TSHTW-64DJEtWECOm5k9nzC2ruwfm59xxTWKinBqHZF6roBmzyVRTcxcqn13X0wZYKhHhHGuuAXHiYql3n8ZtV23HxILBfE_8Z4I9h2HlM=)[[8]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHHBr3lRNSXcGzJUdgKjf3en_Iz6ERUJEiuu26XauWXnXmS7hAd7Ctkhu5FffIwOUnYT8ByiqgFK8fGR1RQl9q-x6P6qsCLF72dWWH_X3CvctQDslwaLdAlozjpj0nsc7JkvHr9rdIiZGp7PTOoqX4Qlj-dmjvPS8gjSzwM8Ohv) . This equation defines the proposal distribution `q(x'|x_t)` as a Gaussian centered at `x_t + ε∇log P(x_t)` with covariance `2εI` [[7]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE4to9G-t9PXMcqj6uIcHAFlXOxcaeFvntnIMMrI9rm7tn-eakbRGXr4Tj9ASXTJi2xGK-76NLHuAMAVBbfbMTP_H8F39Zo5ogp9kJQw8v_yXwdws9GiEUjA53IhCScmwlRrlb9b40nBiWyMfNdwb-3BmUdMnDgFKQl3vHoX7mHKSouc8V-SBxWif4fsKqEQ4bzYprRA5EMaxYaK6oGCARdZO0VcZ7PCG-JFygByfEZARN2885OVz4I)[[9]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQF7q_j4rEW4so9A0IqAm1dCcjNivBC8NvJx_QaQ2dZcJr2NkBv20qDZLtVogrBToE5M5VqU09NzEyXGace83uMdG961iDjnaGgbmWzVtQ-iPw_SB6OLew0EJUL8GWCsAAGm89stDpIufhopNkPz5tO-aRan7K1wuBXBvhMaER0w0PeWPKFJH6G2ETuL) .

#### **The MALA Acceptance Probability**

The discretization introduces a small error, which is corrected by a Metropolis-Hastings acceptance step [[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE6zoQZfz35016yGzIkTyC3ikVi2QG2q41fXtNfddzlMSFyaq798EkF5i3jSt1uBJ49zpmS6MCC6YvkUDshN3kT8NNaS6ECjxDSefTRY0Zm68f1KlgxF0tjXKj96WK_bj6geWltMFDcUqNmalQDNFtkYGK7aOT9URW8)[[3]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFwZrWr5_NH1GdmaHlCh3dRku63M-7hiZXtC28Mxn9Q1f8elDo8WKQbiDwfNNJjvU4CIFc_VAAby8pRF-c-9buLja-PQvbvLl8LQK3pkQdQUu4UD_vTrqj04nCWMrzz1vZjoiygMCYHXw==)[[9]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQF7q_j4rEW4so9A0IqAm1dCcjNivBC8NvJx_QaQ2dZcJr2NkBv20qDZLtVogrBToE5M5VqU09NzEyXGace83uMdG961iDjnaGgbmWzVtQ-iPw_SB6OLew0EJUL8GWCsAAGm89stDpIufhopNkPz5tO-aRan7K1wuBXBvhMaER0w0PeWPKFJH6G2ETuL) . Because the drift term makes the proposal non-symmetric, we must use the full acceptance ratio:

$$
\alpha(x', x_t) = \min\left(1, \frac{P(x') q(x_t|x')}{P(x_t) q(x'|x_t)}\right)
$$

Substituting the Gaussian densities for `q` and simplifying gives the final **MALA acceptance probability**:

$$
\alpha(x', x_t) = \min\left(1, \frac{P(x')}{P(x_t)} \exp\left[ \frac{1}{4\epsilon} \left( \|x' - x_t - \epsilon \nabla \log P(x_t)\|^2 - \|x_t - x' - \epsilon \nabla \log P(x')\|^2 \right) \right]\right)
$$

This correction factor ensures that MALA satisfies detailed balance and samples correctly from `P(x)` [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEqERRtU19c_OiHulbkObilkii10EP0cflaeXeIynioY3oG2feEiyKEnPfGNqzNXmSj6ZTExYR2NKAP0dQs98d6O7NSDW2AcXQ9sX2RJ26DH0dmT97iRHCqV8jLjLKk16kcdZYpPrmsLdmoAmTMlbjHUCKksbdp3xoipJMooMrb_e9FP0SALzehHrwkx-8g17qzRoCkL5ZU5FRw7rCq)[[7]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE4to9G-t9PXMcqj6uIcHAFlXOxcaeFvntnIMMrI9rm7tn-eakbRGXr4Tj9ASXTJi2xGK-76NLHuAMAVBbfbMTP_H8F39Zo5ogp9kJQw8v_yXwdws9GiEUjA53IhCScmwlRrlb9b40nBiWyMfNdwb-3BmUdMnDgFKQl3vHoX7mHKSouc8V-SBxWif4fsKqEQ4bzYprRA5EMaxYaK6oGCARdZO0VcZ7PCG-JFygByfEZARN2885OVz4I) .

### **Hamiltonian Monte Carlo (HMC): A Physics-Inspired Approach**

Another powerful, gradient-based MCMC method is Hamiltonian Monte Carlo (HMC) [[21]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGODXnrlgJI-rlXCbP3teR_4kGgicIPTrrXNTHVQE3JFNeqLmLKQsuS2KHOAmbAmUMZYzCswWryFJ6-zFD8sNJXSi_srs2lj0MWTtFOJGNeXr9tVSSCDzsOfGbbxFgSqf9D9sn9_uOe1DQA4KNh_KAZisS0J-X-RzKr)[[22]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH-BYJxFzTUXDI6WgF_jBvL-wyEfGaYEg29tXwICKnMvBGvDN43N2SNo_TybrucoJFPiZgXf5ZfMRiuC0wV15iHPks8CsZ2J3lBrunvzuQHKl_B0nMWboUbAS4R8mg=) . It leverages principles from Hamiltonian dynamics to propose new states that are distant from the current state but have a very high probability of being accepted [[23]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHYDUsKG_2c2ppcOJ5aRS72oYBEwFrmZwdVsGb9dnR7oANz7q-emceeIngXzvrIg2B12VIIoGEk_eUU_mBR3j8j03NKk4wLKhHuXbdE2hbe4KPMBSr66SywUuOYP0GSZ4UBoC-Q2MDnkHpcHW-GepN3XqJGAlKEGHYwDHC_)[[10]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGbRPexR70URrDq_dDIN2gbrf95MhJt5imU6s1f04frhWcvVCOcdtfeoOP2dFd2wyaOwm9PWQ4rC8WghY3XNlVFU0ZXSojRKp0AcgNLH0y2EgCnTM7LDzKSQ0EF9EK2JhF33Zt-drTSjHD4fgIYxDxEcQ==)[[21]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGODXnrlgJI-rlXCbP3teR_4kGgicIPTrrXNTHVQE3JFNeqLmLKQsuS2KHOAmbAmUMZYzCswWryFJ6-zFD8sNJXSi_srs2lj0MWTtFOJGNeXr9tVSSCDzsOfGbbxFgSqf9D9sn9_uOe1DQA4KNh_KAZisS0J-X-RzKr) . This allows HMC to explore the target distribution more efficiently and with less autocorrelation between samples than simpler methods like RWMH, especially in high-dimensional problems [[13]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHC1E61KpQ5c7ZZREVpv3eaAeoivoAxihsLuimHtqGwvRgLJojwPxwJEHEwRrDOoohiCeOrBEi03T_0OKLruB0xmhpFnmmJopQUmLjxpWP_iYzvkaGXG-5CyFiNS2ATXOlLKe2Xj6II6SdBP99yVVbHNlDXPar4GVJxsR7HY20=)[[11]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGL79R4obsYXg5GitiZr45me2eeJz8gXjv-O5XHln4hv9jonfIsU939zuxtRLco-9zZ5UCYRt103pFMt3KqWB57iJOcvdC7DMBxQAaTkA3bUr1J5RfQuH2JsK86elY7OxZv1FIP_8zqREdemvFOUO0K6Kr6oZbvuNaxnRpH9XBYwE0dTNgWa9eBNO8=)[[24]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEEEQfbuPeP6QImyRETFreLK02BGAyHFc3_J-Ep1yOe90yOEk8kbkbWcsCpsiy3yV8idAdLy-beg1VRr347hLNMmwF8mGmYMnfvzbS7CC_CCiUBkrDBJLnXSkG2zRBZRmOIU8mIItI9OMBVMQWzaSI=) .

#### **The Fundamental HMC Framework**

The core idea of HMC is to introduce auxiliary "momentum" variables, denoted by the vector `p`, to augment the state space of the parameters of interest, which are called "position" variables and denoted by `q` [[13]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHC1E61KpQ5c7ZZREVpv3eaAeoivoAxihsLuimHtqGwvRgLJojwPxwJEHEwRrDOoohiCeOrBEi03T_0OKLruB0xmhpFnmmJopQUmLjxpWP_iYzvkaGXG-5CyFiNS2ATXOlLKe2Xj6II6SdBP99yVVbHNlDXPar4GVJxsR7HY20=)[[11]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGL79R4obsYXg5GitiZr45me2eeJz8gXjv-O5XHln4hv9jonfIsU939zuxtRLco-9zZ5UCYRt103pFMt3KqWB57iJOcvdC7DMBxQAaTkA3bUr1J5RfQuH2JsK86elY7OxZv1FIP_8zqREdemvFOUO0K6Kr6oZbvuNaxnRpH9XBYwE0dTNgWa9eBNO8=)[[25]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGhB8DVL3cBs_5gmGczQwJKxBP0nZ0bnlUKaSa12y3tdYuqCiwDhuV69frFEFSbY3_U8vwWyyCoRekb6_gbi-lmymeq8O6TxtlP27c1YRK_sE3acXMUhXZIjZc0uEwfD65hSPnAoYCQ6JrkLXGg5Xs=) . The dynamics of this augmented `(q, p)` system are then governed by a **Hamiltonian function**, `H(q, p)`, which represents the total energy of a hypothetical physical system [[3]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFwZrWr5_NH1GdmaHlCh3dRku63M-7hiZXtC28Mxn9Q1f8elDo8WKQbiDwfNNJjvU4CIFc_VAAby8pRF-c-9buLja-PQvbvLl8LQK3pkQdQUu4UD_vTrqj04nCWMrzz1vZjoiygMCYHXw==)[[6]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEWIu_Y6vinTr4NTf5ulEF5gdQa5TOlhswKhWmqvaq6K4kemLLt7aWypJQbmGLoat7nY3LiH4eplIrTRvt7GUtAkpahvO3vJk1xFXpcjUrff2R-18NT90MQMdmFjRcyRlJmw6lqjSvEE2f0ay5LS28Y)[[15]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGYbu6dGzE6E3cRkmbipuvn_kTqQ6oGGW1Zh97pGfgu4LRgELSuQgA3JglxJyqXkQkVL0mTeXB1onnU01NPFvxCf6WyrzzrK5h2taaq6MukF52pTN_u1mTCNrauNND3F-AVHdmMRi3xmeu8kyny1KfFrlnY2vCY8bmfJfg=)[[21]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGODXnrlgJI-rlXCbP3teR_4kGgicIPTrrXNTHVQE3JFNeqLmLKQsuS2KHOAmbAmUMZYzCswWryFJ6-zFD8sNJXSi_srs2lj0MWTtFOJGNeXr9tVSSCDzsOfGbbxFgSqf9D9sn9_uOe1DQA4KNh_KAZisS0J-X-RzKr) . This function is the sum of a potential energy term `U(q)` and a kinetic energy term `K(p)` [[14]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEG-zbpZQ3fgcuLz5I0ilNfyipOJ8IKDU7yU8cDgwjZDXDMYJt2-fGKx9y7KKgBsnRD3GedXPi_RskkyJr_RXewbjRoXuNG3UgL7K1tk_4xHeXcLBIfsjKUUjo=)[[21]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGODXnrlgJI-rlXCbP3teR_4kGgicIPTrrXNTHVQE3JFNeqLmLKQsuS2KHOAmbAmUMZYzCswWryFJ6-zFD8sNJXSi_srs2lj0MWTtFOJGNeXr9tVSSCDzsOfGbbxFgSqf9D9sn9_uOe1DQA4KNh_KAZisS0J-X-RzKr)[[24]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEEEQfbuPeP6QImyRETFreLK02BGAyHFc3_J-Ep1yOe90yOEk8kbkbWcsCpsiy3yV8idAdLy-beg1VRr347hLNMmwF8mGmYMnfvzbS7CC_CCiUBkrDBJLnXSkG2zRBZRmOIU8mIItI9OMBVMQWzaSI=) .

$$
H(q, p) = U(q) + K(p)
$$

The joint probability distribution for `q` and `p` is then defined by the canonical distribution from statistical mechanics [[9]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQF7q_j4rEW4so9A0IqAm1dCcjNivBC8NvJx_QaQ2dZcJr2NkBv20qDZLtVogrBToE5M5VqU09NzEyXGace83uMdG961iDjnaGgbmWzVtQ-iPw_SB6OLew0EJUL8GWCsAAGm89stDpIufhopNkPz5tO-aRan7K1wuBXBvhMaER0w0PeWPKFJH6G2ETuL)[[18]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG1nNLYaSi0ewB_aKTnGPySE7sWS5LTogP5Xc_oDDXoNshNthKCeSpT9VHvtxLpLnJiWhU6Cpb4fejkkac9ROrc76_KLnSIcbbj17FGypjpQ9AXa2ldlSEvJMhtWnwFMpsjjz9n4rs9BMEdANR_4R96y0i6t1_QFus2wGkd1q9OWrzZNYAyw6C3_30aTpg8MEoKe88PsEiEiZn4qtsyt7lvWWWwf47D0kpMPg==)[[22]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH-BYJxFzTUXDI6WgF_jBvL-wyEfGaYEg29tXwICKnMvBGvDN43N2SNo_TybrucoJFPiZgXf5ZfMRiuC0wV15iHPks8CsZ2J3lBrunvzuQHKl_B0nMWboUbAS4R8mg=) :

$$
p(q, p) \propto \exp(-H(q, p)) = \exp(-U(q))\exp(-K(p)) \propto p(q)p(p)
$$

#### **Components of the Hamiltonian**

*   **Potential Energy `U(q)`**: This is derived directly from the target probability density `p(q)` that we want to sample from [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEsNYRfktqhMFNEymo6MRcdQhgolfgv7LOvqqwqlgje5fyUHhLoHD9AKMpFomtkYQtRhTBuYN3yR_t4QP0CVqZtks_ohghRiIJgs-HBdUIojug5kMl_xJ2SDKYJdyRKvfrkeTSrve1HqRgb9qPmNia6azWFDiif5KH5JiEB9w==)[[6]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEWIu_Y6vinTr4NTf5ulEF5gdQa5TOlhswKhWmqvaq6K4kemLLt7aWypJQbmGLoat7nY3LiH4eplIrTRvt7GUtAkpahvO3vJk1xFXpcjUrff2R-18NT90MQMdmFjRcyRlJmw6lqjSvEE2f0ay5LS28Y)[[15]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGYbu6dGzE6E3cRkmbipuvn_kTqQ6oGGW1Zh97pGfgu4LRgELSuQgA3JglxJyqXkQkVL0mTeXB1onnU01NPFvxCf6WyrzzrK5h2taaq6MukF52pTN_u1mTCNrauNND3F-AVHdmMRi3xmeu8kyny1KfFrlnY2vCY8bmfJfg=) . It is defined as the negative logarithm of the target density, creating an analogy where high-probability regions correspond to low-energy valleys [[14]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEG-zbpZQ3fgcuLz5I0ilNfyipOJ8IKDU7yU8cDgwjZDXDMYJt2-fGKx9y7KKgBsnRD3GedXPi_RskkyJr_RXewbjRoXuNG3UgL7K1tk_4xHeXcLBIfsjKUUjo=)[[21]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGODXnrlgJI-rlXCbP3teR_4kGgicIPTrrXNTHVQE3JFNeqLmLKQsuS2KHOAmbAmUMZYzCswWryFJ6-zFD8sNJXSi_srs2lj0MWTtFOJGNeXr9tVSSCDzsOfGbbxFgSqf9D9sn9_uOe1DQA4KNh_KAZisS0J-X-RzKr)[[25]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGhB8DVL3cBs_5gmGczQwJKxBP0nZ0bnlUKaSa12y3tdYuqCiwDhuV69frFEFSbY3_U8vwWyyCoRekb6_gbi-lmymeq8O6TxtlP27c1YRK_sE3acXMUhXZIjZc0uEwfD65hSPnAoYCQ6JrkLXGg5Xs=)[[26]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHLdQS9gELTO1omtk9vUo9TH-FOL1tW-bJqth5PtMIO-L4tgZ-hp3GfbrZjpkHvlyufad5KCqtCH1-UZYLstz91vQDoLrG845nYa_soReJdwMxut-2DTkfZoXJaWIHaZ64fwLGes44x1FlXsrfnuV2lPZG3Pq_G0LcFcBElMJJL6xbFY4EGa97Y4zCMCiU0mxLG6TY=) .
    $$
    U(q) = -\log p(q)
    $$
    The gradient of the potential energy, `∇U(q)`, acts as a "force" that guides the sampler towards regions of higher probability [[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE6zoQZfz35016yGzIkTyC3ikVi2QG2q41fXtNfddzlMSFyaq798EkF5i3jSt1uBJ49zpmS6MCC6YvkUDshN3kT8NNaS6ECjxDSefTRY0Zm68f1KlgxF0tjXKj96WK_bj6geWltMFDcUqNmalQDNFtkYGK7aOT9URW8)[[7]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE4to9G-t9PXMcqj6uIcHAFlXOxcaeFvntnIMMrI9rm7tn-eakbRGXr4Tj9ASXTJi2xGK-76NLHuAMAVBbfbMTP_H8F39Zo5ogp9kJQw8v_yXwdws9GiEUjA53IhCScmwlRrlb9b40nBiWyMfNdwb-3BmUdMnDgFKQl3vHoX7mHKSouc8V-SBxWif4fsKqEQ4bzYprRA5EMaxYaK6oGCARdZO0VcZ7PCG-JFygByfEZARN2885OVz4I)[[16]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFk3xPS0BBxEnw-7-U9eYZxoNAHGg6FJqvX4eQVKBm4E49sDkpQfRH9keqZUdWOdF0nSsfJX1p2e-3rqqVBLqNbg71QbpjiKi8DUV43hy39UKGpEGVESzhY_221RdJrZ81sGlRhvZggb9ySou-TYVYX)[[26]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHLdQS9gELTO1omtk9vUo9TH-FOL1tW-bJqth5PtMIO-L4tgZ-hp3GfbrZjpkHvlyufad5KCqtCH1-UZYLstz91vQDoLrG845nYa_soReJdwMxut-2DTkfZoXJaWIHaZ64fwLGes44x1FlXsrfnuV2lPZG3Pq_G0LcFcBElMJJL6xbFY4EGa97Y4zCMCiU0mxLG6TY=) .

*   **Kinetic Energy `K(p)`**: This is a function of the fictitious momentum variables `p` [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEqERRtU19c_OiHulbkObilkii10EP0cflaeXeIynioY3oG2feEiyKEnPfGNqzNXmSj6ZTExYR2NKAP0dQs98d6O7NSDW2AcXQ9sX2RJ26DH0dmT97iRHCqV8jLjLKk16kcdZYpPrmsLdmoAmTMlbjHUCKksbdp3xoipJMooMrb_e9FP0SALzehHrwkx-8g17qzRoCkL5ZU5FRw7rCq)[[9]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQF7q_j4rEW4so9A0IqAm1dCcjNivBC8NvJx_QaQ2dZcJr2NkBv20qDZLtVogrBToE5M5VqU09NzEyXGace83uMdG961iDjnaGgbmWzVtQ-iPw_SB6OLew0EJUL8GWCsAAGm89stDpIufhopNkPz5tO-aRan7K1wuBXBvhMaER0w0PeWPKFJH6G2ETuL) . It is typically defined as a quadratic form, where `p` is drawn from a zero-mean multivariate Gaussian distribution [[13]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHC1E61KpQ5c7ZZREVpv3eaAeoivoAxihsLuimHtqGwvRgLJojwPxwJEHEwRrDOoohiCeOrBEi03T_0OKLruB0xmhpFnmmJopQUmLjxpWP_iYzvkaGXG-5CyFiNS2ATXOlLKe2Xj6II6SdBP99yVVbHNlDXPar4GVJxsR7HY20=)[[16]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFk3xPS0BBxEnw-7-U9eYZxoNAHGg6FJqvX4eQVKBm4E49sDkpQfRH9keqZUdWOdF0nSsfJX1p2e-3rqqVBLqNbg71QbpjiKi8DUV43hy39UKGpEGVESzhY_221RdJrZ81sGlRhvZggb9ySou-TYVYX)[[22]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH-BYJxFzTUXDI6WgF_jBvL-wyEfGaYEg29tXwICKnMvBGvDN43N2SNo_TybrucoJFPiZgXf5ZfMRiuC0wV15iHPks8CsZ2J3lBrunvzuQHKl_B0nMWboUbAS4R8mg=)[[24]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEEEQfbuPeP6QImyRETFreLK02BGAyHFc3_J-Ep1yOe90yOEk8kbkbWcsCpsiy3yV8idAdLy-beg1VRr347hLNMmwF8mGmYMnfvzbS7CC_CCiUBkrDBJLnXSkG2zRBZRmOIU8mIItI9OMBVMQWzaSI=) .
    $$
    K(p) = \frac{1}{2}p^T M^{-1} p
    $$
    Here, `M` is a symmetric, positive-definite "mass matrix" [[6]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEWIu_Y6vinTr4NTf5ulEF5gdQa5TOlhswKhWmqvaq6K4kemLLt7aWypJQbmGLoat7nY3LiH4eplIrTRvt7GUtAkpahvO3vJk1xFXpcjUrff2R-18NT90MQMdmFjRcyRlJmw6lqjSvEE2f0ay5LS28Y)[[10]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGbRPexR70URrDq_dDIN2gbrf95MhJt5imU6s1f04frhWcvVCOcdtfeoOP2dFd2wyaOwm9PWQ4rC8WghY3XNlVFU0ZXSojRKp0AcgNLH0y2EgCnTM7LDzKSQ0EF9EK2JhF33Zt-drTSjHD4fgIYxDxEcQ==)[[15]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGYbu6dGzE6E3cRkmbipuvn_kTqQ6oGGW1Zh97pGfgu4LRgELSuQgA3JglxJyqXkQkVL0mTeXB1onnU01NPFvxCf6WyrzzrK5h2taaq6MukF52pTN_u1mTCNrauNND3F-AVHdmMRi3xmeu8kyny1KfFrlnY2vCY8bmfJfg=)[[22]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH-BYJxFzTUXDI6WgF_jBvL-wyEfGaYEg29tXwICKnMvBGvDN43N2SNo_TybrucoJFPiZgXf5ZfMRiuC0wV15iHPks8CsZ2J3lBrunvzuQHKl_B0nMWboUbAS4R8mg=) . In the simplest case, `M` is the identity matrix `I`, meaning the momentum variables are independent standard normal variables [[11]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGL79R4obsYXg5GitiZr45me2eeJz8gXjv-O5XHln4hv9jonfIsU939zuxtRLco-9zZ5UCYRt103pFMt3KqWB57iJOcvdC7DMBxQAaTkA3bUr1J5RfQuH2JsK86elY7OxZv1FIP_8zqREdemvFOUO0K6Kr6oZbvuNaxnRpH9XBYwE0dTNgWa9eBNO8=)[[24]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEEEQfbuPeP6QImyRETFreLK02BGAyHFc3_J-Ep1yOe90yOEk8kbkbWcsCpsiy3yV8idAdLy-beg1VRr347hLNMmwF8mGmYMnfvzbS7CC_CCiUBkrDBJLnXSkG2zRBZRmOIU8mIItI9OMBVMQWzaSI=)[[26]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHLdQS9gELTO1omtk9vUo9TH-FOL1tW-bJqth5PtMIO-L4tgZ-hp3GfbrZjpkHvlyufad5KCqtCH1-UZYLstz91vQDoLrG845nYa_soReJdwMxut-2DTkfZoXJaWIHaZ64fwLGes44x1FlXsrfnuV2lPZG3Pq_G0LcFcBElMJJL6xbFY4EGa97Y4zCMCiU0mxLG6TY=) . The mass matrix can be tuned to account for the geometry of the target distribution and improve sampling efficiency [[18]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG1nNLYaSi0ewB_aKTnGPySE7sWS5LTogP5Xc_oDDXoNshNthKCeSpT9VHvtxLpLnJiWhU6Cpb4fejkkac9ROrc76_KLnSIcbbj17FGypjpQ9AXa2ldlSEvJMhtWnwFMpsjjz9n4rs9BMEdANR_4R96y0i6t1_QFus2wGkd1q9OWrzZNYAyw6C3_30aTpg8MEoKe88PsEiEiZn4qtsyt7lvWWWwf47D0kpMPg==)[[27]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFQZcSF6g_4s9LoGO39PQlEb4ZZCGDBiWqt77tB98XUSgWsH6yHr4nnVKRq5sD58qWz80bAMae9r_laYslg7TEqHlBMjQgttSFK1sgQQB9RgogsHUH5rxLJShRhs-IrWkTjUtZVe-Te7-T17qEp) .

#### **Hamilton's Equations of Motion**

The evolution of the system through time is described by **Hamilton's equations of motion**, a set of first-order differential equations [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEqERRtU19c_OiHulbkObilkii10EP0cflaeXeIynioY3oG2feEiyKEnPfGNqzNXmSj6ZTExYR2NKAP0dQs98d6O7NSDW2AcXQ9sX2RJ26DH0dmT97iRHCqV8jLjLKk16kcdZYpPrmsLdmoAmTMlbjHUCKksbdp3xoipJMooMrb_e9FP0SALzehHrwkx-8g17qzRoCkL5ZU5FRw7rCq)[[8]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHHBr3lRNSXcGzJUdgKjf3en_Iz6ERUJEiuu26XauWXnXmS7hAd7Ctkhu5FffIwOUnYT8ByiqgFK8fGR1RQl9q-x6P6qsCLF72dWWH_X3CvctQDslwaLdAlozjpj0nsc7JkvHr9rdIiZGp7PTOoqX4Qlj-dmjvPS8gjSzwM8Ohv)[[19]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHNyGqmt5NRKRq1svCc5fxvhg2wTcHJFylA0fuAQzO0qMLxOirKX9uXokRZUc4GZM_SuVaQqzUDIlKqkzJkIo4x7u39KltEgDtuBbQB9BPJmJImS9pN-CmAuRcXmmpkmz4eCJtuAvzy8d2HXesTpnzIB2aXv5MEa0m3jBkJkPGtCD5SxRyA0jpba4iJh2Y3iQ6Jq-5c2WdC5WoNeWUXzr1plmo=)[[24]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEEEQfbuPeP6QImyRETFreLK02BGAyHFc3_J-Ep1yOe90yOEk8kbkbWcsCpsiy3yV8idAdLy-beg1VRr347hLNMmwF8mGmYMnfvzbS7CC_CCiUBkrDBJLnXSkG2zRBZRmOIU8mIItI9OMBVMQWzaSI=) . These equations describe how the system moves along a contour of constant energy `H` [[13]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHC1E61KpQ5c7ZZREVpv3eaAeoivoAxihsLuimHtqGwvRgLJojwPxwJEHEwRrDOoohiCeOrBEi03T_0OKLruB0xmhpFnmmJopQUmLjxpWP_iYzvkaGXG-5CyFiNS2ATXOlLKe2Xj6II6SdBP99yVVbHNlDXPar4GVJxsR7HY20=)[[22]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH-BYJxFzTUXDI6WgF_jBvL-wyEfGaYEg29tXwICKnMvBGvDN43N2SNo_TybrucoJFPiZgXf5ZfMRiuC0wV15iHPks8CsZ2J3lBrunvzuQHKl_B0nMWboUbAS4R8mg=) .

$$
\frac{dq}{dt} = \frac{\partial H}{\partial p} = M^{-1}p
$$
$$
\frac{dp}{dt} = -\frac{\partial H}{\partial q} = -\nabla U(q)
$$

In theory, this deterministic evolution perfectly conserves the total energy `H(q, p)` and preserves the volume of the state space (Liouville's Theorem) [[14]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEG-zbpZQ3fgcuLz5I0ilNfyipOJ8IKDU7yU8cDgwjZDXDMYJt2-fGKx9y7KKgBsnRD3GedXPi_RskkyJr_RXewbjRoXuNG3UgL7K1tk_4xHeXcLBIfsjKUUjo=)[[22]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH-BYJxFzTUXDI6WgF_jBvL-wyEfGaYEg29tXwICKnMvBGvDN43N2SNo_TybrucoJFPiZgXf5ZfMRiuC0wV15iHPks8CsZ2J3lBrunvzuQHKl_B0nMWboUbAS4R8mg=)[[25]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGhB8DVL3cBs_5gmGczQwJKxBP0nZ0bnlUKaSa12y3tdYuqCiwDhuV69frFEFSbY3_U8vwWyyCoRekb6_gbi-lmymeq8O6TxtlP27c1YRK_sE3acXMUhXZIjZc0uEwfD65hSPnAoYCQ6JrkLXGg5Xs=) . This means a proposal generated by perfectly simulating these equations would always be accepted [[25]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGhB8DVL3cBs_5gmGczQwJKxBP0nZ0bnlUKaSa12y3tdYuqCiwDhuV69frFEFSbY3_U8vwWyyCoRekb6_gbi-lmymeq8O6TxtlP27c1YRK_sE3acXMUhXZIjZc0uEwfD65hSPnAoYCQ6JrkLXGg5Xs=) .

#### **The Leapfrog Integrator**

Since solving Hamilton's equations analytically is usually impossible for complex models, a numerical integrator is required [[11]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGL79R4obsYXg5GitiZr45me2eeJz8gXjv-O5XHln4hv9jonfIsU939zuxtRLco-9zZ5UCYRt103pFMt3KqWB57iJOcvdC7DMBxQAaTkA3bUr1J5RfQuH2JsK86elY7OxZv1FIP_8zqREdemvFOUO0K6Kr6oZbvuNaxnRpH9XBYwE0dTNgWa9eBNO8=)[[24]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEEEQfbuPeP6QImyRETFreLK02BGAyHFc3_J-Ep1yOe90yOEk8kbkbWcsCpsiy3yV8idAdLy-beg1VRr347hLNMmwF8mGmYMnfvzbS7CC_CCiUBkrDBJLnXSkG2zRBZRmOIU8mIItI9OMBVMQWzaSI=) . HMC uses the **leapfrog integrator**, which is a symplectic integrator that is both time-reversible and volume-preserving [[14]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEG-zbpZQ3fgcuLz5I0ilNfyipOJ8IKDU7yU8cDgwjZDXDMYJt2-fGKx9y7KKgBsnRD3GedXPi_RskkyJr_RXewbjRoXuNG3UgL7K1tk_4xHeXcLBIfsjKUUjo=)[[18]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG1nNLYaSi0ewB_aKTnGPySE7sWS5LTogP5Xc_oDDXoNshNthKCeSpT9VHvtxLpLnJiWhU6Cpb4fejkkac9ROrc76_KLnSIcbbj17FGypjpQ9AXa2ldlSEvJMhtWnwFMpsjjz9n4rs9BMEdANR_4R96y0i6t1_QFus2wGkd1q9OWrzZNYAyw6C3_30aTpg8MEoKe88PsEiEiZn4qtsyt7lvWWWwf47D0kpMPg==)[[25]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGhB8DVL3cBs_5gmGczQwJKxBP0nZ0bnlUKaSa12y3tdYuqCiwDhuV69frFEFSbY3_U8vwWyyCoRekb6_gbi-lmymeq8O6TxtlP27c1YRK_sE3acXMUhXZIjZc0uEwfD65hSPnAoYCQ6JrkLXGg5Xs=)[[27]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFQZcSF6g_4s9LoGO39PQlEb4ZZCGDBiWqt77tB98XUSgWsH6yHr4nnVKRq5sD58qWz80bAMae9r_laYslg7TEqHlBMjQgttSFK1sgQQB9RgogsHUH5rxLJShRhs-IrWkTjUtZVe-Te7-T17qEp) . These properties are crucial for the long-term stability of the simulation and the correctness of the MH correction step [[14]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEG-zbpZQ3fgcuLz5I0ilNfyipOJ8IKDU7yU8cDgwjZDXDMYJt2-fGKx9y7KKgBsnRD3GedXPi_RskkyJr_RXewbjRoXuNG3UgL7K1tk_4xHeXcLBIfsjKUUjo=)[[25]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGhB8DVL3cBs_5gmGczQwJKxBP0nZ0bnlUKaSa12y3tdYuqCiwDhuV69frFEFSbY3_U8vwWyyCoRekb6_gbi-lmymeq8O6TxtlP27c1YRK_sE3acXMUhXZIjZc0uEwfD65hSPnAoYCQ6JrkLXGg5Xs=) .

The integrator discretizes the trajectory into `L` small steps of size `ε`. A single leapfrog step involves three updates:
1.  **Make a half-step for momentum**:
    $$
    p(t + \epsilon/2) = p(t) - (\epsilon/2) \nabla U(q(t))
    $$
2.  **Make a full-step for position**:
    $$
    q(t + \epsilon) = q(t) + \epsilon M^{-1} p(t + \epsilon/2)
    $$
3.  **Make a final half-step for momentum**:
    $$
    p(t + \epsilon) = p(t + \epsilon/2) - (\epsilon/2) \nabla U(q(t + \epsilon))
    $$
Repeating these `L` times generates the proposal state `(q*, p*)` [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEqERRtU19c_OiHulbkObilkii10EP0cflaeXeIynioY3oG2feEiyKEnPfGNqzNXmSj6ZTExYR2NKAP0dQs98d6O7NSDW2AcXQ9sX2RJ26DH0dmT97iRHCqV8jLjLKk16kcdZYpPrmsLdmoAmTMlbjHUCKksbdp3xoipJMooMrb_e9FP0SALzehHrwkx-8g17qzRoCkL5ZU5FRw7rCq)[[3]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFwZrWr5_NH1GdmaHlCh3dRku63M-7hiZXtC28Mxn9Q1f8elDo8WKQbiDwfNNJjvU4CIFc_VAAby8pRF-c-9buLja-PQvbvLl8LQK3pkQdQUu4UD_vTrqj04nCWMrzz1vZjoiygMCYHXw==)[[14]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEG-zbpZQ3fgcuLz5I0ilNfyipOJ8IKDU7yU8cDgwjZDXDMYJt2-fGKx9y7KKgBsnRD3GedXPi_RskkyJr_RXewbjRoXuNG3UgL7K1tk_4xHeXcLBIfsjKUUjo=)[[21]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGODXnrlgJI-rlXCbP3teR_4kGgicIPTrrXNTHVQE3JFNeqLmLKQsuS2KHOAmbAmUMZYzCswWryFJ6-zFD8sNJXSi_srs2lj0MWTtFOJGNeXr9tVSSCDzsOfGbbxFgSqf9D9sn9_uOe1DQA4KNh_KAZisS0J-X-RzKr)[[28]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHOiqk8WAxfKNwQ_RyBUNwttsjQmQ2uKesWd2LQZNpwicNhg92sMP-WRU5GWlzQ2RglaSoFsQ3Rmwpa7A6HhMobfKREUEXiF4lbl7OIAmSLfe10zFLht_fOPjGqN2F95Uoe_616EhjPoZ1I9LM=) .

#### **The HMC Acceptance Probability**

The leapfrog integrator introduces small numerical errors, so the Hamiltonian `H` is not perfectly conserved along the discrete trajectory [[9]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQF7q_j4rEW4so9A0IqAm1dCcjNivBC8NvJx_QaQ2dZcJr2NkBv20qDZLtVogrBToE5M5VqU09NzEyXGace83uMdG961iDjnaGgbmWzVtQ-iPw_SB6OLew0EJUL8GWCsAAGm89stDpIufhopNkPz5tO-aRan7K1wuBXBvhMaER0w0PeWPKFJH6G2ETuL)[[15]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGYbu6dGzE6E3cRkmbipuvn_kTqQ6oGGW1Zh97pGfgu4LRgELSuQgA3JglxJyqXkQkVL0mTeXB1onnU01NPFvxCf6WyrzzrK5h2taaq6MukF52pTN_u1mTCNrauNND3F-AVHdmMRi3xmeu8kyny1KfFrlnY2vCY8bmfJfg=)[[18]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG1nNLYaSi0ewB_aKTnGPySE7sWS5LTogP5Xc_oDDXoNshNthKCeSpT9VHvtxLpLnJiWhU6Cpb4fejkkac9ROrc76_KLnSIcbbj17FGypjpQ9AXa2ldlSEvJMhtWnwFMpsjjz9n4rs9BMEdANR_4R96y0i6t1_QFus2wGkd1q9OWrzZNYAyw6C3_30aTpg8MEoKe88PsEiEiZn4qtsyt7lvWWWwf47D0kpMPg==)[[25]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGhB8DVL3cBs_5gmGczQwJKxBP0nZ0bnlUKaSa12y3tdYuqCiwDhuV69frFEFSbY3_U8vwWyyCoRekb6_gbi-lmymeq8O6TxtlP27c1YRK_sE3acXMUhXZIjZc0uEwfD65hSPnAoYCQ6JrkLXGg5Xs=) . To correct for this and ensure the chain converges to the exact target distribution `p(q)`, a Metropolis-Hastings acceptance step is used [[6]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEWIu_Y6vinTr4NTf5ulEF5gdQa5TOlhswKhWmqvaq6K4kemLLt7aWypJQbmGLoat7nY3LiH4eplIrTRvt7GUtAkpahvO3vJk1xFXpcjUrff2R-18NT90MQMdmFjRcyRlJmw6lqjSvEE2f0ay5LS28Y)[[23]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHYDUsKG_2c2ppcOJ5aRS72oYBEwFrmZwdVsGb9dnR7oANz7q-emceeIngXzvrIg2B12VIIoGEk_eUU_mBR3j8j03NKk4wLKhHuXbdE2hbe4KPMBSr66SywUuOYP0GSZ4UBoC-Q2MDnkHpcHW-GepN3XqJGAlKEGHYwDHC_)[[15]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGYbu6dGzE6E3cRkmbipuvn_kTqQ6oGGW1Zh97pGfgu4LRgELSuQgA3JglxJyqXkQkVL0mTeXB1onnU01NPFvxCf6WyrzzrK5h2taaq6MukF52pTN_u1mTCNrauNND3F-AVHdmMRi3xmeu8kyny1KfFrlnY2vCY8bmfJfg=)[[20]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEINQNuwP-YOlHjFuVPQU1Z2VCvr5YXQH7AtwAaLUITwyR6z4555YpxmzQdSb78V6VycK9BwyZxKxJDmei3QWPL7RMgCkWLkfKjBcBRu_jepwdiB1f8yIQniP25xhFDT3hoLgN-uyYab92-H2i50-KHIzyJdy1OSXaVo5gCtgDhW-hOuN0lVO2kzSocopbXhcMhB-n7hWmZPnr-S5pqQL8=)[[21]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGODXnrlgJI-rlXCbP3teR_4kGgicIPTrrXNTHVQE3JFNeqLmLKQsuS2KHOAmbAmUMZYzCswWryFJ6-zFD8sNJXSi_srs2lj0MWTtFOJGNeXr9tVSSCDzsOfGbbxFgSqf9D9sn9_uOe1DQA4KNh_KAZisS0J-X-RzKr)[[28]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHOiqk8WAxfKNwQ_RyBUNwttsjQmQ2uKesWd2LQZNpwicNhg92sMP-WRU5GWlzQ2RglaSoFsQ3Rmwpa7A6HhMobfKREUEXiF4lbl7OIAmSLfe10zFLht_fOPjGqN2F95Uoe_616EhjPoZ1I9LM=) .

The acceptance probability `α` for the proposed state `(q*, p*)` is:
$$
\alpha = \min\left(1, \frac{\exp(-H(q^*, p^*))}{\exp(-H(q, p))}\right) = \min(1, \exp(H(q, p) - H(q^*, p^*)))
$$
This can be expanded as:
$$
\alpha = \min\left(1, \exp\left( (U(q) + K(p)) - (U(q^*) + K(p^*)) \right)\right)
$$
Because the leapfrog method approximates the true trajectory well, the energy error is small, and `α` is often close to 1, even for large moves [[14]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEG-zbpZQ3fgcuLz5I0ilNfyipOJ8IKDU7yU8cDgwjZDXDMYJt2-fGKx9y7KKgBsnRD3GedXPi_RskkyJr_RXewbjRoXuNG3UgL7K1tk_4xHeXcLBIfsjKUUjo=)[[25]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGhB8DVL3cBs_5gmGczQwJKxBP0nZ0bnlUKaSa12y3tdYuqCiwDhuV69frFEFSbY3_U8vwWyyCoRekb6_gbi-lmymeq8O6TxtlP27c1YRK_sE3acXMUhXZIjZc0uEwfD65hSPnAoYCQ6JrkLXGg5Xs=)[[29]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFX6mB95csJ3AfgJ65X1sHX7hS3VZNsfDAByUsFdmrbGi_BHmakNJMQsTWGzZsHeFe5WdmsBDFYqlQKM0Edc75yyVyclOqTIIUK1S5_DEq5OsMAEDrXxpUN7yxLhnroeJoljUBoXvBRmIt54ok_IZdUo6J0c4TMLFTbFNKsezY=) . If the proposal is accepted, the new state is `q*`; otherwise, the chain stays at `q` [[23]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHYDUsKG_2c2ppcOJ5aRS72oYBEwFrmZwdVsGb9dnR7oANz7q-emceeIngXzvrIg2B12VIIoGEk_eUU_mBR3j8j03NKk4wLKhHuXbdE2hbe4KPMBSr66SywUuOYP0GSZ4UBoC-Q2MDnkHpcHW-GepN3XqJGAlKEGHYwDHC_)[[20]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEINQNuwP-YOlHjFuVPQU1Z2VCvr5YXQH7AtwAaLUITwyR6z4555YpxmzQdSb78V6VycK9BwyZxKxJDmei3QWPL7RMgCkWLkfKjBcBRu_jepwdiB1f8yIQniP25xhFDT3hoLgN-uyYab92-H2i50-KHIzyJdy1OSXaVo5gCtgDhW-hOuN0lVO2kzSocopbXhcMhB-n7hWmZPnr-S5pqQL8=)[[22]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH-BYJxFzTUXDI6WgF_jBvL-wyEfGaYEg29tXwICKnMvBGvDN43N2SNo_TybrucoJFPiZgXf5ZfMRiuC0wV15iHPks8CsZ2J3lBrunvzuQHKl_B0nMWboUbAS4R8mg=) . At the beginning of each new iteration, the momentum `p` is discarded and resampled from its Gaussian distribution to allow the sampler to explore different energy contours [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEsNYRfktqhMFNEymo6MRcdQhgolfgv7LOvqqwqlgje5fyUHhLoHD9AKMpFomtkYQtRhTBuYN3yR_t4QP0CVqZtks_ohghRiIJgs-HBdUIojug5kMl_xJ2SDKYJdyRKvfrkeTSrve1HqRgb9qPmNia6azWFDiif5KH5JiEB9w==)[[13]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHC1E61KpQ5c7ZZREVpv3eaAeoivoAxihsLuimHtqGwvRgLJojwPxwJEHEwRrDOoohiCeOrBEi03T_0OKLruB0xmhpFnmmJopQUmLjxpWP_iYzvkaGXG-5CyFiNS2ATXOlLKe2Xj6II6SdBP99yVVbHNlDXPar4GVJxsR7HY20=)[[22]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH-BYJxFzTUXDI6WgF_jBvL-wyEfGaYEg29tXwICKnMvBGvDN43N2SNo_TybrucoJFPiZgXf5ZfMRiuC0wV15iHPks8CsZ2J3lBrunvzuQHKl_B0nMWboUbAS4R8mg=) .

### **Parallel Tempering (Metropolis-Coupled MCMC)**

Parallel Tempering (PT), also known as Metropolis-Coupled MCMC (MCMCMC or MC³), is an advanced MCMC method designed to improve sampling from complex, multi-modal distributions where standard samplers often get "stuck" in local modes [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEqERRtU19c_OiHulbkObilkii10EP0cflaeXeIynioY3oG2feEiyKEnPfGNqzNXmSj6ZTExYR2NKAP0dQs98d6O7NSDW2AcXQ9sX2RJ26DH0dmT97iRHCqV8jLjLKk16kcdZYpPrmsLdmoAmTMlbjHUCKksbdp3xoipJMooMrb_e9FP0SALzehHrwkx-8g17qzRoCkL5ZU5FRw7rCq)[[4]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE6zoQZfz35016yGzIkTyC3ikVi2QG2q41fXtNfddzlMSFyaq798EkF5i3jSt1uBJ49zpmS6MCC6YvkUDshN3kT8NNaS6ECjxDSefTRY0Zm68f1KlgxF0tjXKj96WK_bj6geWltMFDcUqNmalQDNFtkYGK7aOT9URW8)[[12]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE_l1KzP4l7Qgm57qvbCESCZGdshIgA8p1w91NS9OilNxTQpQ-Jnkr4wRUMFMJTIar7c7fWtQuW8HtwS7VqAV67o2__AfCgI8SvSyspuT9lc-Yh4b_XUobYy1GqlFKE6NQ9kLUevzTWFe4=)[[8]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHHBr3lRNSXcGzJUdgKjf3en_Iz6ERUJEiuu26XauWXnXmS7hAd7Ctkhu5FffIwOUnYT8ByiqgFK8fGR1RQl9q-x6P6qsCLF72dWWH_X3CvctQDslwaLdAlozjpj0nsc7JkvHr9rdIiZGp7PTOoqX4Qlj-dmjvPS8gjSzwM8Ohv)[[10]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGbRPexR70URrDq_dDIN2gbrf95MhJt5imU6s1f04frhWcvVCOcdtfeoOP2dFd2wyaOwm9PWQ4rC8WghY3XNlVFU0ZXSojRKp0AcgNLH0y2EgCnTM7LDzKSQ0EF9EK2JhF33Zt-drTSjHD4fgIYxDxEcQ==) . Instead of using gradients, it uses a structural approach, running multiple chains in parallel at different "temperatures" and allowing them to exchange information [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEsNYRfktqhMFNEymo6MRcdQhgolfgv7LOvqqwqlgje5fyUHhLoHD9AKMpFomtkYQtRhTBuYN3yR_t4QP0CVqZtks_ohghRiIJgs-HBdUIojug5kMl_xJ2SDKYJdyRKvfrkeTSrve1HqRgb9qPmNia6azWFDiif5KH5JiEB9w==)[[12]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE_l1KzP4l7Qgm57qvbCESCZGdshIgA8p1w91NS9OilNxTQpQ-Jnkr4wRUMFMJTIar7c7fWtQuW8HtwS7VqAV67o2__AfCgI8SvSyspuT9lc-Yh4b_XUobYy1GqlFKE6NQ9kLUevzTWFe4=)[[10]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGbRPexR70URrDq_dDIN2gbrf95MhJt5imU6s1f04frhWcvVCOcdtfeoOP2dFd2wyaOwm9PWQ4rC8WghY3XNlVFU0ZXSojRKp0AcgNLH0y2EgCnTM7LDzKSQ0EF9EK2JhF33Zt-drTSjHD4fgIYxDxEcQ==) .

#### **The Framework: Parallel Chains and Tempered Distributions**

The core idea is to run `N` Markov chains in parallel [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEsNYRfktqhMFNEymo6MRcdQhgolfgv7LOvqqwqlgje5fyUHhLoHD9AKMpFomtkYQtRhTBuYN3yR_t4QP0CVqZtks_ohghRiIJgs-HBdUIojug5kMl_xJ2SDKYJdyRKvfrkeTSrve1HqRgb9qPmNia6azWFDiif5KH5JiEB9w==)[[5]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHJwRbpy65QtNLU5ZFpiNBZA9QAPwpZvSIL4yL8QGEZRQJl2ePC-nRXBwOn59mk00-dkq195AKrTX9_HbofNffRZZwUuVx_sibNCQSYj8e9yRlxN7M0ocp3rp9ghjNYCIVT0TSHTW-64DJEtWECOm5k9nzC2ruwfm59xxTWKinBqHZF6roBmzyVRTcxcqn13X0wZYKhHhHGuuAXHiYql3n8ZtV23HxILBfE_8Z4I9h2HlM=)[[12]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE_l1KzP4l7Qgm57qvbCESCZGdshIgA8p1w91NS9OilNxTQpQ-Jnkr4wRUMFMJTIar7c7fWtQuW8HtwS7VqAV67o2__AfCgI8SvSyspuT9lc-Yh4b_XUobYy1GqlFKE6NQ9kLUevzTWFe4=) . Each chain `i` is assigned an inverse temperature `βᵢ` from a predefined ladder, `1 = β₁ > β₂ > ... > βₙ > 0`. Each chain then samples from a "tempered" version of the target distribution `π(x)` [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEsNYRfktqhMFNEymo6MRcdQhgolfgv7LOvqqwqlgje5fyUHhLoHD9AKMpFomtkYQtRhTBuYN3yR_t4QP0CVqZtks_ohghRiIJgs-HBdUIojug5kMl_xJ2SDKYJdyRKvfrkeTSrve1HqRgb9qPmNia6azWFDiif5KH5JiEB9w==)[[5]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHJwRbpy65QtNLU5ZFpiNBZA9QAPwpZvSIL4yL8QGEZRQJl2ePC-nRXBwOn59mk00-dkq195AKrTX9_HbofNffRZZwUuVx_sibNCQSYj8e9yRlxN7M0ocp3rp9ghjNYCIVT0TSHTW-64DJEtWECOm5k9nzC2ruwfm59xxTWKinBqHZF6roBmzyVRTcxcqn13X0wZYKhHhHGuuAXHiYql3n8ZtV23HxILBfE_8Z4I9h2HlM=) :

$$
\pi_i(x) \propto [\pi(x)]^{\beta_i}
$$

*   **The Cold Chain (β₁ = 1)**: This chain samples from the original, untempered target distribution `π(x)`. The samples from this chain are the ones used for the final analysis [[5]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHJwRbpy65QtNLU5ZFpiNBZA9QAPwpZvSIL4yL8QGEZRQJl2ePC-nRXBwOn59mk00-dkq195AKrTX9_HbofNffRZZwUuVx_sibNCQSYj8e9yRlxN7M0ocp3rp9ghjNYCIVT0TSHTW-64DJEtWECOm5k9nzC2ruwfm59xxTWKinBqHZF6roBmzyVRTcxcqn13X0wZYKhHhHGuuAXHiYql3n8ZtV23HxILBfE_8Z4I9h2HlM=)[[12]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE_l1KzP4l7Qgm57qvbCESCZGdshIgA8p1w91NS9OilNxTQpQ-Jnkr4wRUMFMJTIar7c7fWtQuW8HtwS7VqAV67o2__AfCgI8SvSyspuT9lc-Yh4b_XUobYy1GqlFKE6NQ9kLUevzTWFe4=) .
*   **Hot Chains (βᵢ < 1)**: These chains sample from "flattened" probability landscapes. As `βᵢ` gets smaller (temperature `T = 1/βᵢ` gets higher), the energy barriers between modes in the distribution are reduced, allowing these chains to move freely across the entire parameter space and explore different modes more easily [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEsNYRfktqhMFNEymo6MRcdQhgolfgv7LOvqqwqlgje5fyUHhLoHD9AKMpFomtkYQtRhTBuYN3yR_t4QP0CVqZtks_ohghRiIJgs-HBdUIojug5kMl_xJ2SDKYJdyRKvfrkeTSrve1HqRgb9qPmNia6azWFDiif5KH5JiEB9w==)[[5]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHJwRbpy65QtNLU5ZFpiNBZA9QAPwpZvSIL4yL8QGEZRQJl2ePC-nRXBwOn59mk00-dkq195AKrTX9_HbofNffRZZwUuVx_sibNCQSYj8e9yRlxN7M0ocp3rp9ghjNYCIVT0TSHTW-64DJEtWECOm5k9nzC2ruwfm59xxTWKinBqHZF6roBmzyVRTcxcqn13X0wZYKhHhHGuuAXHiYql3n8ZtV23HxILBfE_8Z4I9h2HlM=)[[12]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE_l1KzP4l7Qgm57qvbCESCZGdshIgA8p1w91NS9OilNxTQpQ-Jnkr4wRUMFMJTIar7c7fWtQuW8HtwS7VqAV67o2__AfCgI8SvSyspuT9lc-Yh4b_XUobYy1GqlFKE6NQ9kLUevzTWFe4=)[[13]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHC1E61KpQ5c7ZZREVpv3eaAeoivoAxihsLuimHtqGwvRgLJojwPxwJEHEwRrDOoohiCeOrBEi03T_0OKLruB0xmhpFnmmJopQUmLjxpWP_iYzvkaGXG-5CyFiNS2ATXOlLKe2Xj6II6SdBP99yVVbHNlDXPar4GVJxsR7HY20=) .

The algorithm alternates between two types of moves:
1.  **Intra-chain MCMC updates**: Each chain evolves independently for one or more steps using a standard MCMC algorithm (like Metropolis-Hastings) targeting its own tempered distribution `πᵢ(x)` [[11]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGL79R4obsYXg5GitiZr45me2eeJz8gXjv-O5XHln4hv9jonfIsU939zuxtRLco-9zZ5UCYRt103pFMt3KqWB57iJOcvdC7DMBxQAaTkA3bUr1J5RfQuH2JsK86elY7OxZv1FIP_8zqREdemvFOUO0K6Kr6oZbvuNaxnRpH9XBYwE0dTNgWa9eBNO8=) .
2.  **Inter-chain swap moves**: Periodically, a swap is proposed between the states of two chains, typically adjacent ones in the temperature ladder (e.g., chain `i` and `j=i+1`) [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEsNYRfktqhMFNEymo6MRcdQhgolfgv7LOvqqwqlgje5fyUHhLoHD9AKMpFomtkYQtRhTBuYN3yR_t4QP0CVqZtks_ohghRiIJgs-HBdUIojug5kMl_xJ2SDKYJdyRKvfrkeTSrve1HqRgb9qPmNia6azWFDiif5KH5JiEB9w==)[[3]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFwZrWr5_NH1GdmaHlCh3dRku63M-7hiZXtC28Mxn9Q1f8elDo8WKQbiDwfNNJjvU4CIFc_VAAby8pRF-c-9buLja-PQvbvLl8LQK3pkQdQUu4UD_vTrqj04nCWMrzz1vZjoiygMCYHXw==)[[8]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHHBr3lRNSXcGzJUdgKjf3en_Iz6ERUJEiuu26XauWXnXmS7hAd7Ctkhu5FffIwOUnYT8ByiqgFK8fGR1RQl9q-x6P6qsCLF72dWWH_X3CvctQDslwaLdAlozjpj0nsc7JkvHr9rdIiZGp7PTOoqX4Qlj-dmjvPS8gjSzwM8Ohv)[[14]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEG-zbpZQ3fgcuLz5I0ilNfyipOJ8IKDU7yU8cDgwjZDXDMYJt2-fGKx9y7KKgBsnRD3GedXPi_RskkyJr_RXewbjRoXuNG3UgL7K1tk_4xHeXcLBIfsjKUUjo=) .

#### **The Swap Acceptance Probability**

The swap move is a crucial Metropolis-Hastings step on the extended joint state of all chains, `X = (x₁, x₂, ..., xₙ)`. The joint distribution is `Π(X) = ∏ πᵢ(xᵢ)`. A proposal to swap the states of chain `i` and chain `j` is symmetric, so the Hastings ratio is 1 [[6]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEWIu_Y6vinTr4NTf5ulEF5gdQa5TOlhswKhWmqvaq6K4kemLLt7aWypJQbmGLoat7nY3LiH4eplIrTRvt7GUtAkpahvO3vJk1xFXpcjUrff2R-18NT90MQMdmFjRcyRlJmw6lqjSvEE2f0ay5LS28Y)[[14]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEG-zbpZQ3fgcuLz5I0ilNfyipOJ8IKDU7yU8cDgwjZDXDMYJt2-fGKx9y7KKgBsnRD3GedXPi_RskkyJr_RXewbjRoXuNG3UgL7K1tk_4xHeXcLBIfsjKUUjo=) . This simplifies the acceptance probability to the ratio of the joint probabilities:

$$
\alpha(x_i \leftrightarrow x_j) = \min\left(1, \frac{\Pi(X')}{\Pi(X)}\right)
$$

Expanding the ratio `Π(X') / Π(X)` for a swap between `xᵢ` and `xⱼ`, most terms cancel, leaving:

$$
\frac{\Pi(X')}{\Pi(X)} = \frac{\pi_i(x_j) \pi_j(x_i)}{\pi_i(x_i) \pi_j(x_j)} = \frac{[\pi(x_j)]^{\beta_i} [\pi(x_i)]^{\beta_j}}{[\pi(x_i)]^{\beta_i} [\pi(x_j)]^{\beta_j}} = \left[\frac{\pi(x_i)}{\pi(x_j)}\right]^{\beta_j - \beta_i}
$$

This gives the final **swap acceptance probability**:

$$
\alpha(x_i \leftrightarrow x_j) = \min\left(1, \left[\frac{\pi(x_i)}{\pi(x_j)}\right]^{\beta_j - \beta_i}\right)
$$

For numerical stability, this is often calculated using log-probabilities [[7]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE4to9G-t9PXMcqj6uIcHAFlXOxcaeFvntnIMMrI9rm7tn-eakbRGXr4Tj9ASXTJi2xGK-76NLHuAMAVBbfbMTP_H8F39Zo5ogp9kJQw8v_yXwdws9GiEUjA53IhCScmwlRrlb9b40nBiWyMfNdwb-3BmUdMnDgFKQl3vHoX7mHKSouc8V-SBxWif4fsKqEQ4bzYprRA5EMaxYaK6oGCARdZO0VcZ7PCG-JFygByfEZARN2885OVz4I) . If the target is defined by an energy function `π(x) ∝ exp(-U(x))`, the formula becomes particularly intuitive:

$$
\alpha(x_i \leftrightarrow x_j) = \min\left(1, \exp\left[(\beta_i - \beta_j)(U(x_i) - U(x_j))\right]\right)
$$
This equation shows that a swap is more likely if the hotter chain (`βᵢ`) has found a lower-energy state (`U(xᵢ)`) than the colder chain's (`βⱼ`) current high-energy state (`U(xⱼ)`) [[12]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE_l1KzP4l7Qgm57qvbCESCZGdshIgA8p1w91NS9OilNxTQpQ-Jnkr4wRUMFMJTIar7c7fWtQuW8HtwS7VqAV67o2__AfCgI8SvSyspuT9lc-Yh4b_XUobYy1GqlFKE6NQ9kLUevzTWFe4=) .

#### **Improved Exploration**

The swap mechanism is constructed to satisfy detailed balance for the joint distribution `Π(X)`, which guarantees that the cold chain correctly samples from the target `π(x)` [[11]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGL79R4obsYXg5GitiZr45me2eeJz8gXjv-O5XHln4hv9jonfIsU939zuxtRLco-9zZ5UCYRt103pFMt3KqWB57iJOcvdC7DMBxQAaTkA3bUr1J5RfQuH2JsK86elY7OxZv1FIP_8zqREdemvFOUO0K6Kr6oZbvuNaxnRpH9XBYwE0dTNgWa9eBNO8=)[[14]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEG-zbpZQ3fgcuLz5I0ilNfyipOJ8IKDU7yU8cDgwjZDXDMYJt2-fGKx9y7KKgBsnRD3GedXPi_RskkyJr_RXewbjRoXuNG3UgL7K1tk_4xHeXcLBIfsjKUUjo=) . The method's power comes from the synergy between the chains:
*   **Global Exploration**: Hot chains traverse the flattened landscape, discovering different modes that are separated by large energy barriers [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEsNYRfktqhMFNEymo6MRcdQhgolfgv7LOvqqwqlgje5fyUHhLoHD9AKMpFomtkYQtRhTBuYN3yR_t4QP0CVqZtks_ohghRiIJgs-HBdUIojug5kMl_xJ2SDKYJdyRKvfrkeTSrve1HqRgb9qPmNia6azWFDiif5KH5JiEB9w==)[[12]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE_l1KzP4l7Qgm57qvbCESCZGdshIgA8p1w91NS9OilNxTQpQ-Jnkr4wRUMFMJTIar7c7fWtQuW8HtwS7VqAV67o2__AfCgI8SvSyspuT9lc-Yh4b_XUobYy1GqlFKE6NQ9kLUevzTWFe4=)[[15]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGYbu6dGzE6E3cRkmbipuvn_kTqQ6oGGW1Zh97pGfgu4LRgELSuQgA3JglxJyqXkQkVL0mTeXB1onnU01NPFvxCf6WyrzzrK5h2taaq6MukF52pTN_u1mTCNrauNND3F-AVHdmMRi3xmeu8kyny1KfFrlnY2vCY8bmfJfg=) .
*   **Information Transfer**: Successful swaps pass the states found by hot chains down the "temperature ladder" to colder chains. This allows the cold chain to make large "jumps" to new modes that it would never have reached on its own [[12]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE_l1KzP4l7Qgm57qvbCESCZGdshIgA8p1w91NS9OilNxTQpQ-Jnkr4wRUMFMJTIar7c7fWtQuW8HtwS7VqAV67o2__AfCgI8SvSyspuT9lc-Yh4b_XUobYy1GqlFKE6NQ9kLUevzTWFe4=)[[10]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGbRPexR70URrDq_dDIN2gbrf95MhJt5imU6s1f04frhWcvVCOcdtfeoOP2dFd2wyaOwm9PWQ4rC8WghY3XNlVFU0ZXSojRKp0AcgNLH0y2EgCnTM7LDzKSQ0EF9EK2JhF33Zt-drTSjHD4fgIYxDxEcQ==)[[14]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEG-zbpZQ3fgcuLz5I0ilNfyipOJ8IKDU7yU8cDgwjZDXDMYJt2-fGKx9y7KKgBsnRD3GedXPi_RskkyJr_RXewbjRoXuNG3UgL7K1tk_4xHeXcLBIfsjKUUjo=) .

In essence, the hot chains act as global explorers, while the cold chain performs a detailed local search of the promising regions it is "fed" by the hot chains, leading to a much more efficient and complete exploration of the target distribution [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEqERRtU19c_OiHulbkObilkii10EP0cflaeXeIynioY3oG2feEiyKEnPfGNqzNXmSj6ZTExYR2NKAP0dQs98d6O7NSDW2AcXQ9sX2RJ26DH0dmT97iRHCqV8jLjLKk16kcdZYpPrmsLdmoAmTMlbjHUCKksbdp3xoipJMooMrb_e9FP0SALzehHrwkx-8g17qzRoCkL5ZU5FRw7rCq)[[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEsNYRfktqhMFNEymo6MRcdQhgolfgv7LOvqqwqlgje5fyUHhLoHD9AKMpFomtkYQtRhTBuYN3yR_t4QP0CVqZtks_ohghRiIJgs-HBdUIojug5kMl_xJ2SDKYJdyRKvfrkeTSrve1HqRgb9qPmNia6azWFDiif5KH5JiEB9w==)[[5]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHJwRbpy65QtNLU5ZFpiNBZA9QAPwpZvSIL4yL8QGEZRQJl2ePC-nRXBwOn59mk00-dkq195AKrTX9_HbofNffRZZwUuVx_sibNCQSYj8e9yRlxN7M0ocp3rp9ghjNYCIVT0TSHTW-64DJEtWECOm5k9nzC2ruwfm59xxTWKinBqHZF6roBmzyVRTcxcqn13X0wZYKhHhHGuuAXHiYql3n8ZtV23HxILBfE_8Z4I9h2HlM=) .

---

### **Practical Considerations**

*   **Burn-in Period**: The initial samples from an MCMC chain are influenced by the starting value and may not be from the stationary distribution [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEqERRtU19c_OiHulbkObilkii10EP0cflaeXeIynioY3oG2feEiyKEnPfGNqzNXmSj6ZTExYR2NKAP0dQs98d6O7NSDW2AcXQ9sX2RJ26DH0dmT97iRHCqV8jLjLKk16kcdZYpPrmsLdmoAmTMlbjHUCKksbdp3xoipJMooMrb_e9FP0SALzehHrwkx-8g17qzRoCkL5ZU5FRw7rCq)[[23]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHYDUsKG_2c2ppcOJ5aRS72oYBEwFrmZwdVsGb9dnR7oANz7q-emceeIngXzvrIg2B12VIIoGEk_eUU_mBR3j8j03NKk4wLKhHuXbdE2hbe4KPMBSr66SywUuOYP0GSZ4UBoC-Q2MDnkHpcHW-GepN3XqJGAlKEGHYwDHC_)[[18]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQG1nNLYaSi0ewB_aKTnGPySE7sWS5LTogP5Xc_oDDXoNshNthKCeSpT9VHvtxLpLnJiWhU6Cpb4fejkkac9ROrc76_KLnSIcbbj17FGypjpQ9AXa2ldlSEvJMhtWnwFMpsjjz9n4rs9BMEdANR_4R96y0i6t1_QFus2wGkd1q9OWrzZNYAyw6C3_30aTpg8MEoKe88PsEiEiZn4qtsyt7lvWWWwf47D0kpMPg==) . It is standard practice to run the chain for a number of iterations and discard these initial samples, a process known as the "burn-in" [[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEsNYRfktqhMFNEymo6MRcdQhgolfgv7LOvqqwqlgje5fyUHhLoHD9AKMpFomtkYQtRhTBuYN3yR_t4QP0CVqZtks_ohghRiIJgs-HBdUIojug5kMl_xJ2SDKYJdyRKvfrkeTSrve1HqRgb9qPmNia6azWFDiif5KH5JiEB9w==)[[19]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHNyGqmt5NRKRq1svCc5fxvhg2wTcHJFylA0fuAQzO0qMLxOirKX9uXokRZUc4GZM_SuVaQqzUDIlKqkzJkIo4x7u39KltEgDtuBbQB9BPJmJImS9pN-CmAuRcXmmpkmz4eCJtuAvzy8d2HXesTpnzIB2aXv5MEa0m3jBkJkPGtCD5SxRyA0jpba4iJh2Y3iQ6Jq-5c2WdC5WoNeWUXzr1plmo=) .
*   **Acceptance Rate**: The fraction of proposed samples that are accepted is a key diagnostic. A very high rate may mean the steps are too small and the chain is exploring slowly, while a very low rate means proposals are too bold and are often rejected. For high-dimensional random-walk Metropolis, an acceptance rate of around 23.4% is often considered optimal [[12]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE_l1KzP4l7Qgm57qvbCESCZGdshIgA8p1w91NS9OilNxTQpQ-Jnkr4wRUMFMJTIar7c7fWtQuW8HtwS7VqAV67o2__AfCgI8SvSyspuT9lc-Yh4b_XUobYy1GqlFKE6NQ9kLUevzTWFe4=) . For HMC, much higher rates (e.g., 60-80%) are desirable. For Parallel Tempering, the swap acceptance rates between adjacent chains are also monitored and should be reasonably high.
*   **Proposal Distribution Tuning**: The parameters of the proposal distribution (e.g., the step size `ε` in RWMH, MALA, and HMC, the number of steps `L` in HMC, and the temperature ladder in PT) are crucial for efficiency and must be tuned for the specific problem.

### **Applications**

The Metropolis-Hastings algorithm and its variants are fundamental tools in many fields:
*   **Bayesian Inference**: Used to sample from complex posterior distributions to estimate parameters and quantify uncertainty [[1]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEqERRtU19c_OiHulbkObilkii10EP0cflaeXeIynioY3oG2feEiyKEnPfGNqzNXmSj6ZTExYR2NKAP0dQs98d6O7NSDW2AcXQ9sX2RJ26DH0dmT97iRHCqV8jLjLKk16kcdZYpPrmsLdmoAmTMlbjHUCKksbdp3xoipJMooMrb_e9FP0SALzehHrwkx-8g17qzRoCkL5ZU5FRw7rCq)[[2]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQEsNYRfktqhMFNEymo6MRcdQhgolfgv7LOvqqwqlgje5fyUHhLoHD9AKMpFomtkYQtRhTBuYN3yR_t4QP0CVqZtks_ohghRiIJgs-HBdUIojug5kMl_xJ2SDKYJdyRKvfrkeTSrve1HqRgb9qPmNia6azWFDiif5KH5JiEB9w==) .
*   **Statistical Physics**: The original Metropolis algorithm was developed to study the behavior of systems of particles [[3]](https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFwZrWr5_NH1GdmaHlCh3dRku63M-7hiZXtC28Mxn9Q1f8elDo8WKQbiDwfNNJjvU4CIFc_VAAby8pRF-c-9buLja-PQvbvLl8LQK3pkQdQUu4UD_vTrqj04nCWMrzz1vZjoiygMCYHXw==) .
*   **Machine Learning**: Employed in training and inference for models like Bayesian neural networks, topic models, and probabilistic graphical models.

---
## **Executive Summary**

The Metropolis-Hastings algorithm is a versatile MCMC method for generating samples from a probability distribution, `P(x)`, particularly when `P(x)` is known only up to a constant. It operates by constructing a Markov chain whose stationary distribution is `P(x)`. The core of the algorithm is an acceptance-rejection step governed by the **acceptance probability `α`**, which is designed to ensure the chain satisfies the detailed balance condition for convergence.

The choice of proposal distribution, `q(x'|x)`, defines different variants.
*   Simple choices include the **Random-Walk Metropolis-Hastings (RWMH)**, which uses a symmetric proposal.
*   **Gibbs Sampling** is a special case where proposals are drawn from the full conditional distributions, leading to an acceptance rate of 100%. It is highly efficient when these conditionals are easy to sample from.

More advanced methods are designed to improve sampling efficiency for complex, high-dimensional, or multi-modal distributions:
*   The **Metropolis-Adjusted Langevin Algorithm (MALA)** uses gradient information (`∇log P(x)`) to propose moves towards high-probability regions, using a Metropolis-Hastings correction step to remove discretization bias.
*   **Hamiltonian Monte Carlo (HMC)** augments the state space with auxiliary "momentum" variables and uses Hamiltonian dynamics, simulated via a leapfrog integrator, to propose distant states with a high probability of acceptance.
*   **Parallel Tempering (PT)**, or Metropolis-Coupled MCMC, runs multiple chains in parallel at different "temperatures." It uses swap moves between chains to allow the sampler to cross large energy barriers and explore multi-modal distributions more effectively.

Practical implementation requires attention to tuning the proposal distribution to achieve a reasonable acceptance rate and discarding an initial **burn-in** period to ensure samples are from the converged chain.